## Импорты

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import re
from urllib.parse import urljoin
import pandas as pd

Банки для парсинга

In [ ]:
BANKS = [
    {'name': 'Т-Банк', 'url': 'https://dreamjob.ru/employers/25607'},
    {'name': 'Сбер', 'url': 'https://dreamjob.ru/employers/25583'},
    {'name': 'ВТБ', 'url': 'https://dreamjob.ru/employers/25930'},
    {'name': 'Альфа-Банк', 'url': 'https://dreamjob.ru/employers/26069'},
    {'name': 'Газпромбанк', 'url': 'https://dreamjob.ru/employers/25929'},
    {'name': 'МКБ', 'url': 'https://dreamjob.ru/employers/39525'},
    {'name': 'Совкомбанк', 'url': 'https://dreamjob.ru/employers/46157'},
    {'name': 'Почта Банк', 'url': 'https://dreamjob.ru/employers/37657'},
    {'name': 'Райффайзенбанк', 'url': 'https://dreamjob.ru/employers/26097'}
]

Определение максимальной страницы

In [ ]:
def get_max_page(soup, base_url):
    pagination = soup.find('ul', class_='pagination-reviews')
    if not pagination:
        return 1

    last_link = pagination.find('li', class_='last').find('a')
    if last_link and last_link.get('href'):
        match = re.search(r'[?&]page=(\d+)', last_link['href'])
        if match:
            return int(match.group(1))
    return 1

Парсинг одного отзыва

In [ ]:
def parse_review(block):
    review = {}

    title_tag = block.find('h2', class_='review__header-title')
    review['position'] = title_tag.get_text(strip=True) if title_tag else ''

    meta_tags = block.find_all('div', class_='tags__item_grey')
    if len(meta_tags) >= 2:
        city_date = meta_tags[1].get_text(strip=True)
        parts = city_date.split(',')
        review['city'] = parts[0].strip() if len(parts) > 0 else ''
        review['date'] = parts[1].strip() if len(parts) > 1 else ''
    else:
        review['city'] = ''
        review['date'] = ''

    rating_tag = block.find('div', class_='dj-rating')
    review['rating'] = rating_tag.get_text(strip=True) if rating_tag else ''

    pros_header = block.find('div', string=re.compile(r'Что нравится\?'))
    if pros_header:
        pros_text = []
        for sibling in pros_header.next_siblings:
            if sibling.name == 'div' and 'review__title' in sibling.get('class', []):
                break
            if hasattr(sibling, 'get_text'):
                text = sibling.get_text(strip=True)
                if text:
                    pros_text.append(text)
            elif isinstance(sibling, str) and sibling.strip():
                pros_text.append(sibling.strip())
        review['pros'] = ' '.join(pros_text)
    else:
        review['pros'] = ''

    cons_header = block.find('div', string=re.compile(r'Что можно улучшить\?'))
    if cons_header:
        cons_text = []
        for sibling in cons_header.next_siblings:
            if sibling.name == 'div' and 'review__answer' in sibling.get('class', []):
                break
            if hasattr(sibling, 'get_text'):
                text = sibling.get_text(strip=True)
                if text:
                    cons_text.append(text)
            elif isinstance(sibling, str) and sibling.strip():
                cons_text.append(sibling.strip())
        review['cons'] = ' '.join(cons_text)
    else:
        review['cons'] = ''

    answer_block = block.find('div', class_='review__answer')
    if answer_block:
        answer_date_tag = answer_block.find('div', class_='review__answer-date')
        answer_date = answer_date_tag.get_text(strip=True) if answer_date_tag else ''
        answer_text_parts = []
        for elem in answer_block.find_all(['p', 'div'], recursive=False):
            if 'review__answer-date' in elem.get('class', []):
                continue
            text = elem.get_text(strip=True)
            if text:
                answer_text_parts.append(text)
        answer_text = ' '.join(answer_text_parts)
        review['company_answer'] = f"{answer_date}: {answer_text}" if answer_date else answer_text
    else:
        review['company_answer'] = ''

    return review

Парсинг всех отзывов банка

In [ ]:
def parse_all_reviews(base_url, bank_name, start_page=1, delay=1.5):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    session = requests.Session()
    session.headers.update(headers)

    all_reviews = []
    page = start_page
    max_page = None

    while True:
        if page == 1:
            url = base_url
        else:
            url = f"{base_url.rstrip('/')}?page={page}"

        print(f"[{bank_name}] Парсинг страницы {page}...")

        try:
            resp = session.get(url, timeout=15)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'lxml')

            if max_page is None:
                max_page = get_max_page(soup, base_url)
                print(f"[{bank_name}] Всего страниц: {max_page}")

            review_blocks = soup.find_all('div', class_='review')
            if not review_blocks:
                print(f"[{bank_name}] На странице {page} нет отзывов. Завершаем.")
                break

            for block in review_blocks:
                if block.find('h2', class_='review__header-title') is None:
                    continue
                review_data = parse_review(block)
                review_data['bank'] = bank_name
                review_data['page'] = page
                all_reviews.append(review_data)

            print(f"[{bank_name}] Собрано отзывов на странице {page}: {len(review_blocks)}")

            if page >= max_page:
                print(f"[{bank_name}] Достигнута последняя страница.")
                break

            page += 1
            time.sleep(delay)

        except requests.exceptions.RequestException as e:
            print(f"[{bank_name}] Ошибка при загрузке страницы {page}: {e}")
            break
        except Exception as e:
            print(f"[{bank_name}] Неожиданная ошибка на странице {page}: {e}")
            break

    return all_reviews

Сохранение в CSV

In [ ]:
def save_to_csv(data, filename):
    if not data:
        print("Нет данных для сохранения.")
        return
    with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys(), delimiter=';')
        writer.writeheader()
        writer.writerows(data)
    print(f"Сохранено {len(data)} отзывов в {filename}")

Основной запуск

In [ ]:
if __name__ == "__main__":
    all_reviews = []
    total_start_time = time.time()

    for bank in BANKS:
        print(f"\n======= Начинаем сбор для банка: {bank['name']} =======")
        reviews = parse_all_reviews(bank['url'], bank['name'], start_page=1, delay=1.5)
        print(f"[{bank['name']}] Собрано отзывов: {len(reviews)}")
        all_reviews.extend(reviews)
        time.sleep(2)

    if all_reviews:
        save_to_csv(all_reviews, 'all_banks_reviews.csv')
        print(f"\nОбщее количество собранных отзывов: {len(all_reviews)}")
        print(f"Время выполнения: {time.time() - total_start_time:.1f} сек.")
    else:
        print("Не удалось собрать ни одного отзыва.")

Категоризация должностей

In [ ]:
#данный набор позиций обрабатывала через deepseek
category = {
  "Специалист контакного центра": "не ИТ",
  "Менеджер по работе с премиальными клиентами": "не ИТ",
  "Специалист технической поддержки": "Администрирование",
  "Оператор разметки данных": "Аналитика данных/ Data Science",
  "Оператор колл центра": "не ИТ",
  "Оператор документооборота": "не ИТ",
  "Менеджер по работе с клиентами": "не ИТ",
  "Ведущий консультант": "не ИТ",
  "Специалист по документообороту": "не ИТ",
  "Специалист контактного центра": "не ИТ",
  "Банковский представитель": "не ИТ",
  "Специалист складского учета": "не ИТ",
  "Представитель банка": "не ИТ",
  "Отзыв сотрудника": "не ИТ",
  "Ведущий специалист отдела взыскания": "не ИТ",
  "Оператор call-центра": "не ИТ",
  "Инженер по качеству": "Тестирование",
  "Специалист по развитию регионов": "не ИТ",
  "Менеджер агентской группы": "не ИТ",
  "Представитель": "не ИТ",
  "Ведущий специалист семейного банкинга": "не ИТ",
  "Банковский сотрудник": "не ИТ",
  "Аудитор": "не ИТ",
  "Менеджер продукта": "не ИТ",
  "Ведущий банковский представитель": "не ИТ",
  "Старший разработчик": "Разработка",
  "Специалист по страхованию": "не ИТ",
  "Инженер семантического поиска": "Разработка",
  "Старший бизнес-аналитик": "не ИТ",
  "Специалист по банковским продуктам": "не ИТ",
  "Специалист отдела поиска угроз информационной безопасности": "Администрирование",
  "Торговый представитель": "не ИТ",
  "Оператор взыскания задолженности": "не ИТ",
  "Специалист отдела клиентского обслуживания": "не ИТ",
  "Специалист по взысканию задолженности": "не ИТ",
  "Специалист по кредитованию": "не ИТ",
  "Руководитель группы развития клиентов малого бизнеса": "не ИТ",
  "Менеджер по работе с клиентами B2B": "не ИТ",
  "Оператор дебетовых карт": "не ИТ",
  "Старший менеджер по продажам b2b": "не ИТ",
  "Менеджер по продажам": "не ИТ",
  "Ведущий специалист отдела обслуживания бизнеса": "не ИТ",
  "Куратор развития продаж": "не ИТ",
  "Инженер технической поддержки": "Администрирование",
  "«Оператор домашнего КЦ»": "не ИТ",
  "Оператор раннего взыскания": "не ИТ",
  "Менеджер по развитию профессиональных компетенций": "не ИТ",
  "Эксперт по развитию на маркетплейсах": "не ИТ",
  "Ведущий оператор call-центра": "не ИТ",
  "Руководитель смены": "не ИТ",
  "Специалист поддержки": "не ИТ",
  "Практикант": "не ИТ",
  "Руководитель структурного подразделения, отдел продаж г. Самара.": "не ИТ",
  "Data Engineer": "Аналитика данных/ Data Science",
  "Оператор": "не ИТ",
  "Ведущий менеджер по продажам": "не ИТ",
  "Ведущий оператор клиентской поддержки": "не ИТ",
  "Оператор контактного центра": "не ИТ",
  "Аналитик данных": "Аналитика данных/ Data Science",
  "Банковский менеджер": "не ИТ",
  "Младший специалист сектора дистанционной обработки данных": "не ИТ",
  "Специалист (Сектор документооборота)": "не ИТ",
  "Менеджер отдела депозитных продуктов": "не ИТ",
  "Оператор отдела взыскания": "не ИТ",
  "Колл-центр": "не ИТ",
  "Бухгалтер": "не ИТ",
  "Оператор в отделе развития качества": "не ИТ",
  "Специалист взыскания": "не ИТ",
  "Специалист службы поддержки клиентов": "не ИТ",
  "Специалист отдела взыскания": "не ИТ",
  "Специалист клиентской поддержки": "не ИТ",
  "Специалист комплаенс проверок физических лиц.": "не ИТ",
  "Агент по продажам": "не ИТ",
  "Специалист отдела страхования": "не ИТ",
  "Региональный менеджер по развитию партнерской сети": "не ИТ",
  "Системный аналитик": "Разработка",
  "Придставитель банка": "не ИТ",
  "Ведущий эксперт  управления развития качества": "не ИТ",
  "Оператор позднего взыскания": "не ИТ",
  "Эксперт в бэк-офисе": "не ИТ",
  "Менеджер по страховым продуктам": "не ИТ",
  "Продуктовый аналитик": "Аналитика данных/ Data Science",
  "Ведущий разработчик": "Разработка",
  "Оператор ПК": "не ИТ",
  "Специалист по технической дистанционной поддержке": "Администрирование",
  "Территориальный представитель": "не ИТ",
  "Golang-разработчик": "Разработка",
  "Руководитель группы продаж": "не ИТ",
  "Оператор первой линии поддержки Мобильной связи": "Администрирование",
  "Главный кредитный инспектор": "не ИТ",
  "Ведущий системный аналитик": "Разработка",
  "Старший менеджер по продажам": "не ИТ",
  "Оператор по сохранению клиентов": "не ИТ",
  "Специалист группы клиентского обслуживания": "не ИТ",
  "Оператор электронных заявок.": "не ИТ",
  "Ведущий персональный менеджер по инвестициям": "не ИТ",
  "Специалист по работе с просроченной задолженностью": "не ИТ",
  "Инкасатор": "не ИТ",
  "Младший специалист бизнесс-процесса": "не ИТ",
  "Ведущий оператор": "не ИТ",
  "Менеджер корпоративной поддержки": "не ИТ",
  "Прелставитель": "не ИТ",
  "Персональный менеджер": "не ИТ",
  "Младший специалист информационной обработки данных": "не ИТ",
  "Персональный менеджер по работе с малым и средним бизнесом": "не ИТ",
  "Руководитель группы (Менеджер)": "не ИТ",
  "Ведущий эксперт-консультант": "не ИТ",
  "Специалист по продажам бизнесу": "не ИТ",
  "Специалист по работе с клиентами": "не ИТ",
  "Java-разработчик": "Разработка",
  "Главный специалист. Группа поддержки Сектора обслуживания иностранных клиентов Отдела обслуживания депозитных продуктов": "не ИТ",
  "Customer Support Specialist": "не ИТ",
  "Специалист контакт-центра": "не ИТ",
  "Оператор по взысканию": "не ИТ",
  "Оператор базы данных": "Администрирование",
  "Специалист по обработке данных": "Аналитика данных/ Data Science",
  "Специалист чата": "не ИТ",
  "Руководитель группы": "не ИТ",
  "Бизнес-аналитик": "не ИТ",
  "Оператор дистанционной обработки данных": "не ИТ",
  "Ведущий специалист группы обслуживания состоятельных клиентов": "не ИТ",
  "Специалист по обработке документов": "не ИТ",
  "Менеджер поддержки": "не ИТ",
  "Агент поддержки": "не ИТ",
  "Специалист фрод-мониторинга": "не ИТ",
  "Консультант": "не ИТ",
  "Старший кредитный специалист": "не ИТ",
  "Менеджер по работе с клиентами среднего и крупного бизнеса": "не ИТ",
  "Специалист поддержки акций и бенефитов.": "не ИТ",
  "Специалист поддержки в чатах": "не ИТ",
  "Оператор службы поддержки клиентов": "не ИТ",
  "Специалист отдела контроля качества": "не ИТ",
  "Специалист call-центра": "не ИТ",
  "Бизнес-консультант": "не ИТ",
  "Руководитель проектов": "не ИТ",
  "Главный специалист по антифрод-сопровождению": "не ИТ",
  "Специалист по документообороту госорганами": "не ИТ",
  "Бизнес консультант": "не ИТ",
  "Ведущий менеджер по работе с крупным бизнесом": "не ИТ",
  "Backend developer": "Разработка",
  "Ведущий менеджер по работе с бизнесом": "не ИТ",
  "Менеджер технической поддержки": "Администрирование",
  "Отдел взыскания": "не ИТ",
  "Ответчик на запросы + Сканировщик": "не ИТ",
  "Отдел обслуживания кредитных карт.": "не ИТ",
  "C#/.NET-разработчик": "Разработка",
  "Специалист в секторе обслуживания дебетовых карт": "не ИТ",
  "Оператор ПК/Оператор базы данных": "Администрирование",
  "Специалист отдела обработки исполнительных документов от внешних взыскателей": "не ИТ",
  "Старший специалист по сопровождению комплаенс проверок бизнеса": "не ИТ",
  "Старший продуктовый аналитик": "Аналитика данных/ Data Science",
  "Оператор облачного чата": "не ИТ",
  "Ведущий Backend-разработчик": "Разработка",
  "Аналитик": "Аналитика данных/ Data Science",
  "Эксперт по работе с юридическими лицами (удалённо)": "не ИТ",
  "Руководитель группы кредитования малого и среднего бизнеса": "не ИТ",
  "Младший специалист отдела дистанционной обработки информации": "не ИТ",
  "Кредитный инспектор": "не ИТ",
  "Backend-разработчик": "Разработка",
  "Разработчик Rust": "Разработка",
  "IOS-разработчик": "Разработка",
  "Помощник бухгалтера": "не ИТ",
  "Менеджер по регистрации бизнеса": "не ИТ",
  "Ведущий специалист депозитных продуктов, ведущий специалист финансовых обращений": "не ИТ",
  "Представитель, ментор, администратор выходного дня": "не ИТ",
  "Инженер по обеспечению качества": "Тестирование",
  "Ведущий менеджер по продажам кредитных продуктов": "не ИТ",
  "Специалист по  взысканию задолженности": "не ИТ",
  "Эксперт отдела депозитных продуктов": "не ИТ",
  "Специалист отдела входящих обращений": "не ИТ",
  "Выездной представитель": "не ИТ",
  "Консультант по банковским продуктам": "не ИТ",
  "Кредитный специалист": "не ИТ",
  "Оператор по назначению встреч": "не ИТ",
  "Преподаватель": "не ИТ",
  "Оператор обработки сообщений": "не ИТ",
  "Методолог": "не ИТ",
  "Специалист по работе с состоятельными клиентами.": "не ИТ",
  "IT Project Manager": "Администрирование",
  "Руководитель группы обслуживания клиентов": "не ИТ",
  "Специалист по разметке данных": "Аналитика данных/ Data Science",
  "Инженер": "Разработка",
  "Оператор электронных заявок": "не ИТ",
  "Ведущий эксперт": "не ИТ",
  "Эксперт управления развития качества": "не ИТ",
  "Представитель группы Коллекшн": "не ИТ",
  "Ведущий менеджер по работе с партнерами": "не ИТ",
  "Консультант по стратегии": "не ИТ",
  "Региональный руководитель": "не ИТ",
  "Менеджер по развитию ключевых клиентов": "не ИТ",
  "Менеджер по продажам банковских продуктов": "не ИТ",
  "Руководитель группы методологии обучения": "не ИТ",
  "Клиентский менеджер": "не ИТ",
  "Руководитель группы развития среднего и крупного бизнеса": "не ИТ",
  "Кредитный инспекторор": "не ИТ",
  "Специалист отдела взыскания просроченной задолженности": "не ИТ",
  "TypeScript-разработчик": "Разработка",
  "Senior Frontend Developer (React)": "Разработка",
  "Генеральный директор ООО «Тинькофф Инвест Лаб» (проф участник РЦБ)": "не ИТ",
  "Главный менеджер  по работе со средним и крупным бизнесом": "не ИТ",
  "Младший методолог": "не ИТ",
  "Эксперт по инвестиционным продуктам": "не ИТ",
  "Актуализация данных": "не ИТ",
  "Главный менеджер по работе с крупным бизнесом": "не ИТ",
  "Эксперт по работе с бизнес клиентами": "не ИТ",
  "Менеджер по продажам В2В": "не ИТ",
  "Кредитный эксперт": "не ИТ",
  "Руководитель строительного проекта": "не ИТ",
  "Главный юрисконсульт": "не ИТ",
  "Главный специалист по работе с клиентами": "не ИТ",
  "Специалист по автострахованию": "не ИТ",
  "Специалист АХО": "не ИТ",
  "QA Engineer": "Тестирование",
  "Главный менеджер": "не ИТ",
  "Администратор-оператор": "Администрирование",
  "Главный специалист. Отдел обслуживания бизнеса": "не ИТ",
  "ETL-Разработчик": "Аналитика данных/ Data Science",
  "Специалист по входящим обращениям": "не ИТ",
  "Инкассатор 2 категории": "не ИТ",
  "Ведущий кредитный инспектор": "не ИТ",
  "Специалист службы поддержки": "не ИТ",
  "Руководитель группы технической поддержки": "Администрирование",
  "Руководитель группы разработки": "Разработка",
  "Руководитель точки продаж": "не ИТ",
  "Контролер отдела качества": "не ИТ",
  "Оператор колл-центра": "не ИТ",
  "Оператор по работе с документами": "не ИТ",
  "Специалист поддержки клиентов": "не ИТ",
  "Оператор на чатах": "не ИТ",
  "Трейдер (самостоятельная работа)": "не ИТ",
  "Менеджер по продажам прямым": "не ИТ",
  "Менеджер по договорной работе": "не ИТ",
  "Оператор исходящих звонков": "не ИТ",
  "Младший специалист фрод-мониторинга": "не ИТ",
  "Специалист поддержки чатов/звонков": "не ИТ",
  "ИТ-специалист": "Администрирование",
  "Ассистент по документообороту": "не ИТ",
  "Главный инвестиционный менеджер": "не ИТ",
  "Сотрудник отдела досудебного урегулирования": "не ИТ",
  "Оператор службы поддержки": "не ИТ",
  "Менеджер по продажам продуктов банка": "не ИТ",
  "Менеджер по продажам b2b": "не ИТ",
  "Специалист по информационной безопасности": "Администрирование",
  "Младший специалист сектора премиального обслуживания": "не ИТ",
  "Оператор технической поддержки": "Администрирование",
  "Оператор мобильной связи": "не ИТ",
  "Куратор по развитию продаж": "не ИТ",
  "Руководитель направления": "не ИТ",
  "Менеджер по продажам кредитных карт": "не ИТ",
  "Менеджер по назначению встреч": "не ИТ",
  "Аналитик-технолог": "не ИТ",
  "Эксперт-интегратор": "не ИТ",
  "Сотрудник отдела раннего взыскания": "не ИТ",
  "Кладовщик": "не ИТ",
  "Специалист по работе с бизнесом": "не ИТ",
  "Руководитель кросс-продаж продукта": "не ИТ",
  "Специалист, Сектор обслуживания дебетовых карт. Отдел обслуживания депозитных продуктов": "не ИТ",
  "Менеджер продуктового направления (РПН)": "не ИТ",
  "Региональный представитель": "не ИТ",
  "Консультант по Т-путешествиям": "не ИТ",
  "Менеджер по персоналу": "не ИТ",
  "Старший персональный менеджер по инвестициям": "не ИТ",
  "SRE-инженер": "Администрирование",
  "Оператор поддержки": "не ИТ",
  "Специалист Управления развития качества": "не ИТ",
  "Ведущий менеджер по продажам среднего и крупного бизнеса": "не ИТ",
  "Специалист по взысканию просроченной задолженности": "не ИТ",
  "Сотрудник отдела взыскания": "не ИТ",
  "Разработчик": "Разработка",
  "Специалист отдела обслуживания клиентов": "не ИТ",
  "Стажер-координатор образовательных программ (команда адаптации)": "не ИТ",
  "Эксперт": "не ИТ",
  "Руководитель группы Сектор сопровождения продаж г. Самара Приволжский блок секторов Отдел сопровождения продаж Управление сопровождения продаж Департамент бизнес-технологий и операций": "не ИТ",
  "Специалист отдела поддержки клиентов": "не ИТ",
  "Консультант по Т-страхованию": "не ИТ",
  "Специалист по обработке данных (Удаленно)": "Аналитика данных/ Data Science",
  "Специалист": "не ИТ",
  "Главный специалист по работе с юридическими лицами": "не ИТ",
  "Консультант Т-Бизнес": "не ИТ",
  "Специалист. Сектор обслуживания кредитных карт": "не ИТ",
  "Оператор call-центра, контроль качества": "не ИТ",
  "Оператор входящих обращений ОМНИ": "не ИТ",
  "Экперт": "не ИТ",
  "Ночной консультант по банковским продуктам": "не ИТ",
  "Консультант Т-инвестиций": "не ИТ",
  "Оператор направления страхования": "не ИТ",
  "Главный Эксперт Отдела Электронных Обращений": "не ИТ",
  "Специалист по поддержке пользователей": "Администрирование",
  "Оператор отдела реструктаризации": "не ИТ",
  "Специалист разметки данных": "Аналитика данных/ Data Science",
  "Оператор входящих звонков": "не ИТ",
  "Старший андеррайтер МСБ": "не ИТ",
  "Менеджер проекта": "не ИТ",
  "Ведущий специалист, бизнес-тренер": "не ИТ",
  "Специалист отдела документооборота": "не ИТ",
  "Специалист раннего взыскания": "не ИТ",
  "Менеджер холодных продаж": "не ИТ",
  "Специалист по работе с премиальными клиентами": "не ИТ",
  "Менеджер по продукту": "не ИТ",
  "Менеджер B2B продаж электронной коммерции": "не ИТ",
  "Эксперт направления": "не ИТ",
  "Специалист поддержки англоговорящих клиентов": "не ИТ",
  "Главный аналитик AML рисков": "Аналитика данных/ Data Science",
  "Специалист отдела обслуживания Депозитных продуктов": "не ИТ",
  "Менеджер по работе с блогерами": "не ИТ",
  "Оператор на ипотечном страховании": "не ИТ",
  "Специалист по инвестиционным продуктам": "не ИТ",
  "Персональный менеджер каско": "не ИТ",
  "Оператор обработки данных(удаленно": "Аналитика данных/ Data Science",
  "Менеджер по развитию бизнеса": "не ИТ",
  "Сборщик банкоматов": "не ИТ",
  "Старший продуктовый маркетолог": "не ИТ",
  "Специалист линии помощи подразделения дебетовых карт": "не ИТ",
  "Сотрудник Call-центра": "не ИТ",
  "Руководитель группы привлечения клиентов (СКБ)": "не ИТ",
  "Оператор call-центра, Специалист по обучению и развитию сотрудников.": "не ИТ",
  "Территориальный представитель по торговому маркетингу": "не ИТ",
  "Заместитель руководителя отдела продаж": "не ИТ",
  "Старший персональный менеджер": "не ИТ",
  "Ведущий специалист отдела": "не ИТ",
  "Оператор в т Мобаил": "не ИТ",
  "Младший инженер": "Разработка",
  "Менеджер по подбору персонала": "не ИТ",
  "Ведущий менеджер по среднему и крупному бизнесу": "не ИТ",
  "Инженер, АТМ мониторинг": "Администрирование",
  "Менеджер по МСП": "не ИТ",
  "Оператор клиентской поддержки": "не ИТ",
  "Сотрудник колл-центра": "не ИТ",
  "Бизнес-тренер": "не ИТ",
  "Ночной консультант по продуктам банка": "не ИТ",
  "Менеджер по коммуникациям": "не ИТ",
  "Менеджер по продажам кредитных продуктов": "не ИТ",
  "Менеджер проектов по профессиональному развитию": "не ИТ",
  "Главный специалист контроля качества": "не ИТ",
  "Менеджер по обучению и адаптации": "не ИТ",
  "Консультант Т-Страхования": "не ИТ",
  "Руководитель сектора": "не ИТ",
  "Специалист кредитного отдела": "не ИТ",
  "Оператор ДКО": "не ИТ",
  "Оператор  колл-центра": "не ИТ",
  "Специалист позднего взыскания": "не ИТ",
  "Специалист службы поддержки мобильной связи": "не ИТ",
  "Консультант Т-страхование": "не ИТ",
  "Вечерний консультант": "не ИТ",
  "Специалист по работе с претензиями": "не ИТ",
  "Консультант по инвестиционным продуктам": "не ИТ",
  "Оператор по взысканию задолженности": "не ИТ",
  "Специалист техподдержки": "Администрирование",
  "Специалист по обработке входящих звонков": "не ИТ",
  "Руководитель направления разработки (": "Разработка",
  "Lead Product Designer": "Дизайн",
  "Оператор взыскания": "не ИТ",
  "Менеджер по продажам Т-Бизнес": "не ИТ",
  "Старший специалист, Отдел логистики": "не ИТ",
  "Специалист по работе с обращениями": "не ИТ",
  "Оператор по подбору персонала": "не ИТ",
  "Старший специалист по безопасности": "не ИТ",
  "Эксперт по развитию бизнеса": "не ИТ",
  "Специалист отдела": "не ИТ",
  "Руководитель группы продаж Ecom": "не ИТ",
  "Консультант Т-путешествия": "не ИТ",
  "Специалист. Чаты": "не ИТ",
  "Консультант Тинькофф Страхование": "не ИТ",
  "Менеджер по взысканию задолженности": "не ИТ",
  "Делопроизводитель": "не ИТ",
  "Ведущий специалист отдела претензий": "не ИТ",
  "Менеджер по работе с премиальными клиентами бизнеса.": "не ИТ",
  "Инженер IT": "Администрирование",
  "Проектный менеджер": "не ИТ",
  "Ведущий менеджер по работе с клиентами среднего и крупного бизнеса": "не ИТ",
  "Эксперт по продукту": "не ИТ",
  "Консультант по дебетовым продуктам": "не ИТ",
  "Ведущий специалист третьей категории": "не ИТ",
  "Вечерний консультант по банковским продуктам": "не ИТ",
  "Оператор отдела сканирования": "не ИТ",
  "Data Scientist (RecSys)": "Аналитика данных/ Data Science",
  "Специалист анти-фрод поддержки": "не ИТ",
  "Специалист ОКО": "не ИТ",
  "Оператор в службе поддержки": "не ИТ",
  "Младший специалист сектор поддержки потребительских кредитов управления бэк офис": "не ИТ",
  "Ночной консультант по банковским продуктам в регионах России": "не ИТ",
  "Старший специалист поддержки": "не ИТ",
  "ОУИ специалист по службе безопасности": "не ИТ",
  "Проектный менеджер по обучению": "не ИТ",
  "Специалист сектора обработки внутренних запросов": "не ИТ",
  "Представитель компании": "не ИТ",
  "Эксперт SME сектор обслуживания бизнеса": "не ИТ",
  "Старший оператор базы данных": "Администрирование",
  "Специалист по антифрод поддержки": "не ИТ",
  "Младший дизайнер": "Дизайн",
  "Ведущий специалист по обслуживанию премиальных клиентов": "не ИТ",
  "Оператор на телефоне": "не ИТ",
  "Руководитель продукта": "не ИТ",
  "FullStack разработчик": "Разработка",
  "Ведущий специалист по работе с клиентами": "не ИТ",
  "Специалист (частичная занятость)": "не ИТ",
  "Специалист удаленного урегулирования убытков": "не ИТ",
  "Ведущий эксперт поддержки отдела нефинансовых продуктов": "не ИТ",
  "Главный эксперт по обслуживанию премиальных клиентов": "не ИТ",
  "Менеджер по работе с бизнесом": "не ИТ",
  "Специалист отдела взысканий": "не ИТ",
  "BI-аналитик": "Аналитика данных/ Data Science",
  "Заместитель директора": "не ИТ",
  "Главный менеджер по работе со средним бизнесом": "не ИТ",
  "Ведущий специалист": "не ИТ",
  "Ведущий специалист ОПП": "не ИТ",
  "Консультант по продуктам Т-банка": "не ИТ",
  "Менеджер по работе с корпоративными клиентами": "не ИТ",
  "Специалист поддержки мобильной связи": "не ИТ",
  "Выездной Инженер  по ремонту банкоматов": "Администрирование",
  "Менеджер по продажам бизнесу": "не ИТ",
  "Специалист потребительского кредитования": "не ИТ",
  "Консультант по инвестициям": "не ИТ",
  "Оператор взыскания задолженности по телефону": "не ИТ",
  "Оператор дистанционного колл-центра": "не ИТ",
  "Консультант ночной по продуктам банка": "не ИТ",
  "Менеджер по работе с бизнес-клиентами": "не ИТ",
  "Специалист по продажам кредитных продуктов бизнесу": "не ИТ",
  "Менеджер по продаже кредитных продуктов": "не ИТ",
  "Главный менеджер по развитию IT продуктов": "не ИТ",
  "Оператор домашнего колл-центра": "не ИТ",
  "Менеджер по продажам кредитных продуктов бизнесу": "не ИТ",
  "Модератор": "не ИТ",
  "QA специалист": "Тестирование",
  "Оператор обработки данных": "Аналитика данных/ Data Science",
  "Специалист по обслуживанию инвестиционных продуктов": "не ИТ",
  "Консультант по продуктам банка": "не ИТ",
  "Заместитель руководителя направления складского учета": "не ИТ",
  "Оператор входящих обращений": "не ИТ",
  "Администратор": "Администрирование",
  "Web-разработчик": "Разработка",
  "Эксперт-консультант": "не ИТ",
  "SRE": "Администрирование",
  "Руководитель отдела маркетинга": "не ИТ",
  "Инвестиционный консультант": "не ИТ",
  "Специалист, сектор поддержки клиентов по взаимодействию с государственными органами": "не ИТ",
  "Менеджер по развитию бизнеса, персональный менеджер": "не ИТ",
  "Чат поддержка клиента": "не ИТ",
  "Специалист консультант по банковским продуктам": "не ИТ",
  "Специалист банка": "не ИТ",
  "Коммерческий представитель": "не ИТ",
  "Менеджер по работе с юридическими лицами": "не ИТ",
  "Комплаенс-менеджер": "не ИТ",
  "Оператор линии": "не ИТ",
  "Специалист отдела по работе с мошенничеством": "не ИТ",
  "Руководитель группы торговых представителей": "не ИТ",
  "Оператор чата": "не ИТ",
  "Специалист контроля качества": "не ИТ",
  "Специалист, сектор противодействия мошенничеству и Дебетовый отдел": "не ИТ",
  "Специалист по продаже банковских продуктов": "не ИТ",
  "Сектор сопровождения продаж": "не ИТ",
  "Эксперт в сопровождении комплаенс проверок фл.": "не ИТ",
  "Руководитель бизнес группы": "не ИТ",
  "Эксперт в отделе обслуживания депозитных продуктов": "не ИТ",
  "Оператор сотовой связи": "не ИТ",
  "Специалист по антифрод поддержке клиентов": "не ИТ",
  "ML-разработчик": "Аналитика данных/ Data Science",
  "Оператор обработки электронных документов": "не ИТ",
  "Специалист по обработке заявок": "не ИТ",
  "Курьер": "не ИТ",
  "Старший менеджер развития по работе с крупным бизнесом": "не ИТ",
  "Консультант дебетовых продуктов": "не ИТ",
  "Оператор назначения встреч": "не ИТ",
  "Руководитель группы по работе с бизнесом": "не ИТ",
  "Ведущий эксперт, сектор обслуживания кредитных карт": "не ИТ",
  "Специалист контактного центра (Т-мабайл)": "не ИТ",
  "Продакт менеджер": "не ИТ",
  "Технический писатель": "Разработка",
  "Стажер-разработчик": "Разработка",
  "Ведущий специалист по безопасности": "не ИТ",
  "Сотрудник банка": "не ИТ",
  "Главный специалист поддержки отдела непрерывного обслуживания бизнеса": "не ИТ",
  "КРП (Куратор по развитию продаж)": "не ИТ",
  "Инженер мониторинга технической подержки банкоматов и терминалов": "Администрирование",
  "Compliance": "не ИТ",
  "Старший системный аналитик": "Разработка",
  "AI-тренер": "Аналитика данных/ Data Science",
  "Сотрудник по обслуживанию инвестиционных продуктов": "не ИТ",
  "Продуктовый маркетолог": "не ИТ",
  "Главный эксперт по работе с клиентами среднего и крупного бизнеса": "не ИТ",
  "Специалист Сектора обработки входящих чатов в регионах": "не ИТ",
  "Стажер-аналитик": "Аналитика данных/ Data Science",
  "Специалист Сектор противодействия мошенничеству Отдел кросс-функционального обслуживания Управление поддержки клиентов Департамент клиентского обслуживания": "не ИТ",
  "Руководитель отдела поддержки бизнеса": "не ИТ",
  "Старший специалист": "не ИТ",
  "Эксперт в секторе поддержки потребительских кредитов. Отдел поддержки кредитных и депозитных продуктов. Управление бэк-офиса. Департамент клиентского обслуживания.": "не ИТ",
  "Специалист линии помощи бизнеса": "не ИТ",
  "Оператор по продаже банковских продуктов": "не ИТ",
  "Специалист в отдел исходящих обращений, управления поддержки клиентов": "не ИТ",
  "Golang разработчик": "Разработка",
  "Ведущий специалист по поиску информации (": "не ИТ",
  "Специалист по входящей корреспонденции": "не ИТ",
  "Оператор технической поддержки клиентов ,направление-дебетовве карты": "Администрирование",
  "Специалист клиентской поддержки по дебетовым картам": "не ИТ",
  "Консультант по кредитным продуктм": "не ИТ",
  "Младший специалист Группа фрод-мониторинга эмиссии Отдел противодействия мошенничеству Центр экосистемной защиты Департамент платежных систем и операционной поддержки": "не ИТ",
  "Специалист по продажам": "не ИТ",
  "Менеджер продаж B2B": "не ИТ",
  "VAS lead product manager / Руководитель направления доп.продуктов": "не ИТ",
  "Курьер на личном автомобиле": "не ИТ",
  "Сотрудник поддержки клиентов": "не ИТ",
  "Оператор call-центра, старший специалист": "не ИТ",
  "Руководитель направления продаж": "не ИТ",
  "Менеджер по работе с ключевыми клиентами": "не ИТ",
  "Разметчик данных": "Аналитика данных/ Data Science",
  "Оператор ввода данных": "Аналитика данных/ Data Science",
  "Главный менеджер группы оценки": "не ИТ",
  "Специалист бэк-офиса": "не ИТ",
  "ФССП, Специалист. Сектор обработки внешних запросов Отдел исходящих обращений Управление поддержки клиентов Департамент клиентского обслуживания": "не ИТ",
  "CRM-менеджер": "не ИТ",
  "Специалист Т-Страхование": "не ИТ",
  "Консультант Т-инвестиции": "не ИТ",
  "Ведущий программист": "Разработка",
  "Оператор ДКЦ": "не ИТ",
  "Оператор домашнего call-центра": "не ИТ",
  "Специалист по развитию бизнеса": "не ИТ",
  "Консультант службы клиентской поддержки": "не ИТ",
  "Специалист отдела электронной коммерции": "не ИТ",
  "Оператор удаленного колл-центра": "не ИТ",
  "Специалист по обслуживанию клиентов Т бизнес": "не ИТ",
  "Главный Специалист Сектор обслуживания дебетовых карт Отдел обслуживания депозитных продуктов": "не ИТ",
  "Специалист ДКО": "не ИТ",
  "Инженер дежурной смены": "Администрирование",
  "Консультант Тинькофф Банк": "не ИТ",
  "Ведущий менеджер по работе с клиентами среднего бизнеса": "не ИТ",
  "Менеджер SME": "не ИТ",
  "Старший  аналитик процессов": "не ИТ",
  "Курьер по доставке": "не ИТ",
  "Специалист по инвестициям": "не ИТ",
  "Администратор баз данных / Инженер по автоматизации": "Администрирование",
  "QA Automation Engineer": "Тестирование",
  "Ведущий специалист департамента клиентского обслуживания": "не ИТ",
  "Старший менеджер по продажам банковских продуктов": "не ИТ",
  "Оператор входящих сообщений": "не ИТ",
  "Руководитель группы развития клиентов среднего и крупного бизнеса": "не ИТ",
  "Senior Golang-разработчик": "Разработка",
  "Специалист обработки данных": "Аналитика данных/ Data Science",
  "Ведущий аналитик данных": "Аналитика данных/ Data Science",
  "Менеджер (чаты)": "не ИТ",
  "Консультант службы поддержки": "не ИТ",
  "Менеджер по В2В продажам": "не ИТ",
  "Оператор Премиального обслуживания": "не ИТ",
  "Руководитель отдела развития бизнеса": "не ИТ",
  "Старший менеджер по продажам среднему и крупному бизнесу": "не ИТ",
  "Специалист электронных заявок": "не ИТ",
  "Оператор по проверке банковских операций": "не ИТ",
  "Специалист Департамента Клиентского Обслуживания": "не ИТ",
  "Специалист технической поддержки бизнеса": "Администрирование",
  "Специалист отдела кадров": "не ИТ",
  "Менеджер по работе со средним бизнесом": "не ИТ",
  "Специалист сектора сопровождения комплаенс проверок": "не ИТ",
  "Руководитель отдела по корпоративным клиентам среднего бизнеса": "не ИТ",
  "Старший инженер": "Разработка",
  "Младший специалист ПОД/ФТ ФРОМУ": "не ИТ",
  "Оператор чат-поддержки": "не ИТ",
  "Менеджер по обучению": "не ИТ",
  "Менеджер по холодным продажам": "не ИТ",
  "Специалист отдела обслуживания депозитных продуктов Управление поддержки клиентов Департамент клиентского обслуживания": "не ИТ",
  "Специалист по кредитным картам": "не ИТ",
  "Инженер 2-й линии поддержки": "Администрирование",
  "Заместитель руководителя группы": "не ИТ",
  "Product Marketing Manager": "не ИТ",
  "Оператор входящей линии": "не ИТ",
  "Специалист обработки данных финансового мониторинга": "Аналитика данных/ Data Science",
  "Разметчик": "Аналитика данных/ Data Science",
  "Эксперт обслуживания бизнеса": "не ИТ",
  "Антифрод": "не ИТ",
  "Product Lead в направлении RnD Т-Бизнеса": "Разработка",
  "МТК(менеджер по товарному кредитованию)": "не ИТ",
  "Cпециалист взыскания задолженности по телефону": "не ИТ",
  "Старший Java/Kotlin разработчик": "Разработка",
  "Доставщик продукции банка": "не ИТ",
  "Консультант автокредитования": "не ИТ",
  "Специалист по анти-фрод поддержке клиентов": "не ИТ",
  "Ведущий менеджер по работе со средним бизнесом": "не ИТ",
  "Куратор": "не ИТ",
  "Ведущий андеррайтер": "не ИТ",
  "Интент-аналитик": "Аналитика данных/ Data Science",
  "Непрерывное обслуживание бизнеса": "не ИТ",
  "Главный специалист отдела инвестиционных сделок": "не ИТ",
  "Менеджер по качеству": "не ИТ",
  "Консультант на линии помощи": "не ИТ",
  "Android-разработчик": "Разработка",
  "Оператор по упаковке карт": "не ИТ",
  "Специалист взыскания задолженности по телефону": "не ИТ",
  "IOS разработчик": "Разработка",
  "Эксперт по кредитным картам": "не ИТ",
  "Менеджер по подбору ключевых сотрудников": "не ИТ",
  "Главный менеджер по работе со средним и крупным бизнесом": "не ИТ",
  "Наставник": "не ИТ",
  "Менеджер по продажам бизнес продуктов": "не ИТ",
  "Руководитель группы разработки Руководитель направления": "Разработка",
  "QA/Специалист по автоматизации": "Тестирование",
  "Специалист антифрод поддержки клиентов": "не ИТ",
  "Scala-разработчик": "Разработка",
  "Специалист сопровождения банковских проверок": "не ИТ",
  "Специалист по обработке входящих обращений": "не ИТ",
  "Заместитель руководителя": "не ИТ",
  "Специалист клиентского сервиса": "не ИТ",
  "Специалист по продаже услуг бизнесу": "не ИТ",
  "Тренер": "не ИТ",
  "Инженер по тестированию ПО": "Тестирование",
  "Тестировщик-автоматизатор": "Тестирование",
  "Руководитель развития клиентов среднего и крупного бизнеса": "не ИТ",
  "Младший HR Бизнес-партнер": "не ИТ",
  "Персональный инвестиционный менеджер": "не ИТ",
  "Главный эксперт по инвестициям": "не ИТ",
  "Специалист по  адаптации новых сотрудников": "не ИТ",
  "Старший оператор входящих звонков": "не ИТ",
  "Менеджер по региональному развитию и логистике": "не ИТ",
  "Менеджер клиентской поддержки": "не ИТ",
  "Региональный Руководитель направления продаж ТОП сегмента": "не ИТ",
  "Специалист страхования": "не ИТ",
  "Главный кредитный инспектор отдела реализации залогового имущества": "не ИТ",
  "Оператор по продажам бизнесу": "не ИТ",
  "Младший специалист направления подготовки исходящей корреспонденции": "не ИТ",
  "Ведущий специалист сектора обслуживания премиальных клиентов": "не ИТ",
  "Руководитель группы адаптации": "не ИТ",
  "Кредитный представитель": "не ИТ",
  "Специалист сектора сопровождения клиентов на бухгалтерском обслуживании Отдел аутсорсных услуг для бизнеса Управление клиентской поддержки бизнеса Департамент клиентского обслуживания": "не ИТ",
  "Шеф-редактор": "не ИТ",
  "Старший менеджер по развитию среднего и крупного бизнеса": "не ИТ",
  "Кредитный авто инспектор": "не ИТ",
  "Асессор": "не ИТ",
  "Эксперт комплаенс-проверок": "не ИТ",
  "Руководитель": "не ИТ",
  "Выездной специалист": "не ИТ",
  "Специалист направления регистрации корреспонденции сектора документооборота": "не ИТ",
  "Ведущий специалист отдела кредитных карт": "не ИТ",
  "Оператор продажи банковских продуктов": "не ИТ",
  "Комплаенс-специалист по мониторингу операций физических лиц AML": "не ИТ",
  "Team leader": "Разработка",
  "Территориальный руководитель по работе с партнерами": "не ИТ",
  "Специалист по разметки данных": "Аналитика данных/ Data Science",
  "Специалист отдела телемаркетинга": "не ИТ",
  "Windows developer": "Разработка",
  "QA инженер по обеспечению качества": "Тестирование",
  "Налоговый специалист": "не ИТ",
  "Специалист сектора обслуживания дебетовых карт": "не ИТ",
  "Compliance АМЛ тренер обучения": "не ИТ",
  "Главный специалист отдела обслуживания депозитных продуктов": "не ИТ",
  "Специалист обработки входящих звонков управления поддержки клиентского обслуживания": "не ИТ",
  "Ведущий менеджер отдела продаж": "не ИТ",
  "Комплаенс специалист": "не ИТ",
  "IT рекрутер": "не ИТ",
  "Обработчик  данных": "Аналитика данных/ Data Science",
  "Оператор клиентской поддеркжи": "не ИТ",
  "Ведущий эксперт по работе с клиентами": "не ИТ",
  "Ведущий менеджер по работе Среднего и Крупного Бизнеса (Корпоративный блок)": "не ИТ",
  "Оператор направления универсальные продажи": "не ИТ",
  "Консультант по банковским продуктом": "не ИТ",
  "Специалист. Группы непрерывного обслуживания Секторов обработки входящих чатов. Отдел по обслуживанию банковских продуктов. Управление поддержки клиентов Департамент клиентского обслуживания": "не ИТ",
  "Senior project manager": "не ИТ",
  "Специалист по контролю качества": "не ИТ",
  "Специалист (2024); Менеджер агентской группы (2024); Ведущий эксперт (2021-2024)": "не ИТ",
  "Младший специалист Т-бизнес": "не ИТ",
  "Аналитик SOC": "Администрирование",
  "Ведущий инженер SRE": "Администрирование",
  "Оператор по обработке документов": "не ИТ",
  "Руководитель направления развития платежей": "не ИТ",
  "Team lead/ Product manager": "Разработка",
  "Оператор контроля качества": "не ИТ",
  "Методист образовательных программ": "не ИТ",
  "Специалист отдела досудебного взыскания": "не ИТ",
  "IT-рекрутер": "не ИТ",
  "Специалист по доставке банковских продуктов": "не ИТ",
  "Специалист по противодействию мошенничеству": "не ИТ",
  "Главный менеджер по продажам": "не ИТ",
  "Ведущий продуктовый маркетолог": "не ИТ",
  "Ведущий менеджер по продажам среднему и крупному бизнесу": "не ИТ",
  "Менеджер по туризму": "не ИТ",
  "Тренер по обучению": "не ИТ",
  "Главный менеджер по работе с Малым и Средним бизнесом": "не ИТ",
  "Младший специалист": "не ИТ",
  "Дизайнер": "Дизайн",
  "Специалист управления  поддержки клиентов": "не ИТ",
  "Специалист документооборота": "не ИТ",
  "Оператор обработки данных (розыск счетов)": "Аналитика данных/ Data Science",
  "Специалист поддержки инвестиции": "не ИТ",
  "Специалист входящей линии": "не ИТ",
  "Специалист антифрод-поддержки": "не ИТ",
  "Специалист службы безопасности": "не ИТ",
  "Младший системный аналитик (стажировка)": "Разработка",
  "Ведущий менеджер": "не ИТ",
  "Консультант Тинькофф Инвестиции": "не ИТ",
  "Консультант по продукту": "не ИТ",
  "Начальник отдела продаж": "не ИТ",
  "Менеджер Премиум поддержки": "не ИТ",
  "Менеджер по кредитованию юридических лиц": "не ИТ",
  "Менеджер по продажам услуг": "не ИТ",
  "Ведущий менеджер по развития среднего бизнеса": "не ИТ",
  "Ведущий эксперт бэк-офиса": "не ИТ",
  "Наставник операторов по обработке банковских операций": "не ИТ",
  "Ведущий специалист обслуживания дебетовых продуктов": "не ИТ",
  "Специалист отдела аутсорсинговых услуг": "не ИТ",
  "Специалист по обслуживанию клиентов": "не ИТ",
  "Главный эксперт сектора обслуживания дебетовых карт": "не ИТ",
  "Специалист онлайн-поддержки": "не ИТ",
  "Партнер управления продаж": "не ИТ",
  "Старший специалист, наставник": "не ИТ",
  "Редактор": "не ИТ",
  "Менеджер отдела продаж": "не ИТ",
  "Консультант обслуживания бизнеса": "не ИТ",
  "Менеджер Т-страхование": "не ИТ",
  "VIP-менеджер по работе с юридическими лицами": "не ИТ",
  "Оператор отдела входящих обращений": "не ИТ",
  "Ведущий специалист по электронной корреспонденции": "не ИТ",
  "Оператор поддержки клиентов": "не ИТ",
  "Специалист профильного подбора": "не ИТ",
  "Delivery manager": "Администрирование",
  "Middle Product Designer": "Дизайн",
  "Главный специалист": "не ИТ",
  "Финансовый менеджер": "не ИТ",
  "Менеджер по работе с  юридическими лица": "не ИТ",
  "Инженер техподдержки": "Администрирование",
  "Аналитик процессов": "не ИТ",
  "Кредитный специалист взыскания": "не ИТ",
  "Руководитель отдела видеографики": "Дизайн",
  "Специалист отдела сканирования": "не ИТ",
  "Специалист по подбору персонала": "не ИТ",
  "Архитектор программного обеспечения": "Разработка",
  "Бухгалтер по расчету заработной платы": "не ИТ",
  "Менеджер по выдаче карт": "не ИТ",
  "Оператор по продаже инвестиций": "не ИТ",
  "Менеджер по привлечению юридических лиц": "не ИТ",
  "Оператор Т-банка": "не ИТ",
  "Оператор по инвестиционным продуктам": "не ИТ",
  "Оператор чатов": "не ИТ",
  "Специалист по работе с юридическими лицами": "не ИТ",
  "Старший менеджер по работе с клиентами среднего бизнеса": "не ИТ",
  "Специалист сектора бухгалтерского учета": "не ИТ",
  "Руководитель группы адаптации новых сотрудников": "не ИТ",
  "Кредитный инспектор отдела верификации Департамента рисков": "не ИТ",
  "Системный администратор": "Администрирование",
  "Специалист инфоподдержки": "не ИТ",
  "Ведущий персональный менеджер": "не ИТ",
  "Продуктовый дизайнер": "Дизайн",
  "Руководитель отдела клиентского обслуживания": "не ИТ",
  "Младший специалист Сектора непрерывного обслуживания Отдела обслуживания бизнеса": "не ИТ",
  "Ведущий менеджер по продажам среднему бизнесу": "не ИТ",
  "Менеджер по продажам инвестиции": "не ИТ",
  "Менеджер": "не ИТ",
  "Product Analyst": "Аналитика данных/ Data Science",
  "Главный менеджер 1 категории": "не ИТ",
  "Сотрудник поддержки": "не ИТ",
  "Руководитель группы операторов первой линии": "не ИТ",
  "Специалист по сохранению клиентов": "не ИТ",
  "Эксперт Сектора по развитию регионов, Отдел по обслуживанию банковских продуктов Главный специалист Сектора поддержки сервиса Путешествия, Отдел обслуживания нефинансовых продуктов": "не ИТ",
  "Ведущий менеджер по работе с клиентами крупного и среднего бизнеса": "не ИТ",
  "Оператор ООУ": "не ИТ",
  "Эксперт сектора сохранения клиентов": "не ИТ",
  "Специалист сектор усиленной идентификации": "не ИТ",
  "Специалист дебетовых продуктов": "не ИТ",
  "Главный специалист управления рисками": "не ИТ",
  "Руководитель группы Методологии и Документации": "не ИТ",
  "Тимлид": "Разработка",
  "Младший специалист обработки данных": "Аналитика данных/ Data Science",
  "Ведущий Менеджер по работе с крупным бизнесом Средний и крупный бизнес": "не ИТ",
  "Руководитель отдела продаж": "не ИТ",
  "Специалист сектора обслуживания клиентов премиального сервиса": "не ИТ",
  "Эксперт по банковским продуктам": "не ИТ",
  "Заместитель руководителя бизнес-группы": "не ИТ",
  "Ведущий специалист технической поддержки": "Администрирование",
  "Младший специалист по подбору персонала": "не ИТ",
  "Старший инженер по тестированию": "Тестирование",
  "Оператор дистанционного обслуживания дебетовых карт": "не ИТ",
  "Оператор входящих обращений (звонки)": "не ИТ",
  "Младший специалист. Сектор бухгалтерских услуг.": "не ИТ",
  "Специалист ДКЦ": "не ИТ",
  "Ведущий кредитный эксперт": "не ИТ",
  "Lead frontend developer": "Разработка",
  "Менеджер по привлечению клиентов": "не ИТ",
  "Оператор СВК": "не ИТ",
  "Менеджер-комплаенс": "не ИТ",
  "Курьер по доставке документов": "не ИТ",
  "Чат-менеджер": "не ИТ",
  "SRE инженер": "Администрирование",
  "Специалист по работе с входящими обращениями": "не ИТ",
  "Заместитель руководителя управления": "не ИТ",
  "Эксперт отдела обслуживания клиентов": "не ИТ",
  "Специалист службы поддержки пользователей": "Администрирование",
  "Support engineer": "Администрирование",
  "Оператор обработки банковских операций": "не ИТ",
  "Юрист": "не ИТ",
  "Старший администратор региона": "Администрирование",
  "Эксперт по депозитным продуктам": "не ИТ",
  "Старший менеджер привлечения клиентов среднего и крупного бизнеса": "не ИТ",
  "Специалист по проверке документов": "не ИТ",
  "Ведущий специалист СБ": "не ИТ",
  "Развивающий менеджер B2B": "не ИТ",
  "Менеджер по кредитованию": "не ИТ",
  "Специалист по работе с корпоративными клиентами": "не ИТ",
  "Оператор актуализации данных": "Аналитика данных/ Data Science",
  "Специалист обработки входящих обращений": "не ИТ",
  "Менеджер отдела верификации ипотечных продуктов": "не ИТ",
  "Руководитель проекта": "не ИТ",
  "Старший бухгалтер": "не ИТ",
  "Специалист по поддержке клиентов": "не ИТ",
  "Специалист поддержки клиентов отдела Т-инвестиций": "не ИТ",
  "Ведущий  специалист по работе  с отказами госорганов": "не ИТ",
  "Специалист отдела позднего взыскания": "не ИТ",
  "Трафик-менеджер": "не ИТ",
  "Оператор в чатах": "не ИТ",
  "Специалист сектора поддержки клиентов по взаимодействию с государственными органами Отдел сопровождения запросов государственных органов Управление бэк-офис Департамент клиентского обслуживания": "не ИТ",
  "Контролёр качества": "не ИТ",
  "Специалист обработки чатов депозитных продуктов": "не ИТ",
  "Менеджер по продажам бизнесу (удаленно)": "не ИТ",
  "Специалист поддержки сервиса Т Инвестиции": "не ИТ",
  "Оператор по обслуживанию депозитных продуктов": "не ИТ",
  "Сотрудник технической поддержки": "Администрирование",
  "Специалист поддержки пользователей": "Администрирование",
  "Куратор группы": "не ИТ",
  "Инженер по эксплуатации зданий и сооружений": "не ИТ",
  "Менеджер по адаптации (продажи)": "не ИТ",
  "Эксперт по развитию клиентов малого бизнеса": "не ИТ",
  "Специалист поддержки клиентов направление бизнес": "не ИТ",
  "Архитектор процессов": "не ИТ",
  "Главный специалист отдела инвестиционной поддержки": "не ИТ",
  "Оператор чат": "не ИТ",
  "Менеджер по развитию партнёрского направления": "не ИТ",
  "Старший специалист КЦ": "не ИТ",
  "Специалист бэк офиса": "не ИТ",
  "Менеджер чат-поддержки": "не ИТ",
  "Главный специалист финансового мониторинга": "не ИТ",
  "Ведущий специалист сектор бухгалтерских услуг": "не ИТ",
  "Специалист административно-хозяйственного отдела": "не ИТ",
  "Представитель Т-Банк": "не ИТ",
  "Персональный менеджер (VIP-обслуживание)": "не ИТ",
  "Frontend-разработчик": "Разработка",
  "Менеджер по страховке": "не ИТ",
  "Специалист по премиальному обслуживанию": "не ИТ",
  "Сорсер-асессор": "не ИТ",
  "Старший менеджер 1 категории": "не ИТ",
  "Спецаилист": "не ИТ",
  "Руководитель отдела персонала": "не ИТ",
  "Менеджер по работе с B2B клиентами": "не ИТ",
  "Оперативный дежурный": "не ИТ",
  "Ведущий кредитный инспектор отдела реализации залогового имущества": "не ИТ",
  "Senior Java Developer": "Разработка",
  "Персональный водитель": "не ИТ",
  "Оператор Т Мобайл": "не ИТ",
  "Старший специалист поддержки процессов": "не ИТ",
  "Специалист по выдаче банковских гарантий": "не ИТ",
  "Специалист по управлению персоналом": "не ИТ",
  "Business Analyst": "не ИТ",
  "Продавец-консультант": "не ИТ",
  "Business Development Manager (BDM)": "не ИТ",
  "Персональный менеджер по обслуживанию бизнес-клиентов SME": "не ИТ",
  "Представитель банка Т": "не ИТ",
  "Ведущий персональный менеджер по инвестициям 2-го уровня": "не ИТ",
  "Frontend разработчик": "Разработка",
  "Ведущий менеджер по инвестициям": "не ИТ",
  "Территориальный руководитель": "не ИТ",
  "Специалист входящих обращений": "не ИТ",
  "Старший менеджер по работе со средним и крупным бизнесом": "не ИТ",
  "QA automation": "Тестирование",
  "Главный клиентский менеджер среднего и крупного бизнеса": "не ИТ",
  "Ведущий специалист бэк-офиса": "не ИТ",
  "Специалист Сектора поддержки путешествий в Отделе поддержки клиентских операций и сервисов. Специалист Управления бэк-офисом, Департамента клиентского обслуживания": "не ИТ",
  "Менеджер по работе с агентами": "не ИТ",
  "Руководитель отдела претензий и коммуникаций": "не ИТ",
  "Старший специалист центра мониторинга": "не ИТ",
  "Team lead": "Разработка",
  "Оператор упаковки": "не ИТ",
  "Старший кредитный инспектор": "не ИТ",
  "Специалист по сопровождению": "не ИТ",
  "Старший тьютор": "не ИТ",
  "Младший java-разработчик": "Разработка",
  "Консультант по продажам": "не ИТ",
  "Менеджер продаж банковских продуктов": "не ИТ",
  "Младший специалист поддержки бизнеса": "не ИТ",
  "Оператор Т-страхование": "не ИТ",
  "Специалист отдела обслуживания": "не ИТ",
  "Специалист отдела продаж": "не ИТ",
  "Менеджер call-центра": "не ИТ",
  "Менеджер проектов": "не ИТ",
  "Специалист отдела сохранения клиентов": "не ИТ",
  "Менеджер отдела обслуживания клиентов": "не ИТ",
  "Специалист по развитию": "не ИТ",
  "Менеджер отдела": "не ИТ",
  "Менеджер-оператор": "не ИТ",
  "Старший software Developer": "Разработка",
  "Главный специалист клиентской поддержки": "не ИТ",
  "Ведущий менеджер по подбору персонала": "не ИТ",
  "Старший руководитель отдела клиентского обслуживания": "не ИТ",
  "Ведущий специалист отделения банка": "не ИТ",
  "Консультант Тинькофф Инвестиций": "не ИТ",
  "Менеджер по продаже банковских услуг": "не ИТ",
  "Младший специалист обработке данных": "Аналитика данных/ Data Science",
  "Оператор электронных заявок в отделе обработки данных": "не ИТ",
  "Ночной специалист клиентской поддержки": "не ИТ",
  "Оператор видеонаблюдения": "не ИТ",
  "Специалист сектора по развитию регионов": "не ИТ",
  "Специалист отдела верификации": "не ИТ",
  "Администратор баз данных": "Администрирование",
  "Специалист по поддержке бизнес-клиентов": "не ИТ",
  "Финансовый консультант": "не ИТ",
  "Менеджер по развитию малого и среднего бизнеса": "не ИТ",
  "Product Manager": "не ИТ",
  "Бухгалтер на удаленке": "не ИТ",
  "Руководитель группы клиентской поддержки / Менеджер по развитию персонала": "не ИТ",
  "Оператор отдела верификации": "не ИТ",
  "Старший клиентский менеджер отдела развития бизнеса": "не ИТ",
  "Ведущий менеджер по привлечению среднего-крупного бизнеса": "не ИТ",
  "Специалист по работе с премиум клиентами": "не ИТ",
  "Администратор систем обучения": "Администрирование",
  "Младший специалист отдела бухгалтерского обслуживания": "не ИТ",
  "Менеджер Т-Бизнес": "не ИТ",
  "Оператор РК": "не ИТ",
  "Специалист по по сохранению клиентов": "не ИТ",
  "Оператор обработки электронных заявок": "не ИТ",
  "Старший специалист по документообороту": "не ИТ",
  "Заместитель руководителя отдела переводов": "не ИТ",
  "Менеджер по адаптации": "не ИТ",
  "Ведущий бизнес-аналитик": "не ИТ",
  "Оператор по раннему взысканию": "не ИТ",
  "Менеджер поддержки сотрудников в регионе": "не ИТ",
  "Специалист по антифрод поддержке": "не ИТ",
  "Business Information Security Officer (BISO) - Security Partner": "Администрирование",
  "Техник": "не ИТ",
  "Senior iOS Developer": "Разработка",
  "Старший Дизайнер": "Дизайн",
  "Специалист по контент-маркетингу": "не ИТ",
  "Ведущий специалист клиентского обслуживания": "не ИТ",
  "CRM-маркетолог, стажер": "не ИТ",
  "Оператор входящей линии обслуживание": "не ИТ",
  "Старший специалист по работе с клиентами": "не ИТ",
  "Консультант колл-центра": "не ИТ",
  "UX-редактор b2b": "Дизайн",
  "Специалист сектора контроля качества": "не ИТ",
  "Консультант поддержки Т-бизнес": "не ИТ",
  "Менеджер по развитию": "не ИТ",
  "Специалист поддержки Т-инвестиции": "не ИТ",
  "Оператор по заполнению электронных заявок": "не ИТ",
  "Предстааитель": "не ИТ",
  "Главный менеджер по привлечению корпоративных клиентов.": "не ИТ",
  "Торговый представитель отдела продаж": "не ИТ",
  "Специалист домашнего колл-центра": "не ИТ",
  "Операционный директор": "не ИТ",
  "Специалист по залоговым кредитам": "не ИТ",
  "Менеджер по взысканию": "не ИТ",
  "Консультант отдела обслуживания ДКО": "не ИТ",
  "Менеджер IT-проектов": "Администрирование",
  "Руководитель направления по работе с партнерами М сегмент": "не ИТ",
  "Ведущий специалист группы поддержки состоятельных клиентов": "не ИТ",
  "Оператор горячей линии": "не ИТ",
  "Руководитель группы логистики": "не ИТ",
  "Младший специалист Направление непрерывного обслуживания бизнеса Отдел сопровождения комплаенс проверок Управление клиентской поддержки бизнеса Департамент клиентского обслуживания": "не ИТ",
  "Go developer": "Разработка",
  "Ведущий специалист по сопровождению": "не ИТ",
  "Специалист по документообороту и разметки данных для ИИ": "Аналитика данных/ Data Science",
  "Менеджер по развитию действующих клиентов": "не ИТ",
  "Оператор бизнеса": "не ИТ",
  "Аналитик бизнес-процессов": "не ИТ",
  "Категорийный менеджер по закупкам": "не ИТ",
  "Cпециалист Т-страхование": "не ИТ",
  "QA Fullstack Engineer": "Тестирование",
  "Территориальный руководитель управления по работе с юридическими лицами": "не ИТ",
  "Оператор в направлении Инвестиции": "не ИТ",
  "Руководитель офиса": "не ИТ",
  "Консультант домашнего колл-центра": "не ИТ",
  "Специалист чат-поддержки": "не ИТ",
  "Оператор по обработке входящих заявок": "не ИТ",
  "Специалист по обслуживанию банковских продуктов физических лиц, входящий чат": "не ИТ",
  "Оператор раннего взыскаяния": "не ИТ",
  "Специалист Сектора противодействия мошенничеству Отдел кросс-функционального обслуживания Управление поддержки клиентов Департамент клиентского обслуживания": "не ИТ",
  "Ведущий специалист премиальной поддержки": "не ИТ",
  "Оператор в дистанционный колл-центр по ДК": "не ИТ",
  "Техник АХО": "не ИТ",
  "Менеджер по страхованию": "не ИТ",
  "Cпециалист поддержки Т-инвестиции": "не ИТ",
  "Младший специалист во фрод-мониторинге": "не ИТ",
  "Персональный менеджер по инвестициям": "не ИТ",
  "SEO-специалист": "не ИТ",
  "Оператор сопровождения процессов взыскания": "не ИТ",
  "Оператор отдела продаж": "не ИТ",
  "Ассесор": "не ИТ",
  "Удаленный оператор ввода данных": "Аналитика данных/ Data Science",
  "Руководитель группы привлечения клиентов среднего и крупного бизнеса": "не ИТ",
  "Brand manager": "не ИТ",
  "Оператор обработки документов": "не ИТ",
  "IT Сорсер": "не ИТ",
  "Ведущий аналитик процессов": "не ИТ",
  "Младший аналитик": "Аналитика данных/ Data Science",
  "Специалист отдел взыскания задолженности": "не ИТ",
  "Старший рекрутер": "не ИТ",
  "Оператор группы продаж": "не ИТ",
  "SMM-дизайнер": "Дизайн",
  "Программист Python": "Разработка",
  "Младший продюсер": "не ИТ",
  "Младший бухгалтер": "не ИТ",
  "Специалист группы сопровождения": "не ИТ",
  "Ведущий специалист по расчету заработной платы": "не ИТ",
  "Оператор по продажам B2B": "не ИТ",
  "Оператор по инвестициям": "не ИТ",
  "Специалист по кредитным продуктам": "не ИТ",
  "Старший эксперт отдела оценки": "не ИТ",
  "Руководитель группы развития крупного и среднего бизнеса": "не ИТ",
  "Оператор заявок": "не ИТ",
  "Руководитель направления развития клиентов партнерского сегмента": "не ИТ",
  "Специалист по работе с залогами": "не ИТ",
  "Специалист первой линии поддержки": "не ИТ",
  "Старший финансовый аналитик": "не ИТ",
  "Оператор на чатах в службе поддержки": "не ИТ",
  "Менеджер по работе с клиентами крупного и среднего бизнеса": "не ИТ",
  "Руководитель смены группы развития качества ареста счетов": "не ИТ",
  "Специалист поддержки клиентов в отделе обслуживания депозитных продуктов": "не ИТ",
  "Контролер ОТК": "не ИТ",
  "Специалист урегулирования убытков КАСКО": "не ИТ",
  "Руководитель группы сметного расчета": "не ИТ",
  "Оператор первой линии": "не ИТ",
  "Главный аналитик процессов": "не ИТ",
  "Специалист по анти-фрод поддержке": "не ИТ",
  "Менеджер по сопровождению клиентов": "не ИТ",
  "Руководитель региональных продаж (ЦФО)": "не ИТ",
  "Менеджер по развитию крупного и среднего бизнеса": "не ИТ",
  "Кредитный консультант": "не ИТ",
  "DevOPS Инженер": "Администрирование",
  "Главный инвестиционный консультант": "не ИТ",
  "Специалист учёта IT оборудования(ЦОД)": "Администрирование",
  "Специалист по назначению встреч": "не ИТ",
  "Менеджер по работе с партнерами": "не ИТ",
  "Специалист по урегулированию убытков": "не ИТ",
  "Специалист по доставке карт": "не ИТ",
  "Оператор обратоки данных": "Аналитика данных/ Data Science",
  "Руководитель отдела обучения Т-Бизнес": "не ИТ",
  "Оператор премиум сервиса": "не ИТ",
  "Специалист службы технической поддержки": "Администрирование",
  "Кредитный менеджер по развитию малого бизнеса": "не ИТ",
  "Senior QA инженер": "Тестирование",
  "Старший оператор": "не ИТ",
  "Специалист отдела обслуживания бизнес клиентов": "не ИТ",
  "Инженер данных": "Аналитика данных/ Data Science",
  "Специалист отдела ОВО": "не ИТ",
  "Специалист по продажам малому бизнесу": "не ИТ",
  "Специалист по анти фрод поддержке": "не ИТ",
  "Руководитель группы в отделе обслуживания состоятельных клиентов": "не ИТ",
  "Руководитель группы продаж малому бизнесу": "не ИТ",
  "Менеджер продаж потребительских кредитов": "не ИТ",
  "Старший специалист поддержки клиентов": "не ИТ",
  "Архивариус": "не ИТ",
  "Непрерывное обслуживание": "не ИТ",
  "Ведущий специалист сопровождения кандидатов": "не ИТ",
  "Специалист колл центра": "не ИТ",
  "Оператор облачного сервиса": "Администрирование",
  "Сотрудник страховой": "не ИТ",
  "Оператор проверки сообщений, актуализации данных": "не ИТ",
  "Старший специалист отдела по работе с клиентами": "не ИТ",
  "Вечерний консультант по продуктам для бизнеса в регионах России": "не ИТ",
  "Специалист по взысканию и просроченной задолженности": "не ИТ",
  "Ведущий менеджер по партнерскому маркетингу": "не ИТ",
  "Оператор контакт-центра": "не ИТ",
  "Старший менеджер по работе с клиентами среднего и крупного бизнеса": "не ИТ",
  "Руководитель направления сопровождения сбоев": "Администрирование",
  "Кредитный менеджер": "не ИТ",
  "Senior QA": "Тестирование",
  "Специалист по работе с состоятельными клиентами": "не ИТ",
  "Менеджер по привлечению корпоративных клиентов": "не ИТ",
  "Начальник производственного отдела": "не ИТ",
  "Старший инженер-программист": "Разработка",
  "QA Backend": "Тестирование",
  "Главный специалист по банковским продуктам": "не ИТ",
  "Руководитель группы сотрудников отдела обслуживания малого и среднего бизнеса": "не ИТ",
  "Торговый представитель по работе с ключевыми клиентами": "не ИТ",
  "Менеджер по продаже интернет-эквайринга": "не ИТ",
  "Специалист сектора финансовых корректировок": "не ИТ",
  "Специалист непрерывного обслуживания отдела бизнес": "не ИТ",
  "Контролер качества": "не ИТ",
  "Специалист по развитию качества сервиса": "не ИТ",
  "Growth Product Manager": "не ИТ",
  "Оператор по верификации задолженностей": "не ИТ",
  "Ведущий специалист банка": "не ИТ",
  "Руководитель группы продаж среднему и крупному бизнесу": "не ИТ",
  "Главный эксперт в развитии качества инвестиций": "не ИТ",
  "Технолог": "не ИТ",
  "Старший инженер по автоматизации": "Разработка",
  "Специалист технической поддержки Инвестиции": "Администрирование",
  "Ведущий специалист отдела аутсорсных услуг бухгалтерского учёта": "не ИТ",
  "Cпециалист по продажам бизнесу": "не ИТ",
  "Главный специалист группы корпоративной поддержки": "не ИТ",
  "Руководитель отдела по работе с клиентами": "не ИТ",
  "QA-engineer": "Тестирование",
  "Младший специалист банка": "не ИТ",
  "Андеррайтер": "не ИТ",
  "Специалист по обработке цифровых обращений": "не ИТ",
  "Руководитель команды": "не ИТ",
  "Senior System analyst / Team lead": "Разработка",
  "Специалист по разработке скриптов продаж": "не ИТ",
  "Редактор-продюсер": "не ИТ",
  "Операторпо продаже потребительских кредитов, оператор чата по продаже кредитов, оператор чата по поддержке клиентов по инвестициям": "не ИТ",
  "Дебетовые счета, оператор входящих обращений": "не ИТ",
  "Специалист обработки обращений": "не ИТ",
  "Специалист в Отделе обслуживания депозитных продуктов Управление поддержки клиентов": "не ИТ",
  "Методолог HR-бренда": "не ИТ",
  "Вечерний консультант по продуктам": "не ИТ",
  "Ведущий специалист по мониторингу операций (AML) Compliance": "не ИТ",
  "Ведущий специалист бэк-офиса (Инвестиции)": "не ИТ",
  "Менеджер по продажам банковских услуг": "не ИТ",
  "Оператор(клекс)": "не ИТ",
  "Оператор на домашнем телефоне": "не ИТ",
  "Специалист отдела электронных обращений": "не ИТ",
  "Ведущий менеджер по привлечению корпоративных клиентов": "не ИТ",
  "Младший инженер-программист": "Разработка",
  "Эксперт по работе с клиентами": "не ИТ",
  "Инвестиционный консультант Private Banking": "не ИТ",
  "Специалист телемаркетинга": "не ИТ",
  "Оператор отдела ПК": "не ИТ",
  "VIP-представитель, наставник": "не ИТ",
  "Cпециалист сопровождения процессов взыскания": "не ИТ",
  "Head of PMO": "не ИТ",
  "Менеджер по продажам кредитных продуктов банка": "не ИТ",
  "Сейлз-менежер": "не ИТ",
  "Специалист сопровождения процессов взыскания": "не ИТ",
  "Программист-разработчик": "Разработка",
  "Консультант по продуктам": "не ИТ",
  "Специалист по взысканию задолженности по телефону": "не ИТ",
  "Ведущий специалист по работе с премиальными клиентами": "не ИТ",
  "Специалист отдела урегулирования убытков": "не ИТ",
  "Ведущий юрист по исполнительному производству": "не ИТ",
  "Консультант.": "не ИТ",
  "Менеджер по продаже банковских продуктов": "не ИТ",
  "Премиум менеджер": "не ИТ",
  "Специалист обслуживания техники": "не ИТ",
  "Специалист ипотечного страхования": "не ИТ",
  "Специалист по обеспечению качества": "Тестирование",
  "Специалист группы обеспечения": "не ИТ",
  "Менеджер по контент-маркетингу": "не ИТ",
  "Водитель-экспедитор": "не ИТ",
  "Маркетинговый аналитик": "не ИТ",
  "Ведущий менеджер по продажам малому бизнесу": "не ИТ",
  "Комплаенс аналитик": "не ИТ",
  "Менеджер продуктов": "не ИТ",
  "Разработчик .net": "Разработка",
  "Персональный Менеджер по Инвестиционным продуктам": "не ИТ",
  "Специалист по продуктам для бизнеса": "не ИТ",
  "Сотрудник чата поддержки Тинькофф": "не ИТ",
  "Менеджер по лизингу": "не ИТ",
  "Младший специалист отдела сопровождения комплаенс проверок": "не ИТ",
  "Специалист отдела сопровождения": "не ИТ",
  "Младший консультант по банковским продуктам": "не ИТ",
  "Консультант ДКО": "не ИТ",
  "Ведущий эксперт сектора персонального обслуживания корпоративного бизнеса": "не ИТ",
  "Отдела продаж": "не ИТ",
  "Оператор по вводу данных": "Аналитика данных/ Data Science",
  "Менеджер Инвестиций": "не ИТ",
  "Консультант Тинькофф бизнес": "не ИТ",
  "Менеджер АХО": "не ИТ",
  "Консультант отдела обслуживания бизнеса": "не ИТ",
  "Руководитель эксплуатации": "Администрирование",
  "Junior Аналитик": "Аналитика данных/ Data Science",
  "Главный специалист кредитного отдела": "не ИТ",
  "Специалист управления обработки вызовов": "не ИТ",
  "Оператор по работе с премиальными клиентами": "не ИТ",
  "Ведущий специалист поддержки": "не ИТ",
  "Оператор дистанционной обработки данных.": "не ИТ",
  "Региональный менеджер по продажам": "не ИТ",
  "Бухгалтер по восстановлению учёта ИП": "не ИТ",
  "Customer Support Consultant": "не ИТ",
  "Специалист по обслуживанию": "не ИТ",
  "Клиент-менеджер": "не ИТ",
  "Специалист по продажам автомобилей": "не ИТ",
  "Консультант дистанционного колл-центра": "не ИТ",
  "B&B": "не ИТ",
  "Консультант по продуктам для бизнеса": "не ИТ",
  "Специалист отдела фрод-мониторинга (платежи и переводы)": "не ИТ",
  "Cпециалист ДКО": "не ИТ",
  "Senior Software engineer": "Разработка",
  "Cпециалист по обслуживанию клиентов": "не ИТ",
  "Оператор банковских продуктов": "не ИТ",
  "Младший разработчик": "Разработка",
  "Представитель  банка": "не ИТ",
  "Менеджер группы офисных услуг": "не ИТ",
  "Менеджер по POS кредитованию": "не ИТ",
  "Специалист по работе с партнерами": "не ИТ",
  "Product-designer": "Дизайн",
  "Главный специалист по реализации залогового имущества": "не ИТ",
  "Специалист колл-центра": "не ИТ",
  "Оператор по страховым продуктам": "не ИТ",
  "Представитель банка по работе с бизнесом": "не ИТ",
  "Специалист контент-маркетинга": "не ИТ",
  "Кредитный спициалист": "не ИТ",
  "Специалист направления регистрации корреспонденции": "не ИТ",
  "Менеджер по развитию сети банкоматов": "не ИТ",
  "Cпециалист отдела взыскания": "не ИТ",
  "Специалист кредитных продуктов": "не ИТ",
  "Оператор удаленной обработки данных": "Аналитика данных/ Data Science",
  "Оператор по заполнению заявок": "не ИТ",
  "Оператор по обработке электронных заявок": "не ИТ",
  "Менеджер по продажам эквайринга крупному бизнесу": "не ИТ",
  "Эксперт отдела обслуживания бизнеса": "не ИТ",
  "Специалист отдела развития и контроля качества в направлении дебетовых продуктов": "не ИТ",
  "Менеджер по развитию партнерских проектов": "не ИТ",
  "Оператор по работе с ранней задолженностью": "не ИТ",
  "Логист-кассир": "не ИТ",
  "Специалист по продаже кредитных продуктов": "не ИТ",
  "Frontend разработчик React": "Разработка",
  "Эксперт-аналитик": "Аналитика данных/ Data Science",
  "Специалист поддержки по дебетовым картам": "не ИТ",
  "Специалист обслуживания": "не ИТ",
  "Специалист Т-страхования": "не ИТ",
  "Оператор чат поддержки": "не ИТ",
  "Оператор чата поддержки клиентов": "не ИТ",
  "Аналитик продаж": "не ИТ",
  "Оператор ПК сканирования": "не ИТ",
  "Менеджер по развитию юридических лиц": "не ИТ",
  "Менеджер по работе с юридическими лицами в чате": "не ИТ",
  "Эксперт команды Т-Банк по обслуживанию клиентов в сфере депозитных продуктов": "не ИТ",
  "Старший представитель": "не ИТ",
  "Специалист по бронированию": "не ИТ",
  "Оператор подразделения домашнего колл-центра": "не ИТ",
  "Продакт-менеджер": "не ИТ",
  "Менеджер по продажам автомобилей": "не ИТ",
  "Контент-маркетолог (стажировка)": "не ИТ",
  "Региональный агентский менеджер": "не ИТ",
  "Эксперт чата поддержки": "не ИТ",
  "Ночной оператор электронных заявок": "не ИТ",
  "Менеджер по развитию бизнес-клиентов": "не ИТ",
  "Менеджер банковских продуктов": "не ИТ",
  "Младший специалист непрерывного обслуживания по комплаенс сопровождению": "не ИТ",
  "Опаратор обработки заявок": "не ИТ",
  "Оператор отдела телемаркетинга": "не ИТ",
  "Руководитель группы операторов": "не ИТ",
  "Специалист сопровождения договорных отношений": "не ИТ",
  "Андроид разработчик": "Разработка",
  "Представитель Департамента бизнес-технологий и операций": "не ИТ",
  "Ведущий специалист по тестированию": "Тестирование",
  "Рекрутер": "не ИТ",
  "Java-разработчик, Tech Lead": "Разработка",
  "Менеджер по продажам потребительских кредитов": "не ИТ",
  "Персональный менеджер по работе с юр. лицами": "не ИТ",
  "Младший бизнес-аналитик": "не ИТ",
  "Оператор по продаже страхования от несчастных случаев": "не ИТ",
  "Оператор сканирования": "не ИТ",
  "Менеджер по продажам среднему и крупному бизнесу": "не ИТ",
  "Консультант Т-инвестиций ДКО": "не ИТ",
  "Представитель банка в городе Саров": "не ИТ",
  "Главный специалист отдела развития процессов": "не ИТ",
  "Оператор по обработке дистанционных данных": "не ИТ",
  "Старший аналитик": "Аналитика данных/ Data Science",
  "Оператор входящей линии помощи": "не ИТ",
  "Delivery-manager": "Администрирование",
  "Software Engineer": "Разработка",
  "Главный менеджер по развитию клиентов среднего и крупного бизнеса": "не ИТ",
  "Оператор ПК вскрытие и регистрация входящей корреспонденции": "не ИТ",
  "оператор ДКЦ(2 ступени)": "не ИТ",
  "Ведущий специалист по кредитным картам": "не ИТ",
  "Руководитель проектной группы": "не ИТ",
  "Руководитель направления стратегии и макро": "не ИТ",
  "Представитель Тинькофф": "не ИТ",
  "Менеджер проекта/Руководитель проектов": "не ИТ",
  "Менеджер по инвестициям, менеджер b2b": "не ИТ",
  "Директор макрорегиона": "не ИТ",
  "Специалист внутреннего контроля": "не ИТ",
  "Старший менеджер по привлечению корпоративных клиентов среднего и крупного бизнеса": "не ИТ",
  "Менеджер дистанционного колл-центра": "не ИТ",
  "Сотрудник линии непрерывного урегулирования": "не ИТ",
  "Эксперт отдела входящих обращений": "не ИТ",
  "Специалист по охране труда": "не ИТ",
  "Главный менеджер Отдела продаж и развития среднего и крупного бизнеса": "не ИТ",
  "Специалист по расчёту вознаграждения": "не ИТ",
  "Оператор по сохранению клиентов ДКО": "не ИТ",
  "специалист Т-страхование ДКО": "не ИТ",
  "Менеджер по привлечению юридических лиц в чате": "не ИТ",
  "Специалист поддержки бизнесс-клиентов": "не ИТ",
  "Специалист отдела сохранения": "не ИТ",
  "ML Engineer": "Аналитика данных/ Data Science",
  "Оператор упаковки карт": "не ИТ",
  "Менеджер по продаже потребительских кредитов": "не ИТ",
  "Консультант Т-страхование ОВО": "не ИТ",
  "Оператор страхования": "не ИТ",
  "Консультант т-инвестиции отдел электронных обращений": "не ИТ",
  "Специалист сектора входящих чатов и звонков": "не ИТ",
  "Специалист по урегулированию убытков НС": "не ИТ",
  "Инженер по обеспечению качества (мобильное приложение)": "Тестирование",
  "Поддержка.Дебетовые карты.": "не ИТ",
  "Инспектор отдела кадров": "не ИТ",
  "Менеджер по работе с партнерами POS-кредитование": "не ИТ",
  "Менеджер по продажам инвестиционных продуктов": "не ИТ",
  "Специалист Департамента по обслуживанию клиентов": "не ИТ",
  "Специалист досудебного взыскания отдела реструктуризации": "не ИТ",
  "Персональный менеджер по адаптации юридических лиц": "не ИТ",
  "Руководитель группы управления развития качества": "не ИТ",
  "Старший специалист Центра экосистемой  безопасности Отдела усиленной идентификации": "не ИТ",
  "Менеджер по продажам инвестиций": "не ИТ",
  "Специалист отдела раннего взыскания": "не ИТ",
  "Оператор по просроченной задолженности": "не ИТ",
  "Оператор по продажам физ. лицам": "не ИТ",
  "Консультант Т-страхование ДОК": "не ИТ",
  "Финансовый аналитик": "не ИТ",
  "Специалист по вводу данных": "Аналитика данных/ Data Science",
  "Менеджер по корпоративной поддержки": "не ИТ",
  "Предстовитель банка": "не ИТ",
  "Представитель УСП": "не ИТ",
  "Мобильный разработчик Android платформенной команды": "Разработка",
  "Специалист мониторинга и сопровождения операций (клиентских сервисов)": "не ИТ",
  "Консультант Т-страхование Отдел входящих обращений": "не ИТ",
  "Представитель бизнес технологий и операций": "не ИТ",
  "Менеджер розничных продаж": "не ИТ",
  "Разработчик электронных курсов": "Разработка",
  "Консультант Т-страхование ДКО": "не ИТ",
  "Оператор по сохранению кредитных карт": "не ИТ",
  "Главный менеджер по привлечению корпоративных клиентов средний и крупный бизнес": "не ИТ",
  "оператор клиентской поддержки тинькофф": "не ИТ",
  "Консультант Т-Банк": "не ИТ",
  "Менеджер по закупкам и снабжению": "не ИТ",
  "Рекрутер IT": "не ИТ",
  "Персональный менеджер по развитию бизнес клиентов": "не ИТ",
  "Специалист группы": "не ИТ",
  "Специалист (отдел взыскания)": "не ИТ",
  "Технический специалист": "Администрирование",
  "Эксперт технической поддержки": "Администрирование",
  "PR-специалист": "не ИТ",
  "Консультант (страхование)": "не ИТ",
  "Оператор отдела взыскания в Т-Банке": "не ИТ",
  "Оператор Тинькофф страхование": "не ИТ",
  "Специалист отдела взыскания Т-Банк": "не ИТ",
  "Специалист отдела по обслуживанию юридических лиц": "не ИТ",
  "Менеджер по кадровому администрированию": "не ИТ",
  "Менеджер по продажам для бизнеса": "не ИТ",
  "Представитель Т-банка": "не ИТ",
  "Оператор (страхование)": "не ИТ",
  "Специалист отдела мониторинга": "не ИТ",
  "Начинающий специалист по документообороту": "не ИТ",
  "Оператор электронных заявок автострахование": "не ИТ",
  "Консультант Тинькофф-банк": "не ИТ",
  "Водитель-курьер": "не ИТ",
  "Оператор ПК отдела сканирования": "не ИТ",
  "Аналитик моделей рисков": "Аналитика данных/ Data Science",
  "Оператор отдела обслуживания клиентов": "не ИТ",
  "Руководитель группы премиальной поддержки": "не ИТ",
  "Консультант по страхованию": "не ИТ",
  "Специалист по дебетовым продуктам": "не ИТ",
  "младший специалист по обработке данных": "Аналитика данных/ Data Science",
  "Scala Developer (Lead)": "Разработка",
  "Специалист сектора обучения и контроля качества": "не ИТ",
  "Главный специалист отдела продаж, заместитель руководителя группы отдела продаж": "не ИТ",
  "Амбассадор": "не ИТ",
  "Специалист (Страхование)": "не ИТ",
  "Оператор КЦ": "не ИТ",
  "Кредитный инспектор (отдел верификации)": "не ИТ",
  "Java Team Lead": "Разработка",
  "Специалист по взысканию": "не ИТ",
  "Специалист сектора финансовых потерь": "не ИТ",
  "Консультант по дебетовым картам": "не ИТ",
  "Менеджер по продаже услуг бизнесу": "не ИТ",
  "Инвестиционный менеджер": "не ИТ",
  "HR менеджер": "не ИТ",
  "Ведущий Андеррайтер. Отдел кредитования малого и среднего бизнеса": "не ИТ",
  "Оператор по работе с просроченной задолженностью": "не ИТ",
  "Оператор call-центра и чатов": "не ИТ",
  "Помощник руководителя": "не ИТ",
  "Менеджер отдела корпоративных продаж": "не ИТ",
  "Специалист сектора обработки данных": "Аналитика данных/ Data Science",
  "Специалист по контролю качества/Оператор Управления поддержки клиентов": "не ИТ",
  "Старший разработчик электронных курсов": "Разработка",
  "Юрист в продакшене": "не ИТ",
  "Специалист по клиентскому обслуживанию": "не ИТ",
  "Территориальный менеджер развития партнёрской сети": "не ИТ",
  "Младший специалист документооборота": "не ИТ",
  "Ведущий специалист отдела отчетности": "не ИТ",
  "Оператор электронных обращений": "не ИТ",
  "Главный менеджер по продажам малому и среднему бизнесу": "не ИТ",
  "Верификатор": "не ИТ",
  "Веб-аналитик": "не ИТ",
  "Специалист отдела поддержки депозитных продуктов": "не ИТ",
  "Специалист по планированию": "не ИТ",
  "Эксперт по работе с юрлицами, наставник": "не ИТ",
  "Генеральный  директор": "не ИТ",
  "Администратор построения процессов": "не ИТ",
  "Менеджер по развитию корпоративных клиентов малого бизнеса": "не ИТ",
  "Консультант по детским картам": "не ИТ",
  "Специалист Тинькофф Страхование": "не ИТ",
  "Оператор вскрытия и регистрации корреспонденции": "не ИТ",
  "Консультант банка": "не ИТ",
  "Менеджер активных продаж крупному бизнесу": "не ИТ",
  "Ведущий специалист отдела судебного производства": "не ИТ",
  "Оператор упаковки и отправки": "не ИТ",
  "Младший системный аналитик": "Разработка",
  "Специалист сектора специальных задач": "не ИТ",
  "Представитель банка Тинькофф": "не ИТ",
  "Ведущий специалист отдела обслуживания физических лиц": "не ИТ",
  "Специалист Тинькофф Страхования": "не ИТ",
  "Аналитик-стажер, стажер-координатор": "Аналитика данных/ Data Science",
  "Оператор Т-Банк": "не ИТ",
  "Старший специалист по работе с премиальными клиентами": "не ИТ",
  "Менеджер B2B продаж": "не ИТ",
  "Оператор Тинькофф Страхования": "не ИТ",
  "Представитель Т Банк": "не ИТ",
  "Менеджер по продажам (исходящие Purchase)": "не ИТ",
  "Ведущий менеджер по работе с ключевыми клиентами": "не ИТ",
  "Консультант Тинькофф Страхования": "не ИТ",
  "Инкассатор": "не ИТ",
  "Ведущий специалист поддержки заказчиков": "не ИТ",
  "Специалист по бизнес-процессам": "не ИТ",
  "Специалист 3 категории": "не ИТ",
  "Младший специалист по персоналу": "не ИТ",
  "Специалист управления БЭК": "не ИТ",
  "Специалист отдела по работе с клиентами": "не ИТ",
  "Эксперт отдела развития бизнеса": "не ИТ",
  "Стажер на менеджера по подбору пресонала": "не ИТ",
  "Специалист фрод мониторинга": "не ИТ",
  "Консультант Тинькофф Банка": "не ИТ",
  "Руководитель группы представителей": "не ИТ",
  "менеджер по развитию агентской сети": "не ИТ",
  "Ведущий специалист отдела Compliance": "не ИТ",
  "Руководитель сектора продаж": "не ИТ",
  "Ведущий менеджер по работе с клиентами": "не ИТ",
  "Коснультант Тинькофф Путешествия": "не ИТ",
  "Специалист сектора обслуживания кредитных карт": "не ИТ",
  "Младший SRE-инженер": "Администрирование",
  "Менеджер по работе с клиентами ( юридические и физические лица)": "не ИТ",
  "Главный менеджер по продажам среднему и крупному бизнесу": "не ИТ",
  "Финансовый менеджер (FP&A)": "не ИТ",
  "Старший менеджер по привлечению и сопровождению клиентов МСБ": "не ИТ",
  "Ведущий аналитик": "Аналитика данных/ Data Science",
  "Оператор по упаковке и отправке карт": "не ИТ",
  "Ведущий бизнес тренер": "не ИТ",
  "Младший бизнес аналитик (технолог)": "не ИТ",
  "Ведущий менеджер прямых продаж малому бизнесу": "не ИТ",
  "Ведущий Специалист отдела сопровождения судебного производства": "не ИТ",
  "Руководитель группы по обслуживанию состоятельных клиентов": "не ИТ",
  "Менеджер по привлечению корпоративных клиентов среднего и крупного бизнеса": "не ИТ",
  "Ведущий кредитный специалист": "не ИТ",
  "Специалист по поддержке бизнеса": "не ИТ",
  "Оператор Тинькофф Банка": "не ИТ",
  "Региональный менеджер по работе с партнёрами": "не ИТ",
  "Заведующий кассой": "не ИТ",
  "Менеджер по работе с претензиями": "не ИТ",
  "Руководитель группы отдела технического обеспечения пользователей": "Администрирование",
  "Руководитель операционного сектора отдела развития качества": "не ИТ",
  "Старший специалист отдела развития качества": "не ИТ",
  "Руководитель группы операторов управления поддержки клиентов": "не ИТ",
  "Оператор раннего взыскания задолженности": "не ИТ",
  "Персональный менеджер Инвестиции": "не ИТ",
  "Ведущий специалист по документообороту": "не ИТ",
  "Главный менеджер развития по работе с ключевыми клиентами.": "не ИТ",
  "Специалист отдела поддержки инвестиционных продуктов": "не ИТ",
  "Старший менеджер по кредитованию корпоративных клиентов": "не ИТ",
  "Специалист по работе с клиентами бизнес отдела": "не ИТ",
  "Главный эксперт": "не ИТ",
  "Представитель Тинькофф Банк": "не ИТ",
  "Руководитель группы управления": "не ИТ",
  "Консультант Тинькофф": "не ИТ",
  "Эксперт технической поддержки торговых решений Бэк офис": "Администрирование",
  "Комплаенс менеджер": "не ИТ",
  "Старший риск-менеджер": "не ИТ",
  "Старший специалист поддержки бизнеса": "не ИТ",
  "Координатор | Специалист по поддержке бизнес-процесса комплаенс-проверок": "не ИТ",
  "Специалист по взысканию кредиторской задолженности": "не ИТ",
  "Ведущий специалист кредитного отдела": "не ИТ",
  "Ведущий специалист отдела расчёта заработной платы": "не ИТ",
  "Обработка данных удалённо": "Аналитика данных/ Data Science",
  "Заместитель руководителя группы отдела продаж": "не ИТ",
  "Специалист дистанционной обработки данных": "не ИТ",
  "Старший специалист по работе с клиентами бизнес": "не ИТ",
  "Оператор поддержки в чате": "не ИТ",
  "Ведущий аналитик бизнес-процессов": "не ИТ",
  "Менеджер по работе со средним и крупным бизнесом": "не ИТ",
  "Менеджер по работе с бизнес клиентами": "не ИТ",
  "Консультант Тинькофф Путешествия": "не ИТ",
  "Главный клиентский менеджер": "не ИТ",
  "Специалист по работе с физическими лицами": "не ИТ",
  "Специалист по проверке кредитных заявок": "не ИТ",
  "Оператор по обработке обращений от физических лиц": "не ИТ",
  "Руководитель группы адаптации новичков": "не ИТ",
  "оператор по продаже сотовой связи": "не ИТ",
  "Специалист по работе с поздней задолженностью": "не ИТ",
  "Главный специалист отдела взыскания": "не ИТ",
  "Ассистент отдела продаж": "не ИТ",
  "Персональный менеджер по инвистициям, Премиум": "не ИТ",
  "Оператор Soft": "не ИТ",
  "Сотрудник отдела технической поддержки": "Администрирование",
  "Курьер с автомобилем": "не ИТ",
  "Руководитель сектора распределения и координации маршрутов": "не ИТ",
  "Специалист отдела взыскания задолженности": "не ИТ",
  "Специалисты сектора обслуживания карт": "не ИТ",
  "Менеджер по работе с Vip  клиентами": "не ИТ",
  "Ведущий менеджер по развитию бизнес": "не ИТ",
  "Руководитель проектного офиса": "не ИТ",
  "Оператор обработки заявок": "не ИТ",
  "Оператор Тинькофф Бизнес": "не ИТ",
  "Оператор по упаковке банковских карт": "не ИТ",
  "Специалист бухгалтерских услуг": "не ИТ",
  "Главный специалист Отдела электронных обращений направления Инвестиций": "не ИТ",
  "Рыбообработчик": "не ИТ",
  "Менеджер по работе с клиентами в чатах": "не ИТ",
  "Специалист по работе с документами": "не ИТ",
  "Эксперт по урегулированию убытков КАСКО": "не ИТ",
  "Ведущий специалист, управление развития качества": "не ИТ",
  "Специалист мониторинга бизнеса": "не ИТ",
  "Специалист по обслуживанию юридических лиц": "не ИТ",
  "Копирайтер": "не ИТ",
  "Менежжер по привлечению корпоративных клиентов": "не ИТ",
  "Специалист документарной поддержки": "не ИТ",
  "VIP представитель": "не ИТ",
  "Ведущий клиентский менеджер по работе с крупным бизнесом": "не ИТ",
  "Ведущий менеджер продаж среднему бизнесу": "не ИТ",
  "Старший  аналитик": "Аналитика данных/ Data Science",
  "Кредитный брокер": "не ИТ",
  "Руководитель группы (Бэк-офис)": "не ИТ",
  "Старший менеджер направления прямых продаж по кредитованию покупателей": "не ИТ",
  "Ведущий специалист сектора электронных коммуникаций": "не ИТ",
  "Старший инженер по обеспечению качества": "Тестирование",
  "Проектный менеджер, оператор, тренер, наставник.": "не ИТ",
  "Ведущий специалист отдела обслуживания депозитных продуктов, а также наставник в отделе обучения компании.": "не ИТ",
  "ведущий менеджер по привлечению среднего и крупного бизнеса": "не ИТ",
  "Младший специалист в направлении бэк-офиса": "не ИТ",
  "Менеджер по развитию корпоративных клиентов": "не ИТ",
  "Менеджер продаж B2B (удалённо)": "не ИТ",
  "Разработчик DWH": "Аналитика данных/ Data Science",
  "Старший контент-менеджер": "не ИТ",
  "Менеджер по продажам и работе с клиентами": "не ИТ",
  "Оператор call-центра по страховым продуктам": "не ИТ",
  "Оператор кол центра, специалист по продажам кредитных карт": "не ИТ",
  "Специалист отдела по обслуживанию физических лиц": "не ИТ",
  "Менеджер по работе с клиентами (удаленно)": "не ИТ",
  "Младший менеджер по работе с клиентами": "не ИТ",
  "Главный эксперт по работе с клиентами": "не ИТ",
  "Телемаркетолог": "не ИТ",
  "Менеджер образовательных продуктов": "не ИТ",
  "Консультант (бизнес)": "не ИТ",
  "Product design lead": "Дизайн",
  "Специалист отдела обслуживания бизнеса": "не ИТ",
  "Руководитель группы отдела обслуживания банковских продуктов": "не ИТ",
  "Представитель Тинькофф Банка": "не ИТ",
  "Водитель-инкассатор": "не ИТ",
  "Специалист пропускного режима": "не ИТ",
  "Data Analyst": "Аналитика данных/ Data Science",
  "Оператор клиентской поддержки ( оператор чата)": "не ИТ",
  "Менеджер по спецпроектам": "не ИТ",
  "Младший программист": "Разработка",
  "Бухгалтер по сопровождению ИП": "не ИТ",
  "Кредитный инспектор отдела взыскания": "не ИТ",
  "Специалист по продажам банковских продуктов": "не ИТ",
  "Оператор ООД": "не ИТ",
  "Специалист поддержки премиальных клиентов": "не ИТ",
  "Менеджер по поддержки премиальных клиентов": "не ИТ",
  "Руководитель группы продаж b2b": "не ИТ",
  "Ведущий менеджер среднего и крупного бизнеса": "не ИТ",
  "Специалист Сектора беззалоговых кредитов": "не ИТ",
  "Оператор входящих обращений (звонки и чаты).": "не ИТ",
  "HR-специалист": "не ИТ",
  "Начальник операционного отдела": "не ИТ",
  "Специалист по обслуживанию бизнеса": "не ИТ",
  "Менеджер по работе с корпоративными клиентами средний бизнес": "не ИТ",
  "Аналитик систем мотивации и ее расчетов": "не ИТ",
  "Главный эксперт противодействия мошенничеству": "не ИТ",
  "Менеджер по продажам финансовых услуг": "не ИТ",
  "Персональный менеджер по бизнесу": "не ИТ",
  "Специалист поддержки на чатах": "не ИТ",
  "Старший специалист отдела продаж": "не ИТ",
  "Персональный менеджер в Тинькофф Бизнес": "не ИТ",
  "Младший сотрудник бэк-офиса (2-я линия техподдержки)": "Администрирование",
  "Специалист по расчету заработной платы": "не ИТ",
  "Оператор на входящих сообщениях": "не ИТ",
  "Старший специалист группы сопровождения обслуживания состоятельных клиентов": "не ИТ",
  "Оператор дистанционной обработки (подработка, удаленно)": "не ИТ",
  "HRBP процессов В2С": "не ИТ",
  "Старший специалист отдела взыскания": "не ИТ",
  "Ведущий специалист отдела входящих обращений": "не ИТ",
  "Специалист по рекламе и связям с общественностью": "не ИТ",
  "Специалист кредитного непрерывного развития": "не ИТ",
  "Руководитель команды привлечения и развития продуктов малого и среднего бизнеса": "не ИТ",
  "Специалист поддержки сектора кредитных продуктов": "не ИТ",
  "Специалист Back Office": "не ИТ",
  "Менеджер образовательных программ": "не ИТ",
  "руководитель направления \"Корпоративная социальная ответственность\"": "не ИТ",
  "Специалист по складскому учету": "не ИТ",
  "Ведущий эксперт отдела инвестиционных продуктов": "не ИТ",
  "Торговый представитель  банка": "не ИТ",
  "Тренер по продажам": "не ИТ",
  "Оператор чата поддержки": "не ИТ",
  "Региональный торговый представитель": "не ИТ",
  "Старший аналитик данных": "Аналитика данных/ Data Science",
  "Telemarketing department specialist": "не ИТ",
  "Менеджер по продаже бизнесу": "не ИТ",
  "Специалист клиентской поддержки Тинькофф банк в регионах России": "не ИТ",
  "Оператор банковских операций": "не ИТ",
  "Ведущий менеджер регистрации бизнеса": "не ИТ",
  "Специалист службы поддержки пользователей (Бизнес)": "Администрирование",
  "Младший специалист отдела сопровождения судебного производства": "не ИТ",
  "Старший специалист службы финансового мониторинга": "не ИТ",
  "Персональный менеджер (бизнес)": "не ИТ",
  "Младший юрист": "не ИТ",
  "Оператор обработки сообщений по операциям": "не ИТ",
  "Оператор поддержки входящей линии": "не ИТ",
  "Бизнес-тренер и ведущий эксперт": "не ИТ",
  "Главный менеджер по работе со среднем и крупным бизнесом": "не ИТ",
  "Middle Data Steward": "Аналитика данных/ Data Science",
  "Специалист по работе с клиентами (дебетовые карты)": "не ИТ",
  "Менеджер по работе с VIP-клиентами": "не ИТ",
  "Эксперт отдела сопровождения комплаенс проверок": "не ИТ",
  "Менеджер В2В": "не ИТ",
  "Специалист по кадрам": "не ИТ",
  "Редактор-корректор": "не ИТ",
  "Юрист начального уровня": "не ИТ",
  "Специалист по работе с клиентами, группа непрерывного обслуживания, отдел по обслуживанию банковских продуктов": "не ИТ",
  "Специалист развития качества в сфере страхования": "не ИТ",
  "Руководитель группы отдела взыскания": "не ИТ",
  "Оператор чата, администратор внутренней поддержки сотрудников": "не ИТ",
  "Преподаватель банка": "не ИТ",
  "Клинер банкоматов": "не ИТ",
  "Motion designer": "Дизайн",
  "Специалист обработки клиентских данных": "Аналитика данных/ Data Science",
  "Менеджер активных продаж": "не ИТ",
  "Специалист сектора по развитию регионов отдела исходящих обращений управления поддержки клиентов департамента клиентского обслуживания": "не ИТ",
  "Project Manager": "не ИТ",
  "Специалист отдела дебетовых продуктов": "не ИТ",
  "Оператор call-центра отдел взыскания": "не ИТ",
  "Наставник отдела инвестиций": "не ИТ",
  "Персональный менеджер по среднему и малому бизнесу": "не ИТ",
  "Менеджер проектов развития": "не ИТ",
  "Эксперт сектора обработки  входящих чатов в регионах": "не ИТ",
  "Risk Analyst": "Аналитика данных/ Data Science",
  "Оператор инвестиции": "не ИТ",
  "Директор по продажам": "не ИТ",
  "Главный клиентский менеджер по работе с клиентами среднего и крупного бизнеса": "не ИТ",
  "Руководитель регионального подразделения": "не ИТ",
  "Главный менеджер по продажам среднему крупному бизнесу": "не ИТ",
  "Старший специалист премиального сегмента": "не ИТ",
  "Стажёр": "не ИТ",
  "Главный специалист отдела валютного контроля": "не ИТ",
  "Senior UX/UI Designer": "Дизайн",
  "mobile QA engineer": "Тестирование",
  "Агент прямых продаж": "не ИТ",
  "Младший специалист управления бэк-офиса": "не ИТ",
  "Оператор на линии помощи": "не ИТ",
  "Старший персональный менеджер по работе со средним и крупным бизнесом": "не ИТ",
  "Специалист премиальной поддержки": "не ИТ",
  "Ведущий специалист по работе с ключевыми клиентами": "не ИТ",
  "Ведущий менеджер по подбору и адаптации персонала": "не ИТ",
  "Оператор тех поддержки": "Администрирование",
  "Оператор по продажам инвестиций": "не ИТ",
  "Специалист по сопровождению продаж": "не ИТ",
  "Специалист отдела электронной корреспонденции": "не ИТ",
  "Специалист по обработке сообщений": "не ИТ",
  "Меж региональный представитель": "не ИТ",
  "Специалист по обслуживанию премиальных клиентов": "не ИТ",
  "Кредитный инспектор отдела верификации Департамента рисков.": "не ИТ",
  "Младший специалист сопровождения бизнеса": "не ИТ",
  "Территориальный руководитель отдела привлечения юридических лиц": "не ИТ",
  "Оператор по продаже банковских продуктов сегмент Бизнес": "не ИТ",
  "Старший программист": "Разработка",
  "Младший специалист сектора непрерывного обслуживания бизнеса": "не ИТ",
  "Заместитель руководителя группы продаж": "не ИТ",
  "Директор по развитию бизнеса": "не ИТ",
  "Главный менеджер среднего крупного бизнеса": "не ИТ",
  "Менедже по продажам бизнесу": "не ИТ",
  "Администратор портала": "Администрирование",
  "Главный менеджер по работе с клиентами": "не ИТ",
  "Графический дизайнер": "Дизайн",
  "Специалист сектора обработки входящих звонков": "не ИТ",
  "Специалист сервисного центра": "не ИТ",
  "Налоговый консультант": "не ИТ",
  "Специалист по регистрации юридических лиц": "не ИТ",
  "Специалист по продажам пластиковых карт в ТРЦ": "не ИТ",
  "Ведущий эксперт отдела обработки входящих звонков": "не ИТ",
  "Оператор-консультант": "не ИТ",
  "QA Lead": "Тестирование",
  "Менеджер по обслуживанию премиальных клиентов": "не ИТ",
  "Специалист по закупкам": "не ИТ",
  "Опереатор": "не ИТ",
  "Специалист по работе с b2b клиентами": "не ИТ",
  "Менеджер отдела розничных продаж": "не ИТ",
  "Оператор входящих обращений (чаты)": "не ИТ",
  "Ответственный за представителей; представитель": "не ИТ",
  "Старший Персональный менеджер по инвестициям 2-го уровня": "не ИТ",
  "Финансовый специалист": "не ИТ",
  "Ведущий специалист по телемаркетингу": "не ИТ",
  "Специалист поддержки корпоративных клиентов": "не ИТ",
  "Специалист отдела исходящих обращений": "не ИТ",
  "Специалист сектора поддержки заказчиков": "не ИТ",
  "Оператор отдела ТМЦ": "не ИТ",
  "Главный клиентский менеджер по продажам среднему и крупному бизнесу": "не ИТ",
  "Специалист сохранения клиентов": "не ИТ",
  "Специалист сектора сохранения клиентов": "не ИТ",
  "Специалист сектора непрерывного обслуживания": "не ИТ",
  "Торговый представитель отдела прямых продаж": "не ИТ",
  "Региональный менеджер": "не ИТ",
  "Lead Senior Product Designer": "Дизайн",
  "Оператор по продаже страховых продуктов": "не ИТ",
  "Technical Support": "Администрирование",
  "Старший специалист отдел сопровождения и открытия счетов юридических лиц (Compliance)": "не ИТ",
  "Менеджер отдела развития сети самообслуживания": "не ИТ",
  "Руководитель группы отдела обслуживания бизнес-клиентов": "не ИТ",
  "Оператор отдела эмиссии": "не ИТ",
  "Консультант отдела сопровождения продаж": "не ИТ",
  "Специалист отдела помощи": "не ИТ",
  "Оператор сопровождения судебного и исполнительного производства": "не ИТ",
  "Оператор дистанционного call- центра": "не ИТ",
  "Руководитель группы управления сопровождения продаж": "не ИТ",
  "Технический консультант": "Администрирование",
  "Старший менеджер по подбору IT персонала": "не ИТ",
  "Бармен": "не ИТ",
  "Аналитик актуарного управления": "Аналитика данных/ Data Science",
  "Руководитель направления по работе с ключевыми партнерами": "не ИТ",
  "оператор  отдела взыскания ,отдел продаж": "не ИТ",
  "QA инженер": "Тестирование",
  "Оператор по обработке входящих обращений от физических лиц": "не ИТ",
  "Специалист направления поддержки пользователей инвестиционных продуктов": "не ИТ",
  "Менеджер по работе с ключевыми партнерами": "не ИТ",
  "Инженер по тестированию": "Тестирование",
  "Специалист сектора персонального обслуживания VIP-клиентов.": "не ИТ",
  "Специалист по доставке банковских карт": "не ИТ",
  "Ведущий бухгалтер": "не ИТ",
  "Главный специалиста отдела электронных обращений": "не ИТ",
  "Специалист по продажам B2B": "не ИТ",
  "Специалист по управлению качеством": "не ИТ",
  "Специалист по онлайн урегулированию убытков": "не ИТ",
  "Специалист оспаривания мошеннических операций": "не ИТ",
  "Главный эксперт техподдержки": "Администрирование",
  "Marketing Team Lead": "не ИТ",
  "Ведущий менеджер по привлечению крупного и среднего бизнеса": "не ИТ",
  "Секретарь": "не ИТ",
  "Стажер в отеле кадров": "не ИТ",
  "Старший менеджер продукта (Бэк-офис ФЛ)": "не ИТ",
  "Специалист по связям с общественностью": "не ИТ",
  "Automation QA": "Тестирование",
  "Специалист по сопровождению электронных систем": "Администрирование",
  "Инженер-аналитик DWH в отделе переводов и трансграничных переводов": "Аналитика данных/ Data Science",
  "Менеджер по работе с клиентам среднего и крупного бизнеса": "не ИТ",
  "Senior Software Developer": "Разработка",
  "тренер по продажам и РСВ": "не ИТ",
  "Специалист по поддержке премиальных клиентов": "не ИТ",
  "Специалист бэк офиса СПКП": "не ИТ",
  "Консультант физических лиц": "не ИТ",
  "Android разработчик": "Разработка",
  "Специалист по работе со входящими обращениями": "не ИТ",
  "Старший специалист противодействия мошенничеству": "не ИТ",
  "Специалист сектора обслуживания беззалоговых кредитов": "не ИТ",
  "Специалист по сопровождению кандидатов": "не ИТ",
  "Ведущий специалист отдела кадров": "не ИТ",
  "Специалист отдела обслуживания банковских продуктов": "не ИТ",
  "Старший менеджер проектов": "не ИТ",
  "Руководитель группы отдела инвестиций": "не ИТ",
  "Менеджер по работе с партнёрами": "не ИТ",
  "Менеджер по работе с клиентами (удалённо)": "не ИТ",
  "Оператор в клиентских продажах": "не ИТ",
  "Менеджер по эквайрингу": "не ИТ",
  "Специалист отдела подбора персонала": "не ИТ",
  "Младший специалист непрерывного обслуживания": "не ИТ",
  "Эксперт отдела обслуживания банковских продуктов": "не ИТ",
  "Lead Software Engineer": "Разработка",
  "Специалист по обучению и контролю качества работы": "не ИТ",
  "Инженер ОУУ": "не ИТ",
  "Специалист отдела удержания клиентов": "не ИТ",
  "Оператор исходящих холодных звонков": "не ИТ",
  "Менеджер по привлечению корпоративных клиентов среднего бизнеса": "не ИТ",
  "Специалист сектора обработки внешних запросов": "не ИТ",
  "Специалист сектора по обработке входящих звонков": "не ИТ",
  "Специалист отдела усиленной идентификации": "не ИТ",
  "Эксперт отдела сохранения клиентов": "не ИТ",
  "Специалист по обработке входящих чатов": "не ИТ",
  "Эксперт по развитию регионов": "не ИТ",
  "Промоутер": "не ИТ",
  "Специалист сектора обслуживания состоятельных клиентов": "не ИТ",
  "Оператор по валидации и верификации данных": "Аналитика данных/ Data Science",
  "Оператор по обслуживанию клиентов": "не ИТ",
  "Специалист по работе с государственными органами": "не ИТ",
  "Оператор call центра": "не ИТ",
  "Специалист налогового учета": "не ИТ",
  "Консультант технической поддержки": "Администрирование",
  "Ведущий специалист отдела продаж продуктов некэптивного страхования": "не ИТ",
  "Tech Lead отдела персонализации": "Разработка",
  "Ведущий специалист по инвестициям": "не ИТ",
  "Стажер DevOps": "Администрирование",
  "Специалист поддержки бизнеса": "не ИТ",
  "Инвестиционный аналитик": "не ИТ",
  "Эксперт по взысканию задолженности": "не ИТ",
  "Оператор call-center": "не ИТ",
  "Специалист сектора обработки входящих чатов": "не ИТ",
  "Главный менеджер отдела обслуживания и развития среднего бизнеса": "не ИТ",
  "Эксперт по обслуживанию банковских продуктов": "не ИТ",
  "Менеджер отдела инвестиций": "не ИТ",
  "Менеджер по работе с крупным бизнесом": "не ИТ",
  "Младший специалист по непрерывному обслуживанию": "не ИТ",
  "Специалист выделенной группы непрерывного обслуживания юридических лиц": "не ИТ",
  "Ведущий менеджер по привлечению клиентов крупного бизнеса": "не ИТ",
  "Middle Android Developer": "Разработка",
  "Product-analyst": "Аналитика данных/ Data Science",
  "Руководитель группы продаж BNPL": "не ИТ",
  "Руководитель группы прямых продаж": "не ИТ",
  "Коллектор": "не ИТ",
  "Старший кредитный эксперт": "не ИТ",
  "Региональный менеджер по продажам b2b": "не ИТ",
  "Специалист развития регионов": "не ИТ",
  "Специалист сектора поддержки самообслуживания и партнеров": "не ИТ",
  "Младший специалист отдела обслуживания бизнеса": "не ИТ",
  "Специалист отдела обслуживания состоятельных клиентов": "не ИТ",
  "Заместитель руководителя сектора": "не ИТ",
  "Ведущий специалист сектора экспертного обслуживания": "не ИТ",
  "Эксперт сектора обработки входящих звонков": "не ИТ",
  "Эксперт входящих обращений": "не ИТ",
  "Менеджер продукта Ипотека": "не ИТ",
  "Специалист отдела финансовых потерь": "не ИТ",
  "Специалист сектора экспертного обслуживания премиум": "не ИТ",
  "Главный методолог": "не ИТ",
  "Территориальный менеджер по развитию бизнеса": "не ИТ",
  "Персональный менеджер по поддержке бизнес-клиентов": "не ИТ",
  "Специалист сектора оптимизации финансовых потерь": "не ИТ",
  "Эксперт сектора развития регионов": "не ИТ",
  "Главный эксперт поддержки чатов физических лиц": "не ИТ",
  "Специалист центра обслуживания клиентов": "не ИТ",
  "Ведущий оператор клиентского сервиса": "не ИТ",
  "Оператор колл-центра удаленно": "не ИТ",
  "Специалист по работе с электронными заявками": "не ИТ",
  "Эксперт комплаенс-сопровождения": "не ИТ",
  "Менеджер МСБ": "не ИТ",
  "Младший специалист обслуживания ВИП-клиентов": "не ИТ",
  "Аналитик DWH": "Аналитика данных/ Data Science",
  "Старший персональный менеджер по работе с премиум клиентами": "не ИТ",
  "Оператор обработки текста и ввода данных": "Аналитика данных/ Data Science",
  "Менеджер по привлечению клиентов среднего бизнеса": "не ИТ",
  "Эксперт сектора омниканальных обращений": "не ИТ",
  "Аналитик по оптимизации бизнес процессов": "не ИТ",
  "Старший персональный менеджер по инфестициям": "не ИТ",
  "Старший инженер по технической поддержке": "Администрирование",
  "Специалист сектора исходящих коммуникаций": "не ИТ",
  "Аналитик-стажер": "Аналитика данных/ Data Science",
  "Специалист отдела обработки клиентских данных": "Аналитика данных/ Data Science",
  "Стажёр-разработчик": "Разработка",
  "Координатор проекта": "не ИТ",
  "Mobile-Developer": "Разработка",
  "Эксперт обработки чатов": "не ИТ",
  "Специалист отдела обработки персональных данных": "не ИТ",
  "Специалист по работе с крупным бизнесом": "не ИТ",
  "Оператор обработки входящих обращений": "не ИТ",
  "Оператор склада": "не ИТ",
  "Руководитель команды по массовому подбору персонала": "не ИТ",
  "Младший специалист сектора поддержки инвестиционных и брокерских сервисов": "не ИТ",
  "Менеджер проектов отдела обучения": "не ИТ",
  "Ведущий специалист отдела сопровождения платежей валютного контроля": "не ИТ",
  "Главный специалист отдела исходящих обращений": "не ИТ",
  "Торговый представитель прямой доставки": "не ИТ",
  "Специалист отдела автокредитования": "не ИТ",
  "Менеджер прямых продаж": "не ИТ",
  "Офис-менеджер": "не ИТ",
  "Персональный менеджер по работе с бизнес-клиентами": "не ИТ",
  "Оператор по работе с клиентами": "не ИТ",
  "Руководитель направления развития продаж": "не ИТ",
  "Эксперт сектора обработки входящих чатов": "не ИТ",
  "Ведущий специалист службы поддержки": "не ИТ",
  "Специалист обработки входящих чатов": "не ИТ",
  "Специалист отдела обработки электронных обращений": "не ИТ",
  "Специалист обработки информации": "не ИТ",
  "Специалист отдела инвестиций": "не ИТ",
  "Главный персональный менеджер": "не ИТ",
  "Руководитель группы подбора персонала": "не ИТ",
  "Специалист сектора обработки входящих чатов в регионе": "не ИТ",
  "Analyst": "Аналитика данных/ Data Science",
  "Оператор поздних взысканий": "не ИТ",
  "Ведущий специалист по развитию качества": "не ИТ",
  "Ведущий специалист отдела продаж": "не ИТ",
  "Тренер-преподаватель": "не ИТ",
  "Ведущий специалист отдела качества": "не ИТ",
  "Ведущий менеджер по привлечению клиентов среднего и крупного бизнеса": "не ИТ",
  "Специалист по продаже кредитных карт": "не ИТ",
  "Специалист обработки государственных обращений": "не ИТ",
  "Специалист сектора поддержки клиентских операций": "не ИТ",
  "Менеджер по привлечению среднего бизнеса": "не ИТ",
  "Ведущий инженер-программист": "Разработка",
  "Руководитель группы аналитики сквозных процессоы": "Аналитика данных/ Data Science",
  "Специалист бухгалтерского обслуживания юридических лиц": "не ИТ",
  "Менеджер по продажам бизнес направление": "не ИТ",
  "Руководитель региональных продаж": "не ИТ",
  "Специалист по работе с клиентами в TCRM системе": "не ИТ",
  "Главный специалист отдела планирования": "не ИТ",
  "Представитель по банковским продуктам": "не ИТ",
  "Главный специалист группы поддержки иностранных клиентов": "не ИТ",
  "Оператор по сопровождению клиентов": "не ИТ",
  "Оператор исходящей линии": "не ИТ",
  "Менеджер отдела взыскания": "не ИТ",
  "Продуктовый редактор": "не ИТ",
  "Эксперт по инвестиционным продуктам банка": "не ИТ",
  "Специалист по сохранению клиентов по кредитным картам": "не ИТ",
  "Менеджер банка": "не ИТ",
  "Старший эксперт": "не ИТ",
  "Представитель сектора сопровождения продаж": "не ИТ",
  "Старший специалист направления поддержки бизнес-процессов": "не ИТ",
  "Менеджер по сохранению кредитных договоров": "не ИТ",
  "Главный специалист обработки входящих обращений": "не ИТ",
  "Специалист по клиентскому сервису": "не ИТ",
  "Старший специалист по сохранению клиентов Тинькофф Бизнес": "не ИТ",
  "Product Owner": "не ИТ",
  "Руководитель отдела разработки": "Разработка",
  "Оператор входящих чатов": "не ИТ",
  "Инспектор по взысканию просроченной задолженности": "не ИТ",
  "Эксперт по обработке входящих обращений": "не ИТ",
  "Эксперт отдела комплаенс": "не ИТ",
  "Эксперт отдела входящего обращения": "не ИТ",
  "Страховой представитель": "не ИТ",
  "Секретарь на ресепшн": "не ИТ",
  "Специалист по привлечению клиентов": "не ИТ",
  "Ведущий специалист по мониторингу операций AML": "не ИТ",
  "Эксперт отдела проверки надежности клиентов": "не ИТ",
  "Специалист отдела валютного контроля": "не ИТ",
  "Главный менеджер по привлечению клиентов среднего бизнеса": "не ИТ",
  "Старший менеджер по работе с ключевыми клиентами": "не ИТ",
  "Эксперт отдела входящих обращений на английском языке": "не ИТ",
  "Специалист отдела оптимизации ресурсов": "не ИТ",
  "Специалист сектора обработки и обслуживания входящих звонков": "не ИТ",
  "Специалист непрерывного обслуживания": "не ИТ",
  "Специалист клиентского обслуживания": "не ИТ",
  "Руководитель отдела": "не ИТ",
  "Специалист по обработке обращений": "не ИТ",
  "Руководитель службы внутреннего контроля": "не ИТ",
  "Региональный управляющий": "не ИТ",
  "Эксперт бэк-офиса": "не ИТ",
  "Оператор на дому": "не ИТ",
  "Аналитик SQL": "Аналитика данных/ Data Science",
  "Эксперт по работе с входящими обращениями": "не ИТ",
  "Менеджер по обслуживанию ВИП-клиентов": "не ИТ",
  "Менеджер дистанционного подбора персонала": "не ИТ",
  "Региональный представитель по работе с партнерами": "не ИТ",
  "Младший специалист по сопровождению проектов": "не ИТ",
  "Engineer": "Разработка",
  "Ведущий менеджер по привлечению клиентов среднего бизнеса": "не ИТ",
  "Программист-аналитик": "Разработка",
  "Эксперт по входящим обращениям": "не ИТ",
  "Специалист по работе с клиентами банка": "не ИТ",
  "Эксперт департамента клиентского обслуживания": "не ИТ",
  "Специалист обслуживания бизнеса": "не ИТ",
  "Руководитель направления по работе с партнерами": "не ИТ",
  "Руководитель группы по работе  с клиентами": "не ИТ",
  "Менеджер по кредитным продуктам": "не ИТ",
  "Специалист раннего взыскания просроченной задолженности": "не ИТ",
  "Территориальный менеджер": "не ИТ",
  "Эксперт по маркетинговым коммуникациям": "не ИТ",
  "Выездной клиентский менеджер": "не ИТ",
  "Младший специалист сектора мониторинга": "не ИТ",
  "Оператор сопровождения клиентов": "не ИТ",
  "Главный специалист отдела электронных обращений": "не ИТ",
  "UI/UX дизайнер": "Дизайн",
  "Менеджер по работе с документами": "не ИТ",
  "Designer": "Дизайн",
  "Оператор отдела исходящих обращений": "не ИТ",
  "Эксперт сектора сопровождения": "не ИТ",
  "Системный аналитик DWH": "Аналитика данных/ Data Science",
  "Главный специалист отдела продаж": "не ИТ",
  "Менеджер по продажам ипотечных программ": "не ИТ",
  "Консультант по работе с клиентами": "не ИТ",
  "Главный специалист отдела развития продуктовых направлений": "не ИТ",
  "Специалист по кредитованию физических лиц": "не ИТ",
  "Менеджер регионального отдела": "не ИТ",
  "Главный кредитный эксперт": "не ИТ",
  "Менеджер по развитию партнерской сети В2В": "не ИТ",
  "Менеджер премиальной поддержки": "не ИТ",
  "Специалист отдела входящих обращений департамента клиентского обслуживания": "не ИТ",
  "Руководитель группы отдела входящих обращений облачного сервиса": "не ИТ",
  "Compliance Analyst": "не ИТ",
  "Оператор по работе с задолженностью": "не ИТ",
  "Специалист сектора обработки омниканальных обращений": "не ИТ",
  "Оператор службы поддержки на чатах": "не ИТ",
  "Специалист по работе с документацией": "не ИТ",
  "Руководитель группы бэк-офис": "не ИТ",
  "Главный специалист отдела последующего контроля": "не ИТ",
  "Руководитель сектора оптимизации процессов": "не ИТ",
  "Специалист группы поддержки иностранных клиентов": "не ИТ",
  "Специалист обработки входящих сообщений": "не ИТ",
  "Главный специалист отдела обслуживания потребительских кредитов": "не ИТ",
  "Оператор группы поддержки иностранных клиентов": "не ИТ",
  "Специалист группы обзвона": "не ИТ",
  "Старший менеджер развития среднего и крупного бизнеса": "не ИТ",
  "Ведущий менеджер корпоративного управления": "не ИТ",
  "Ведущий специалист сектора по развитию регионов": "не ИТ",
  "Специалист отдела финансового мониторинга": "не ИТ",
  "Специалист по взаимодействию со спецслужбами": "не ИТ",
  "Младший специалист отдела дистанционной обработки данных": "не ИТ",
  "Менеджер-администратор": "Администрирование",
  "Специалист сектора сопровождения исполнительного производства": "не ИТ",
  "Специалист сектора обслуживания клиентов": "не ИТ",
  "Оператор по обработке обращений физических лиц": "не ИТ",
  "Главный эксперт по обслуживанию банковских продуктов": "не ИТ",
  "Эксперт по обработке входящих звонков": "не ИТ",
  "Специалист входящего обслуживания": "не ИТ",
  "Эксперт обслуживания бизнес-клиентов": "не ИТ",
  "Эксперт сектора обработки входящих чатов в регионах": "не ИТ",
  "Специалист инвестиционных продуктов": "не ИТ",
  "Оператор целевых кредитов": "не ИТ",
  "Территориальный руководитель по привлечению партнеров": "не ИТ",
  "Специалист центра обработки вызовов": "не ИТ",
  "Специалист по финансовым услугам": "не ИТ",
  "Эксперт претензионного отдела": "не ИТ",
  "Специалист поддержки юридических лиц": "не ИТ",
  "Менеджер прямых продаж РКО": "не ИТ",
  "Оператор по работе с просроченой задолженностью": "не ИТ",
  "SMM-Менеджер": "не ИТ",
  "Специалист по тестированию ПО": "Тестирование",
  "Менеджер по закупкам": "не ИТ",
  "Менеджер по сопровождению агрегаторов": "не ИТ",
  "Ведущий специалист по взысканию просроченной задолженности": "не ИТ",
  "Промоутер-распространитель": "не ИТ",
  "Кредитный инспектор сектора залогового кредитования": "не ИТ",
  "Консультант call-центра": "не ИТ",
  "Менеджер по развитию клиентов": "не ИТ",
  "Оператор по залоговому кредитованию": "не ИТ",
  "Инвестиционный аналитик направления по работе с премиальными клиентами": "не ИТ",
  "Эксперт в отделе потребительских кредитов": "не ИТ",
  "Специалист по проверке заявок": "не ИТ",
  "Специалист отдела поддержки сотрудников": "не ИТ",
  "Стажер": "не ИТ",
  "Middle Kotlin Developer": "Разработка",
  "Специалист отдела обращений": "не ИТ",
  "Специалист отдела качества": "не ИТ",
  "Ведущий специалист отдела телемаркетинга": "не ИТ",
  "Специалист по претензионной работе": "не ИТ",
  "Оператор по продажам": "не ИТ",
  "Ведущий разработчик Angular": "Разработка",
  "Премиум-менеджер": "не ИТ",
  "Младший специалист управления по обслуживанию состоятельных клиентов": "не ИТ",
  "Старший менеджер по работе с клиентами": "не ИТ",
  "Ведущий специалист по подбору персонала": "не ИТ",
  "Специалист отдела криптографической защиты информации": "Администрирование",
  "Специалист отдела электронных коммуникаций": "не ИТ",
  "Оператор отдела взысканий": "не ИТ",
  "Специалист отдела по обслуживанию банковских продуктов": "не ИТ",
  "Старший менеджер развития клиентов среднего и крупного бизнеса": "не ИТ",
  "Ведущий специалист по работе с корпоративными клиентами": "не ИТ",
  "Программист-стажер": "Разработка",
  "Специалист по работе с инвестиционными продуктами": "не ИТ",
  "Оператор по работе с юридическими лицами": "не ИТ",
  "Эксперт по работе с премиальными клиентами": "не ИТ",
  "Инженер по обслуживанию POS-терминалов": "Администрирование",
  "Специалист управления контроля качества": "не ИТ",
  "Эксперт отдела электронных обращений": "не ИТ",
  "Ведущий специалист входящих обращений": "не ИТ",
  "Оператор регистрации входящей корреспонденции": "не ИТ",
  "Старший специалист службы поддержки": "не ИТ",
  "Специалист отдела обработки данных": "Аналитика данных/ Data Science",
  "Кредитный верификатор": "не ИТ",
  "Аналитик HR": "не ИТ",
  "Оператор по холодным продажам": "не ИТ",
  "Главный эксперт урегулирования претензий": "не ИТ",
  "Главный аналитик развития процессов": "не ИТ",
  "Специалист по обработке заявок на сайте": "не ИТ",
  "Оператор работы с просроченной задолженностью": "не ИТ",
  "Оператор по работе с заявками": "не ИТ",
  "Специалист отдела обучения": "не ИТ",
  "Менеджер по продажам услуг бизнесу": "не ИТ",
  "Ведущий менеджер по развитию качества обслуживания": "не ИТ",
  "Главный специалист отдела входящих обращений": "не ИТ",
  "Специалист сектора сопровождения и обслуживания премиальных клиентов": "не ИТ",
  "Специалист по доставке": "не ИТ",
  "Старший Android разработчик": "Разработка",
  "Младший консультант": "не ИТ",
  "Специалист по возврату просроченной задолженности": "не ИТ",
  "UX-редактор": "Дизайн",
  "Старший QA инженер": "Тестирование",
  "Банковский сотрудник входящих и исходящих звонков": "не ИТ",
  "Менеджер проектов образовательных продуктов": "не ИТ",
  "Специалист группы технической поддержки": "Администрирование",
  "Менеджер по продаже услуг": "не ИТ",
  "Выездной менеджер по доставке карт": "не ИТ",
  "Специалист сектора  обработки входящих чатов": "не ИТ",
  "Редактор direct-маркетинга": "не ИТ",
  "Специалист отдела по обслуживанию клиентов": "не ИТ",
  "Ведущий специалист по валютному контролю": "не ИТ",
  "Главный менеджер по привлечению корпоративных клиентов": "не ИТ",
  "Специалист отдела кредитования малого и среднего бизнеса": "не ИТ",
  "Специалист по работе с ключевыми клиентами": "не ИТ",
  "Эксперт отдела по обслуживанию": "не ИТ",
  "Специалист по сопровождению клиентов": "не ИТ",
  "Продавец": "не ИТ",
  "Оператор на линии": "не ИТ",
  "Эксперт по работе с малым и средним бизнесом": "не ИТ",
  "Эксперт по обслуживанию клиентов": "не ИТ",
  "Автор статей": "не ИТ",
  "Специалист по ипотечному кредитованию": "не ИТ",
  "Инженер второй линии технической поддержки": "Администрирование",
  "Ведущий специалист отдела исходящих обращений": "не ИТ",
  "Руководитель группы продаж B2G": "не ИТ",
  "Ведущий специалист отдела по обслуживанию юридических лиц": "не ИТ",
  "Специалист по сопровождению исполнительного производства": "не ИТ",
  "Старший менеджер по кредитованию": "не ИТ",
  "Главный специалист отдела расчетов": "не ИТ",
  "Специалист по бронированию авиабилетов": "не ИТ",
  "Специалист отдела реструктуризации": "не ИТ",
  "Эксперт технической поддержки клиентов": "Администрирование",
  "Менеджер продаж": "не ИТ",
  "Специалист отдела ввода данных": "Аналитика данных/ Data Science",
  "Эксперт по работе с бизнесом": "не ИТ",
  "Специалист по продажам кредитных продуктов": "не ИТ",
  "Специалист сопровождения обслуживания премиальных клиентов": "не ИТ",
  "Младший кредитный инспектор": "не ИТ",
  "Ведущий специалист по работе с иностранными партнерами": "не ИТ",
  "Ведущий специалист по контекстной рекламе": "не ИТ",
  "Эксперт сектора обработки входящих обращений": "не ИТ",
  "Специалист по открытию счетов": "не ИТ",
  "Менеджер по продаже услуг страхования": "не ИТ",
  "Территориальный руководитель по работе с юридическими лицами": "не ИТ",
  "Специалист-консультант": "не ИТ",
  "Customer Service Representative": "не ИТ",
  "SRE Engineer": "Администрирование",
  "Старший инвестиционный консультант": "не ИТ",
  "Ведущий специалист по оптимизации бизнес-процессов": "не ИТ",
  "Старший редактор": "не ИТ",
  "Специалист верификации данных": "не ИТ",
  "Менеджер по работе с регионами": "не ИТ",
  "Главный менеджер по развитию клиентов": "не ИТ",
  "Главный менеджер сектора продаж": "не ИТ",
  "Менеджер по поддержке бизнес-клиентов": "не ИТ",
  "Менеджер по госзакупкам": "не ИТ",
  "Менеджер в отделе сохранения клиентов": "не ИТ",
  "Ведущий инвестиционный советник": "не ИТ",
  "Scala Developer": "Разработка",
  "Кредитный инспектор отдела верификации": "не ИТ",
  "Эксперт отдела по обслуживанию банковских продуктов": "не ИТ",
  "Lead Delivery Manager": "Администрирование",
  "Специалист по взысканию залогового имущества": "не ИТ",
  "Руководитель обособленного подразделения": "не ИТ",
  "Специалист по продуктам": "не ИТ",
  "Специалист отдела управления маркетинга": "не ИТ",
  "Выездной представитель банка": "не ИТ",
  "3D-дизайнер": "Дизайн",
  "Аудитор по информационным технологиям": "Администрирование",
  "Заместитель руководителя региона": "не ИТ",
  "Эксперт по поддержке клиентов": "не ИТ",
  "Специалист отдела кредитования": "не ИТ",
  "Account Manager": "не ИТ",
  "Ведущий специалист по контекстной и медийной рекламе": "не ИТ",
  "Руководитель группы входящей корреспонденции": "не ИТ",
  "Менеджер по работе с клиентами и партнерами": "не ИТ",
  "Сканировщик документов": "не ИТ",
  "Performance Marketing Manager": "не ИТ",
  "Специалист по обработке входящих вызовов": "не ИТ",
  "Специалист поддержки чата": "не ИТ",
  "Специалист чата-поддержки": "не ИТ",
  "Старший инженер по тестированию ПО": "Тестирование",
  "Эксперт службы поддержки клиентов": "не ИТ",
  "Специалист группы сохранения клиентов": "не ИТ",
  "Менеджер E-commerce": "не ИТ",
  "Главный менеджер по работе с партнёрами": "не ИТ",
  "Менеджер по контролю качества": "не ИТ",
  "Специалист по обработке звонков": "не ИТ",
  "Эксперт по развитию": "не ИТ",
  "Эксперт непрерывного обслуживания": "не ИТ",
  "Редактор продукта": "не ИТ",
  "Специалист по финансовому мониторингу": "не ИТ",
  "Трейдер": "не ИТ",
  "Специалист обслуживания инвестиционных продуктов": "не ИТ",
  "Специалист по телемаркетингу": "не ИТ",
  "Ведущий специалист по качеству": "не ИТ",
  "Бизнес-аналитик 1С": "не ИТ",
  "Ведущий дизайнер": "Дизайн",
  "Специалист бухгалтерского обслуживания": "не ИТ",
  "Fronted разработчик": "Разработка",
  "Менеджер по продажам карт": "не ИТ",
  "Менеджер-консультант": "не ИТ",
  "Специалист технической поддержки клиентов": "Администрирование",
  "Главный специалист управления развития качества": "не ИТ",
  "Операционист банка": "не ИТ",
  "Менеджер по подбору и адаптации персонала": "не ИТ",
  "Программист": "Разработка",
  "Специалист непрерывного обслуживания бизнес клиентов": "не ИТ",
  "Главный технолог": "не ИТ",
  "Ведущий клиентский менеджер": "не ИТ",
  "Оператор службы взыскания": "не ИТ",
  "Ведущий специалист валютного контроля": "не ИТ",
  "Сотрудник склада": "не ИТ",
  "Специалист по обслуживанию банковских продуктов": "не ИТ",
  "Ведущий специалист управления развития качества": "не ИТ",
  "Эксперт сектора по развитию регионов": "не ИТ",
  "Старший кредитный инспектор отдела правового сопровождения": "не ИТ",
  "QA Head": "Тестирование",
  "Оператор производственной линии": "не ИТ",
  "Менеджер ипотечного кредитования": "не ИТ",
  "Руководитель группы по работе с государственными органами": "не ИТ",
  "Ведущий специалист тренинг центра": "не ИТ",
  "Младший специалист сектора обслуживания клиентов": "не ИТ",
  "Менеджер по региональному развитию": "не ИТ",
  "Персональный ассистент": "не ИТ",
  "Старший специалист сопровождения состоятельных клиентов": "не ИТ",
  "Специалист по автокредитованию": "не ИТ",
  "Product Designer": "Дизайн",
  "Старший специалист отдела систем управления бизнес-процессами": "не ИТ",
  "Кредитный аналитик": "не ИТ",
  "Страховой агент": "не ИТ",
  "Специалист отдела мониторинга и развития внутренних систем": "не ИТ",
  "Главный специалист отдела кадров": "не ИТ",
  "Специалист КЦ": "не ИТ",
  "Специалист по кадровому делопроизводству": "не ИТ",
  "Менеджер по обработке входящих обращений": "не ИТ",
  "Ведущий менеджер отдела ипотечного кредитования": "не ИТ",
  "Младший специалист информационной безопасности": "Администрирование",
  "Эксперт по сохранению кредитных карт": "не ИТ",
  "Специалист по электронной корреспонденции": "не ИТ",
  "Руководитель регионального отдела продаж": "не ИТ",
  "Специалист отдела обработки внутренних запросов": "не ИТ",
  "Специалист сектора дистанционной обработки данных": "не ИТ",
  "Специалист по работе с корпоративнми клиентами": "не ИТ",
  "Маркетолог-аналитик": "не ИТ",
  "Специалист по работе с задолженностью": "не ИТ",
  "Менеджер по работе с состоятельными клиентами": "не ИТ",
  "Специалист отдела операционного планирования и аналитики продаж": "не ИТ",
  "Специалист обработки электронных обращений": "не ИТ",
  "Ведущий специалист сектора оптимизации финансовых потерь": "не ИТ",
  "Менеджер по сохранению клиентов": "не ИТ",
  "Специалист обработки входящей документации": "не ИТ",
  "Тестировщик ПО": "Тестирование",
  "Специалист по обработке клиентских данных": "Аналитика данных/ Data Science",
  "Руководитель группы менеджеров по инвестициям": "не ИТ",
  "Специалист отдела налогообложения брокерского обслуживания": "не ИТ",
  "Менеджер B2B": "не ИТ",
  "Младший специалист отдела электронных обращений": "не ИТ",
  "Руководитель группы маркетинга": "не ИТ",
  "Эксперт по обслуживанию бизнеса": "не ИТ",
  "Старший специалист по клиентскому опыту": "не ИТ",
  "Эксперт непрерывного обслуживания бизнес-клиентов": "не ИТ",
  "Менеджер по развитию e-commerce": "не ИТ",
  "Ведущий кредитный инспектор сектора залогового кредитования": "не ИТ",
  "Программист-консультант": "Разработка",
  "Менеджер кредитного отдела": "не ИТ",
  "Специалист по дистанционному обслуживанию": "не ИТ",
  "Специалист по работе с юридическими компаниями": "не ИТ",
  "Руководитель отдела HR": "не ИТ",
  "Руководитель отдела по работе с персоналом": "не ИТ",
  "Специалист телефонных продаж": "не ИТ",
  "Специалист отдела сопровождения продаж": "не ИТ",
  "Младший специалист отдела развития и сопровождения инфраструктуры": "не ИТ",
  "Младший специалист по инвестициям": "не ИТ",
  "Операционист по работе с юридическими лицами": "не ИТ",
  "Главный аналитик": "Аналитика данных/ Data Science",
  "Специалист поддержки иностранных клиентов": "не ИТ",
  "Территориальный менеджер по продажам": "не ИТ",
  "Специалист отдела обработки обращений": "не ИТ",
  "Системный инженер": "Администрирование",
  "Ведущий специалист отдела досудебного взыскания задолженности": "не ИТ",
  "Главный специалист по информационной безопасности": "Администрирование",
  "Руководитель сектора сопровождения продаж": "не ИТ",
  "Персональный менеджер по работе с юридическими лицами": "не ИТ",
  "Менеджер сопровождения по инвестициям": "не ИТ",
  "Старший менеджер прямых продаж": "не ИТ",
  "Инспектор": "не ИТ",
  "Руководитель региона": "не ИТ",
  "Оператор по проверке кредитных заявок": "не ИТ",
  "Ведущий специалист сектора контроля отдела анализа и планирования": "не ИТ",
  "Специалист по контекстной рекламе": "не ИТ",
  "Специалист в секторе непрерывного обслуживания бизнеса": "не ИТ",
  "Руководитель колл-центра": "не ИТ",
  "Заместитель руководителя проекта": "не ИТ",
  "Специалист по работе с контрагентами": "не ИТ",
  "Руководитель отдела информационной поддержки и мониторинга": "не ИТ",
  "Руководитель регионального представительства": "не ИТ",
  "Специалист по продажам услуг бизнесу": "не ИТ",
  "Оператор входящих электронных обращений": "не ИТ",
  "Специалист отдела обработки электронных документов": "не ИТ",
  "Главный кредитный инспектор отдела взыскания": "не ИТ",
  "Специалист по проверке банковских операций": "не ИТ",
  "Младший специалист отдела кадров": "не ИТ",
  "Оператор по обработке заявок": "не ИТ",
  "Специалист по договорам": "не ИТ",
  "Руководитель по корпоративному бизнесу": "не ИТ",
  "Специалист клиентской поддержки банка и инвестиций": "не ИТ",
  "Младший специалист поддержки": "не ИТ",
  "Старший менеджер по подбору персонала": "не ИТ",
  "Специалист группы непрерывного обслуживания": "не ИТ",
  "Специалист по экономической безопасности": "не ИТ",
  "Security Expert": "Администрирование",
  "Главный специалист по мониторингу": "не ИТ",
  "Android Developer": "Разработка",
  "Оператор инвестиционных продуктов": "не ИТ",
  "Старший менеджер продаж": "не ИТ",
  "Ведущий специалист call-центра": "не ИТ",
  "Специалист сектора контроля качества и обучения": "не ИТ",
  "Эксперт сектора по развитию регионов управления поддержки клиентов": "не ИТ",
  "Руководитель отдела внутренних коммуникаций": "не ИТ",
  "Менеджер непрерывного обслуживания бизнеса": "не ИТ",
  "Аккаунт менеджер по работе с ключевыми партнерами": "не ИТ",
  "Кредитный инспектор отдела автокредитования": "не ИТ",
  "Специалист по обработке кредитов": "не ИТ",
  "Младший специалист непрерывного обслуживания клиентов": "не ИТ",
  "Специалист по работе с иностранными клиентами": "не ИТ",
  "Специалист отдела планирования": "не ИТ",
  "Ведущий специалист по технической поддержке": "Администрирование",
  "Младший специалист по разработке ПО": "Разработка",
  "Менеджер по прямым продажам": "не ИТ",
  "Главный специалист по подбору персонала": "не ИТ",
  "Старший разработчик Oracle": "Разработка",
  "Старший консультант": "не ИТ",
  "Эксперт отдела обслуживания премиальных клиентов": "не ИТ",
  "Эксперт отдела по развитию регионов": "не ИТ",
  "Оператор по упаковке пластиковых карт": "не ИТ",
  "Руководитель направления по работе с клиентами среднего и крупного бизнеса": "не ИТ",
  "Middle Backend System Analyst": "Разработка",
  "Инженер по контролю качества": "Тестирование",
  "Менеджер по инвестициям": "не ИТ",
  "Менеджер по привлечению клиентов малого и среднего бизнеса": "не ИТ",
  "Network Engineer": "Администрирование",
  "Руководитель учебного центра": "не ИТ",
  "Оператор верификации": "не ИТ",
  "Специалист по прямым продажам": "не ИТ",
  "Технический менеджер проектов": "не ИТ",
  "Менеджер по работе с ключевыми юридическими лицами": "не ИТ",
  "Старший менеджер обслуживания и развития": "не ИТ",
  "Младший специалист отдела премиального обслуживания": "не ИТ",
  "Эксперт клиентского сервиса": "не ИТ",
  "Старший менеджер по привлечению среднего бизнеса": "не ИТ",
  "Главный специалист отдела претензий": "не ИТ",
  "Диспетчер такси": "не ИТ",
  "Оператор отдела обработки данных": "Аналитика данных/ Data Science",
  "Менеджер отдела сохранения клиентов": "не ИТ",
  "Руководитель направления оценки персонала": "не ИТ",
  "Старший архитектор": "Разработка",
  "Главный менеджер группы корпоративных продаж": "не ИТ",
  "Менеджер развития программ лояльности": "не ИТ",
  "Специалист технической поддержки инвестиционных и брокерских счетов": "Администрирование",
  "Специалист по верификации": "не ИТ",
  "Менеджер-курьер": "не ИТ",
  "Заместитель руководителя отдела телемаркетинга": "не ИТ",
  "Оператор входящих заявок": "не ИТ",
  "Верификатор отдела залогового кредитования": "не ИТ",
  "Оператор и наставник": "не ИТ",
  "Главный разработчик": "Разработка",
  "Эксперт отдела контроля и развития процессов": "не ИТ",
  "Специалист верификации": "не ИТ",
  "Руководитель отдела сопровождения": "не ИТ",
  "Специалист отдела развития продуктовых направлений": "не ИТ",
  "Оператор баз данных": "Администрирование",
  "Ведущий инженер": "Разработка",
  "Менеджер по внутренним коммуникациям": "не ИТ",
  "Support Manager": "Администрирование",
  "Менеджер по развитию партнерской сети": "не ИТ",
  "Специалист по государственным закупкам": "не ИТ",
  "Оператор по продаже услуг бизнесу": "не ИТ",
  "Специалист отдела обслуживания физических лиц": "не ИТ",
  "Специалист по обслуживанию привелигированных клиентов": "не ИТ",
  "Менеджер по отношениям с инвесторами": "не ИТ",
  "Аналитик отдела портфельного анализа": "не ИТ",
  "Внутренний аудитор": "не ИТ",
  "Контент-менеджер": "не ИТ",
  "Эксперт по бизнесу": "не ИТ",
  "Оператор по кредитным картам": "не ИТ",
  "Специалист по электронной коммерции": "не ИТ",
  "Специалист отдела бухгалтерских услуг": "не ИТ",
  "Эксперт отдела оптимизации и планировании ресурсов": "не ИТ",
  "Менеджер по маркетинговым коммуникациям": "не ИТ",
  "Менеджер малого бизнеса": "не ИТ",
  "IT Auditor": "Администрирование",
  "Эксперт отдела омниканальных обращений": "не ИТ",
  "Старший инвестиционный менеджер": "не ИТ",
  "Специалист группы обработки чатов": "не ИТ",
  "Ведущий специалист отдела оценки контроля и сопровождения": "не ИТ",
  "Специалист отдела регитрации документов": "не ИТ",
  "Старший специалист группы контроля инвестиционных сервисов": "не ИТ",
  "Специалист отдела поддержки продуктов": "не ИТ",
  "Специалист по работе с должниками": "не ИТ",
  "Менеджер отдела сопровождения продаж": "не ИТ",
  "Специалист отдела по развитию региона": "не ИТ",
  "Старший программист-аналитик": "Разработка",
  "Ведущий специалист группы Compliance": "не ИТ",
  "Старший специалист отдела кредитования": "не ИТ",
  "Бухгалтер по начислению заработной платы": "не ИТ",
  "Персональный менеджер отдела по работе с юридическими лицами": "не ИТ",
  "Оператор по согласованию заявок": "не ИТ",
  "Главный специалист отдела обучения": "не ИТ",
  "Территориальный руководитель по развитию партнерской сети": "не ИТ",
  "Менеджер по сопровождению бизнес-клиентов": "не ИТ",
  "Супервайзер": "не ИТ",
  "Специалист группы сопровождения сбоев": "Администрирование",
  "Ведущий специалист отдела контроля качества": "не ИТ",
  "Менеджер по технической поддержке": "Администрирование",
  "Эксперт обработки входящих обращений": "не ИТ",
  "Менеджер по обслуживанию юридических лиц": "не ИТ",
  "Ведущий инженер по работе с бизнесом": "Разработка",
  "Специалист по отчетности": "не ИТ",
  "Специалист по информационным технологиям": "Администрирование",
  "Специалист горячей линии": "не ИТ",
  "Младший специалист отдела выявления и анализа сомнительных операций": "не ИТ",
  "Специалист по тестированию": "Тестирование",
  "Специалист обработки запросов": "не ИТ",
  "Руководитель отдела технической поддержки": "Администрирование",
  "Сейлз": "не ИТ",
  "Руководитель группы поиска клиентов": "не ИТ",
  "Client Support Manager": "не ИТ",
  "Оператор по продажам кредитных продуктов": "не ИТ",
  "Руководитель сектора продаж и проектов": "не ИТ",
  "Менеджер по работе с постоянными клиентами": "не ИТ",
  "Эксперт по работе с документацией": "не ИТ",
  "Ведущий специалист открытия счетов": "не ИТ",
  "Директор по развитию": "не ИТ",
  "Супервайзер отдела продаж": "не ИТ",
  "Заместитель директора по персоналу": "не ИТ",
  "Методолог ИТ-обучения": "не ИТ",
  "Специалист сектора обработки входящих обращений": "не ИТ",
  "Отдел обработки персональных данных": "не ИТ",
  "Младший специалист сектора непрерывного обслуживания": "не ИТ",
  "Специалист по ИТ": "Администрирование",
  "Специалист запросов от спецслужб": "не ИТ",
  "Performance Manager": "не ИТ",
  "Инженер сопровождения антифрод-систем": "Администрирование",
  "Сотрудник отдела верификации": "не ИТ",
  "Оператор производственно-логистического центра": "не ИТ",
  "Эксперт по развитию бизнес-процессов": "не ИТ",
  "IT специалист": "Администрирование",
  "Руководитель отдела по расчету с сотрудниками и отчетности": "не ИТ",
  "Главный специалист разработки методологии обучения": "не ИТ",
  "Специалист по исходящим обращениям": "не ИТ",
  "Руководитель группы андеррайтеров": "не ИТ",
  "Аналитик качества данных": "Аналитика данных/ Data Science",
  "Эксперт поддержки продуктов для юридических лиц": "не ИТ",
  "Специалист поддержки инвестиционных продуктов": "не ИТ",
  "Специалист мониторинга": "не ИТ",
  "Разработчик бизнес правил": "Разработка",
  "Руководитель группы продаж рассчетно-кассового оборудования": "не ИТ",
  "Ведущий специалист отдела оптимизации и поддержки процессов": "не ИТ",
  "Руководитель направления по мотивации": "не ИТ",
  "DevOps Engineer": "Администрирование",
  "Тестировщик": "Тестирование",
  "Менеджер интернет-проекта": "не ИТ",
  "Идеолог системы обучения продаж": "не ИТ",
  "Ночной специалист по обработке данных": "Аналитика данных/ Data Science",
  "Специалист технической поддержки 2 линии": "Администрирование",
  "Функциональный тренер": "не ИТ",
  "Менеджер по обслуживанию B2B": "не ИТ",
  "React Developer": "Разработка",
  "Verification Specialist": "не ИТ",
  "HR": "не ИТ",
  "Начальник отдела сопровождения": "не ИТ",
  "Ведущий специалист по продажам": "не ИТ",
  "SME Manager": "не ИТ",
  "Специалист отдела развития процессов": "не ИТ",
  "Специалист по эквайрингу": "не ИТ",
  "Руководитель филиала банка": "не ИТ",
  "Менеджер по автострахованию": "не ИТ",
  "Младший специалист по обеспечению качества": "Тестирование",
  "Оператор 1С": "не ИТ",
  "Главный специалист по работе с корпоративными клиентами": "не ИТ",
  "Специалист контроля качества отдела обработки данных": "не ИТ",
  "Ведущий менеджер по корпоративным продажам": "не ИТ",
  "Ведущий специалист отдела информационной поддержки и оптимизации": "не ИТ",
  "Эксперт поддержки сотрудников": "не ИТ",
  "Старший специалист сектора анализа разговоров": "не ИТ",
  "Заместитель начальника отдела среднего и крупного бизнеса": "не ИТ",
  "Senior QA Engineer": "Тестирование",
  "Менеджер по привлечению среднего и крупного бизнеса": "не ИТ",
  "Старший Java-разработчик": "Разработка",
  "Главный специалист отдела": "не ИТ",
  "Архитектор": "Разработка",
  "Data Scientist": "Аналитика данных/ Data Science",
  "Менеджер по удержанию клиентов": "не ИТ",
  "Оператор проверки": "не ИТ",
  "Ведущий специалист по отчетности валютного контроля": "не ИТ",
  "Ведущий программист-аналитик": "Разработка",
  "Ведущий специалист отдела кредитования": "не ИТ",
  "Оператор по обработке обращений": "не ИТ",
  "Ведущий администратор систем хранения данных": "Администрирование",
  "Старший администратор": "Администрирование",
  "Junior QA Engineer": "Тестирование",
  "Специалист отдела контроля доступа": "не ИТ",
  "Специалист сопровождения": "не ИТ",
  "Удаленный менеджер по продажам": "не ИТ",
  "Digital Manager": "не ИТ",
  "Удаленный оператор": "не ИТ",
  "Маркетолог": "не ИТ",
  "Брокер": "не ИТ",
  "Региональный руководитель группы ВСП": "не ИТ",
  "Начальник управления продаж клиентам малого бизнеса": "не ИТ",
  "Исполнительный директор": "не ИТ",
  "Юрисконсульт, Ведущий юрисконсульт": "не ИТ",
  "Территориальный менеджер по продажам продуктов глобальных рынков": "не ИТ",
  "Старший инженер по сопровождению АС": "Администрирование",
  "Старший специалист РКЦ": "не ИТ",
  "Старший клиентский менеджер": "не ИТ",
  "Главный инженер по разработке, Руководитель направления": "Разработка",
  "Senior Product designer": "Дизайн",
  "Директор управления": "не ИТ",
  "Ведущий специалист судебного и исполнительного производства Новосибирского отделения 8047 ПАО Сбербанк": "не ИТ",
  "Контролер кассир": "не ИТ",
  "Руководитель внутреннего структурного подразделения": "не ИТ",
  "Персональный менеджер премьер": "не ИТ",
  "Финансовый советник СберПервый": "не ИТ",
  "DevOps-инженер": "Администрирование",
  "Помощница начальницы менеджеров отдела  эквайринга": "не ИТ",
  "Директор проектов Центральный аппарат": "не ИТ",
  "Менеджер It-продуктов": "не ИТ",
  "Главный инженер по надзору за строительством": "не ИТ",
  "Менеджер по обслуживанию": "не ИТ",
  "Ведущий специалист по обслуживанию клиентов": "не ИТ",
  "Главный инженер": "не ИТ",
  "Менеджер выездного сервиса": "не ИТ",
  "Delivery Lead (Главный инженер)": "Администрирование",
  "Начальник отдела недвижимости и развития инфоаструктуры": "не ИТ",
  "Специалист Сектора брокерского обслуживания": "не ИТ",
  "Главный инспектор": "не ИТ",
  "Старший менеджер по обслуживанию": "не ИТ",
  "Старший менеджер по развитию малого бизнеса": "не ИТ",
  "HR Business Partner": "не ИТ",
  "Специалист контакт центра": "не ИТ",
  "Заместитель управляющего филиалом": "не ИТ",
  "Главный менеджер по работе с  ключевыми клиентами": "не ИТ",
  "Старший инженер по разработке": "Разработка",
  "Персональный менеджер СберПремьер": "не ИТ",
  "Ведущий it инжинер": "Администрирование",
  "Эксперт по цифровым технологиям аудита": "не ИТ",
  "Главный инженер по мат обеспечению ДКА": "не ИТ",
  "Помощник/ассистент руководителя": "не ИТ",
  "Начальник административного отдела": "не ИТ",
  "Главный специалист по работе с предприятиями и партнерами/ Главный специалист отдела по работе с проблемной задолженностью/кредитный специалист": "не ИТ",
  "Диспетчер по заявкам": "не ИТ",
  "Старший менеджер по привлечению корпаротивных клиентов ММБ": "не ИТ",
  "Юрисконсульт": "не ИТ",
  "Менеджер по привлечению корпоративных  клиентов": "не ИТ",
  "Специалист по работе с проблемными активами малого и микробизнеса": "не ИТ",
  "Старший клиентский менеджер по работе с клиентами": "не ИТ",
  "Старший менеджер по обслуживанию физ. и юр.лиц": "не ИТ",
  "Ведущий специалист по обслуживанию физических лиц": "не ИТ",
  "Риск-менеджер сбербанк": "не ИТ",
  "Начальник отдела по работе с проблемной задолженностью": "не ИТ",
  "Инженер по сопровождению и эксплуатации": "Администрирование",
  "Специалист центра оперативной поддержки ипотечного кредитования": "не ИТ",
  "Старший менеджер": "не ИТ",
  "Старший менеджер по обслуживанию клиентов": "не ИТ",
  "Ст.кассир": "не ИТ",
  "Senior Project manager / Senior Delivery Lead": "не ИТ",
  "Старший менеджер по работе с клиентамм": "не ИТ",
  "Сотрудник УБ": "не ИТ",
  "Ведущий специалист по обслуживанию физ лиц": "не ИТ",
  "Ведущий специалист судебного и исполнительного производства": "не ИТ",
  "Ведущий специалист группы противодейстия мошенничеству": "не ИТ",
  "Старший кассир": "не ИТ",
  "Инспектор службы экономической безопасности": "не ИТ",
  "Заместитель руководителя дополнительного офиса": "не ИТ",
  "Специалист отдела судебного исполнительного производства (стажер)": "не ИТ",
  "Менеджер по ипотечному кредитованию": "не ИТ",
  "Водитель инкассатор": "не ИТ",
  "Кассир-операционист": "не ИТ",
  "Ведущий инспектор": "не ИТ",
  "Младший специалист по работе с корпоративными клиентами": "не ИТ",
  "Руководитель дополнительного офиса": "не ИТ",
  "Управляющий директор": "не ИТ",
  "Старший менеджер по обслуживанию физических лиц": "не ИТ",
  "Финансовый контролер": "не ИТ",
  "Главный менеджер по работе с ключевыми клиентами Управления развития малого бизнеса": "не ИТ",
  "Старший менеджер по работе с клиентами Дополнительный офис № 5940/0123 Уральского банка": "не ИТ",
  "Ведущий менеджер по сервису": "не ИТ",
  "Главный специалист по продажам": "не ИТ",
  "Контролер-кассир": "не ИТ",
  "Региональный руководитель группы": "не ИТ",
  "Старший менеджер по обслуживанию физических лиц и юридических лиц": "не ИТ",
  "Старший инспектор": "не ИТ",
  "Начальник административно-хозяйственного отдела": "не ИТ",
  "Заместитель руководителя офиса": "не ИТ",
  "Старший аудитор IT розничного бизнеса": "Администрирование",
  "Менеджер по обслуживанию ипотечных кредитов": "не ИТ",
  "Заместитель руководителя филиала": "не ИТ",
  "Секретарь руководителя организационного сектора административного отдела": "не ИТ",
  "Кредитный инспектор Отдела кредитования физических лиц": "не ИТ",
  "Сервис-менеджер": "не ИТ",
  "Главный инженер-разработчик": "Разработка",
  "ABAP/BackEnd разработчик": "Разработка",
  "Старший менеджер по обслуживани": "не ИТ",
  "Корпоративный психолог": "не ИТ",
  "Экономист": "не ИТ",
  "Community manager (IT)": "не ИТ",
  "Сетевой инженер": "Администрирование",
  "Клиентский менеджер прямых продаж": "не ИТ",
  "Ведущий юрисконсульт": "не ИТ",
  "Старший специалист сектора расследования претензий физических лиц по банковским картам": "не ИТ",
  "Менеджер по сделкам с недвижимостью Домклик": "не ИТ",
  "Старший охранник": "не ИТ",
  "Frontend Engineer (Команда платформы «Пульс»)": "Разработка",
  "Эксперт отдела подбора и адаптации персонала": "не ИТ",
  "Ассистент клиентского менеджера": "не ИТ",
  "Ведущий менеджер по работе с корпоративными клиентами.": "не ИТ",
  "Старший менеджер по привлечению корпоративных клиентов": "не ИТ",
  "Менеджер по кадровому делопроизводству": "не ИТ",
  "Специалист по работе с обращениями клиентов": "не ИТ",
  "Начальник отдела": "не ИТ",
  "Менеджер отдела служебных расследований и дисциплинарных взысканий": "не ИТ",
  "Специалист по учету договоров аренды": "не ИТ",
  "Начальник сектора продаж": "не ИТ",
  "Руководитель направления разработки ИИ решений": "Разработка",
  "Старший специалист управления недвижимостью": "не ИТ",
  "Стажер Риск-Аналитик": "Аналитика данных/ Data Science",
  "Исполнительный директор (CEO) Центра Компетенций финансовое планирование и контроль": "не ИТ",
  "Тереториальный менеджер по работе с партнерами": "не ИТ",
  "Старший эксперт/Руководитель направления": "не ИТ",
  "Руководитель центра персонального обслуживания": "не ИТ",
  "Руководитель бизнес-офиса": "не ИТ",
  "Специалист отдела повышения квалификации": "не ИТ",
  "Менеджер зарплатного проекта": "не ИТ",
  "Ведущий юрисконсульт отдела правового обеспечения среднего и крупного бинеса": "не ИТ",
  "Ведущий инженер-разработчик": "Разработка",
  "Премиальный менеджер": "не ИТ",
  "Инженер по нагрузочному тестированию": "Тестирование",
  "Менеджер по заключению и сопровождению договоров на прием платежей от физических лиц в пользу юридических": "не ИТ",
  "AI/ML инженер": "Аналитика данных/ Data Science",
  "Старший специалист операционной поддержки": "не ИТ",
  "Руководитель Премьер Офиса": "не ИТ",
  "Руководитель проектов цифровой трансформации Управления корпоративных клиентов (КСБ)": "не ИТ",
  "Старший менеджер по обсуживанию": "не ИТ",
  "Middle QA Engineer": "Тестирование",
  "Специалист по правовым вопросам": "не ИТ",
  "Ведущий продуктовый дизайнер": "Дизайн",
  "Специалист по сопровождению офисов Направления продаж и обслуживания в сети ВСП Среднерусского Банка г. Москва": "не ИТ",
  "Ведущий инженер по разработке": "Разработка",
  "Аналитик клиентского опыта": "не ИТ",
  "Старший менеджер по обслуживанию,": "не ИТ",
  "Наставник новых сотрудников": "не ИТ",
  "Ст.инспектор валютного контроля отдела валютных и неторговых операций": "не ИТ",
  "Старший менеджер по обучению клиентов": "не ИТ",
  "Вице-президент": "не ИТ",
  "Региональный менеджер агентского канала": "не ИТ",
  "Специалист по обработке входящих обращений сотрудников": "не ИТ",
  "Руководитель направления ИТ (": "Администрирование",
  "Главный инженер по разработке": "Разработка",
  "Эксперт по контролю операций": "не ИТ",
  "Project manager |": "не ИТ",
  "Заместитель руководителя офисом": "не ИТ",
  "Ведущий специалист по работе с ключевыми клиентами, менеджер по работе с клиентами в офисе": "не ИТ",
  "Менеджер сопровождения ипотеки": "не ИТ",
  "Старший инженер ИТ-поддержки": "Администрирование",
  "Старший специалист взыскания задолженности": "не ИТ",
  "Заместитель Управляющего Волгоградским отделением": "не ИТ",
  "Скм": "не ИТ",
  "Стажировка": "не ИТ",
  "Кредитный инспектор по сопровождению кредитных операция юридических лиц": "не ИТ",
  "Ведущий специалист контактного центра": "не ИТ",
  "Ведущий специалист отдела по работе с проблемными активами": "не ИТ",
  "Главный специалист отдела судебного и исполнительного производства": "не ИТ",
  "Стажер Python-разработчик": "Разработка",
  "Старший менеджер по обслуживанию физических и юридических лиц": "не ИТ",
  "Middle Java Developer": "Разработка",
  "Ведущий специалист отдела досудебного взыскания": "не ИТ",
  "Менеджер зарплатных проектов": "не ИТ",
  "Старший специалист отдела по работе с мошенничеством": "не ИТ",
  "Руководитель премьер-офиса": "не ИТ",
  "Специалист офиса": "не ИТ",
  "Менеджер направления": "не ИТ",
  "Старший специалист внутрибанквского учёта и отчетности": "не ИТ",
  "Клиентский менеджер Премьер": "не ИТ",
  "Старший специалист группы отложенных операций и сохранения клиентов": "не ИТ",
  "Старший водитель инкассатор": "не ИТ",
  "Главный специалист ЕРКЦ": "не ИТ",
  "Менеджер по работе с ключевым клиентами": "не ИТ",
  "Стажёр разработчик": "Разработка",
  "Руководитель направления отдела обучения": "не ИТ",
  "Охранник": "не ИТ",
  "Старший специалист по обращениям физических лиц": "не ИТ",
  "Дистанционный клиентский менеджер": "не ИТ",
  "Ведущий специалист подготовки данных для машинного обучения": "Аналитика данных/ Data Science",
  "Дизайнер департамента": "Дизайн",
  "Старший клиентский менеджер по работе с корпоративными клиентами КСБ": "не ИТ",
  "Главный специалист ОДВЗ": "не ИТ",
  "Директор по дизайну": "Дизайн",
  "Исполнительный директор по внешней дистрибуции": "не ИТ",
  "Смрк": "не ИТ",
  "Старший специалист группы контроля": "не ИТ",
  "Старший водитель-инкассатор": "не ИТ",
  "Старший специалист по работе с корпоративными клиентами": "не ИТ",
  "Руководитель направление внешней дистрибуции": "не ИТ",
  "Старший инженер по нагрузочному тестированию": "Тестирование",
  "IT Analyst": "Аналитика данных/ Data Science",
  "Главный менеджер по ключевым клиентам": "не ИТ",
  "Начальник отдела кредитования малого бизнеса": "не ИТ",
  "Директор Регионального центра": "не ИТ",
  "Клиентский менеджер отдела телемаркетинга": "не ИТ",
  "Ведущий специалист по взысканию": "не ИТ",
  "Исполнительный  директор": "не ИТ",
  "Старший специалист отдела технических средств охраны управления безопасности": "не ИТ",
  "Senior developer, Team Lead": "Разработка",
  "Официант": "не ИТ",
  "Зам.начальника отдела/Менеджер по продажам и сопровождению платежных сервисов": "не ИТ",
  "Главный кредитный аналитик": "не ИТ",
  "Начальник  сектора обслуживания корпоративных клиентов": "не ИТ",
  "QA": "Тестирование",
  "Старший экономист отдела корпоративных клиентов": "не ИТ",
  "Инженер-программист": "Разработка",
  "Разработчик Android приложения для биометрии": "Разработка",
  "Старший специалист сектора обработки обращений физических лиц": "не ИТ",
  "Frontend developer React": "Разработка",
  "Специалист по работе с VIP клиентами": "не ИТ",
  "Контролировать-кассир": "не ИТ",
  "Эксперт по анализу клиентского опыта / Фрод-аналитик (экспертная группа Antifraud)": "не ИТ",
  "Специалист по обслуживанию частных лиц": "не ИТ",
  "Начальник сектора": "не ИТ",
  "Ст. Специалист по борьбе с мошенничеством": "не ИТ",
  "Delivery manager (Розница, B2C)": "Администрирование",
  "Ведущий специалист по работе с персоналом": "не ИТ",
  "Руководитель направления по развитию ИИ": "Разработка",
  "Начальник отдела информационных технологий": "Администрирование",
  "Руководитель Премиального офиса": "не ИТ",
  "Главный специалист отдела банковского сопровождения инвестиционный проектов": "не ИТ",
  "Руководитель IT проектов": "Администрирование",
  "Главный инженер по сопровождению EDR": "Администрирование",
  "Ведущий специалист по обслуживанию частных лиц": "не ИТ",
  "Исполнительный директор-начальник отдела по работе с корпоративными клиентами крупного среднего бизнеса": "не ИТ",
  "Ведущий менеджер по привлечению": "не ИТ",
  "Ведущий специалист отдела судебного и исполнительного производства": "не ИТ",
  "Delivery-lead": "Администрирование",
  "Менеджер по сервисной поддержке продуктов эквайринга": "не ИТ",
  "Эксперт строитель": "не ИТ",
  "Сервис- менеджер": "не ИТ",
  "Аналитик 1С": "не ИТ",
  "Персональный клиентский менеджер": "не ИТ",
  "Специалист-диспетчер": "не ИТ",
  "Старший менеджер по обучению учебного центра/ Центра развития талантов": "не ИТ",
  "Java разработчик": "Разработка",
  "Исполнительный директор (CEO)": "не ИТ",
  "Главный юрисконсульт юридического отдела": "не ИТ",
  "Специалист по зарплатным проектам": "не ИТ",
  "Специалист отдела корпоративных клиентов и бюджетов": "не ИТ",
  "Начальник отдела, направление организации расследований (безопасность)": "не ИТ",
  "Андеррайтер СМБ": "не ИТ",
  "Ведущий специалист судебного отдела": "не ИТ",
  "Инженер по кибербезопасности": "Администрирование",
  "Заместитель управляющего": "не ИТ",
  "Старший менеджер по сделкам с недвижимостью": "не ИТ",
  "Старший инженер по сопровождению": "Администрирование",
  "Ведущий инженер по тестированию": "Тестирование",
  "Python-разработчик": "Разработка",
  "Главный инженер проекта": "не ИТ",
  "Ведущий специалист по обслуживанию клиентов ЕРКЦ": "не ИТ",
  "Заместитель начальника отдела": "не ИТ",
  "Заместитель начальника отдела охраны": "не ИТ",
  "Старший специалист сектора обращений физических лиц": "не ИТ",
  "Старший специалист банка": "не ИТ",
  "Руководитель направления / Исполнительный директор по исследованию данных в Центре корпоративного риск-моделирования": "Аналитика данных/ Data Science",
  "Старший специалист по кредитованию": "не ИТ",
  "Руководитель отделения": "не ИТ",
  "Начальник управления продаж и обслуживания сети": "не ИТ",
  "Директор офиса": "не ИТ",
  "Менеджер по сделкам с недвижимостью": "не ИТ",
  "Менеджер по работе с ключевыми ЮЛклиентами": "не ИТ",
  "Инженер по эксплуатации": "не ИТ",
  "Главный специалист Отдела ведения НСИ управления бухгалтерского учета и отчетности": "не ИТ",
  "Ведущий инженер по охране окружающей среды": "не ИТ",
  "Кредитный инспектор Управления кредитования и проектного  финансирования Юго-Западного  Банка": "не ИТ",
  "Директор юридического управления": "не ИТ",
  "Директор филиала": "не ИТ",
  "Старший кассир-операционист": "не ИТ",
  "Руководитель всп": "не ИТ",
  "Менеджер по оформлению документов": "не ИТ",
  "Бухгалтер,специалист.": "не ИТ",
  "Главный инженер по сопровождению": "Администрирование",
  "Ведущий контролер": "не ИТ",
  "Персональный менеджер офиса Сбербанк Первый по работе с VIP клиентами (физические лица)": "не ИТ",
  "Начальник смены": "не ИТ",
  "UX/UI-дизайнер": "Дизайн",
  "Главный аудитор": "не ИТ",
  ", специалист службы заботы о клиентах": "не ИТ",
  "Главный эксперт по цифровым технологиям аудита": "не ИТ",
  "Начальник управления ЦКП": "не ИТ",
  "Специалист по сервисной поддержке продуктов эквайринга": "не ИТ",
  "Ведущий инженер ППРБ.Комиссия/КВ, ЕФС.РМ ОЦ": "Администрирование",
  "Старший специалист отдела отчетности": "не ИТ",
  "Врач-куратор": "не ИТ",
  "Главный менеджер по работе с ключевыми клиентами": "не ИТ",
  ", ведущий юрисконсульт": "не ИТ",
  "Клиентский менеджер Сбер Премьер": "не ИТ",
  "Руководитель направления по исследованию данных. Продуктовый аналитик.": "Аналитика данных/ Data Science",
  "Senior IT Recruiter": "не ИТ",
  "Старший специалист по сделкам с недвижимостью": "не ИТ",
  "QA Manual Engineer": "Тестирование",
  "Менеджер по зарплатным проектам": "не ИТ",
  "Менеджер по продаже": "не ИТ",
  "Персональный менеджер Премьер Центр Персонального обслуживания": "не ИТ",
  "Группа обслуживания вызовов": "не ИТ",
  "Руководитель направления блок Технологии": "Администрирование",
  "Менеджер проекта Департамента по работе с проблемными активами": "не ИТ",
  "Специалист прямых продаж": "не ИТ",
  "Fullstack QA / SDET (Java)": "Тестирование",
  "Ведущий специалист по залоговым операциям": "не ИТ",
  "Владелец продукта": "не ИТ",
  "Ведущий специалист операционной поддержки": "не ИТ",
  "Ведущий специалист отдела технических средств охраны Управления безопасности Пермского отделения №6984": "не ИТ",
  "HRBP": "не ИТ",
  "Java-программист": "Разработка",
  "Старший специалист по взысканию задолженности на ранних стадиях": "не ИТ",
  "Инвестиционный страховой менеджер": "не ИТ",
  "Старший специалист отдела делового администрирования": "не ИТ",
  "Менеджер ключевых клиентов": "не ИТ",
  "Ведущий инженер административного отдела": "не ИТ",
  "Финансовый  консультант": "не ИТ",
  "Старший менеджер офисов домклик": "не ИТ",
  "Руководитель направления по тестированию, Менеджер по тестированию": "Тестирование",
  "Ведущий инженер разработчик": "Разработка",
  "Начальник сектора продаж клиентам малого бизнеса": "не ИТ",
  "Middle DevOps": "Администрирование",
  "Начальник сектора ценных бумаг": "не ИТ",
  "Главный бухгалтер": "не ИТ",
  "Специалист зарплатного проекта": "не ИТ",
  "Психолог": "не ИТ",
  "Главный инженер по эксплуатации инженерных систем": "не ИТ",
  "Заведующий складом": "не ИТ",
  "Старший менеджер по продажам.": "не ИТ",
  "Бизнес-аналитик команды аналитики HR-процессов.": "не ИТ",
  "Ведущий специалист ВИП линии": "не ИТ",
  "Начальник отдела экономической безопасности": "не ИТ",
  "Специалист Отдела ведения нормативно-справочной информации": "не ИТ",
  "Заместитель начальника Административного Управления": "не ИТ",
  "SQL-разработчик": "Разработка",
  "Менеджер по продаже зарплатного проекта": "не ИТ",
  "Главный клиентский менеджер по работе с ключевыми клиентами среднего и малого бизнеса": "не ИТ",
  "Начальник сектора продаж розничных продуктов": "не ИТ",
  "Ведущий специалист по взысканию задолженности": "не ИТ",
  "Руководитель направления, Управление развития технологических инициатив": "не ИТ",
  "Региональный менеджер отдела внешней дистрибуции": "не ИТ",
  "Старший менеджер по привлечению КК": "не ИТ",
  "Intern Data Engineer": "Аналитика данных/ Data Science",
  "Шеф-редактор ai тренеров": "не ИТ",
  "Менеджер  зарплатных проектов": "не ИТ",
  "Водитель": "не ИТ",
  "Руководитель направления по работе с региональным госсектором(GR)": "не ИТ",
  "Заместитель руководителя дополнительного офиса.  Розничный блок": "не ИТ",
  "Старший iOS-разработчик": "Разработка",
  "Контролер": "не ИТ",
  "Старший специалист управления недвижимостью.": "не ИТ",
  "Старший клиентский менеджеи": "не ИТ",
  "Специалист по сопровождению договоров": "не ИТ",
  "Главный эксперт по развитию ВЭД": "не ИТ",
  "Специалист центра оперативной поддержки партнёров и ипотечного кредитования": "не ИТ",
  "Product lead (": "не ИТ",
  "Консультант банковских продуктов": "не ИТ",
  "Ведущий менеджер, руководитель": "не ИТ",
  "Региональный руководитель группы офисов": "не ИТ",
  "Региональный руководитель отдела продаж": "не ИТ",
  "Начальник Центра охраны и реагирования": "не ИТ",
  "Дистанционный клиентский менеджер Премьер": "не ИТ",
  "Старший специалист сервиса экосистемы": "не ИТ",
  "Главный инспектор внутрибанковской безопасности": "не ИТ",
  "Специалист по сервисной поддержке продуктов эквайринга Отдела по сервисной поддержке продуктов эквайринга Управления «Безналичные решения» Башкирского отделения № 8598 (г. Уфа)": "не ИТ",
  "Инженер по проектированию ИИ-запросов": "Разработка",
  "Старший специалист управления судебного и внесудебного взыскания": "не ИТ",
  "Руководитель группы обучения": "не ИТ",
  "Начальник отдела кассовых операций": "не ИТ",
  "Начальник сектора управления прямых продаж": "не ИТ",
  "Креативный менеджер": "не ИТ",
  "Руководитель филиала": "не ИТ",
  "Начальник отдела организации кредитования клиентов малого бизнеса": "не ИТ",
  "Старший кредитный менеджер": "не ИТ",
  "Начальник отдела инкассации": "не ИТ",
  "Специалист по мониторингу": "не ИТ",
  "Главный инженер разработки": "Разработка",
  "Художник-иллюстратор": "Дизайн",
  "Клиентский менеджер Сбербанк Премьер": "не ИТ",
  "Аналитик Social Media Listening": "не ИТ",
  "Дистанционный менеджер премиальных клиентов": "не ИТ",
  "Менеджер направления Управления внутреннего аудита": "не ИТ",
  "Главный  инженер по сопровождению": "Администрирование",
  "Сотрудник колцентра": "не ИТ",
  "Руководитель. Сети Премьер": "не ИТ",
  "Инспектор по расследованию претензий": "не ИТ",
  "Водитель - инкассатор": "не ИТ",
  "Ведущий экономист планово-экономического отдела": "не ИТ",
  "Старший специалист управления безопасности": "не ИТ",
  "Python разработчик AI агентов": "Разработка",
  "Руководитель направления в Департаменте интегрированного риск-менеджмента": "не ИТ",
  "Менеджер премьер": "не ИТ",
  "Junior Java разработчик": "Разработка",
  "Мультипликатор/наставник": "не ИТ",
  "Стажер по работе с проблебными активами": "не ИТ",
  "Продюсер": "не ИТ",
  "Региональный менеджер отдела продаж": "не ИТ",
  "Старший специалист по урегулированию вопросов с проблемными активами малого и микробизнеса": "не ИТ",
  "Ведущий инженер группы сопровождения переводов": "Администрирование",
  "Ведущий менеджер по обслуживанию": "не ИТ",
  "Старший специалист по обработке обращений клиентов": "не ИТ",
  "Специалист сервисной службы": "не ИТ",
  "Стажер, управление модельных рисков": "Аналитика данных/ Data Science",
  "Специалист сервисной поддержки торгового эквайринга. Управление Безналичные решения.": "не ИТ",
  "Инженер по разработке": "Разработка",
  "Специалист по обслуживанию физических и юридических лиц": "не ИТ",
  "Senior Java/Kotlin Backend Engineer": "Разработка",
  "Руководитель офиса по обслуживанию юридических и физических лиц": "не ИТ",
  "Старший инженер по эксплуатации": "Администрирование",
  "Сбер Агент": "не ИТ",
  "Ведущий специалист центра подготовки данных для машинного обучения": "Аналитика данных/ Data Science",
  "Ведущий менеджер обслуживания": "не ИТ",
  "Инженер-исследователь": "Разработка",
  "Региональный менеджер-начальник отдела": "не ИТ",
  "Ведущий специалист корпаротивного направления": "не ИТ",
  "Дистанционный клиентский менеджер отдел обслуживания ключевых клиентов": "не ИТ",
  "Специалист дистанционных продаж": "не ИТ",
  "Старший менеджер активных продаж": "не ИТ",
  "UX specialist": "Дизайн",
  "Секретарь-делопроизводитель": "не ИТ",
  "Старший специалист отдела телемаркетинга": "не ИТ",
  "Главный специалист управления недвижимости": "не ИТ",
  "Главный специалист отделения банка": "не ИТ",
  "Старший специалист СБ": "не ИТ",
  "Ведущий инженер по сопровождению УРМ г. Екатеринбург управления сопровождения сервисов Департамента ИТ блока \"Сеть продаж\"": "Администрирование",
  "Начальник центра закупок": "не ИТ",
  "Ведущий инженер-разработчик (Data Science)": "Аналитика данных/ Data Science",
  "Менеджержер по работе с клиентами": "не ИТ",
  "Специалист по сопровождению процедур банкротства": "не ИТ",
  "Аналитик отдела подбора и привлечение персонала": "не ИТ",
  "Старший кредитный аналитик": "не ИТ",
  "Менеджер по продажам безналичных решений": "не ИТ",
  "Руководитель отдела по работе с ключевыми клиентами": "не ИТ",
  "Менеджер отдела прямых продаж": "не ИТ",
  "Администратор сопровождения АС": "Администрирование",
  "Главный менеджер по работе с ключевыми клиентами (МСБ)": "не ИТ",
  "Ведущий специалист Отдела Поддержки Процессов Обслуживания Устройств Самообслуживания": "не ИТ",
  "Старший QA-engineer": "Тестирование",
  "Ведущий аудитор": "не ИТ",
  "Начальник сектора продаж ОПКМБ": "не ИТ",
  "Персональный менеджер Сбер Премьер": "не ИТ",
  "Главный менеджер по обучению": "не ИТ",
  "Кассир кассового -инкассаторского центра": "не ИТ",
  "Менеджер по обслуживанию клиентов": "не ИТ",
  "Старший эксперт по технологиям": "не ИТ",
  "Оператор поддержки чаты": "не ИТ",
  "Начальник Управления по работе с персоналом": "не ИТ",
  "Ведущий специалист по сопровождению зарплатных проектов Сектор продаж зарплатных проектов УПП \"Север\" Московского банка ПАО Сбербанк": "не ИТ",
  "Менеджер по продажам, Персональный менеджер Премьер": "не ИТ",
  "Ведущий инженер надзора за строительством": "не ИТ",
  "Старший (дежурный)инкассатор": "не ИТ",
  "Ведущий инженер сопровождения": "Администрирование",
  "Менеджер по качеству и технологической поддержки": "не ИТ",
  "Старший специалист по работе в социальных сетях и цифровых каналах": "не ИТ",
  "Бухгалтер-кассир": "не ИТ",
  "Senior Android Developer": "Разработка",
  "HR-бизнес-партнер (HR BP)": "не ИТ",
  "Ведущий специалист отдела по обслуживанию физических лиц": "не ИТ",
  "Бизнес партнер по развитию команд": "не ИТ",
  "Руководитель проектов по цифровому развитию клиентов (Customer Success Manager)": "не ИТ",
  "Менеджер по безопасности": "не ИТ",
  "Менеджер отдела развития": "не ИТ",
  "Старший специалист исполнительного производства": "не ИТ",
  "Заместитель руководителя ВСП": "не ИТ",
  "Главный специалист управления по решения вопросов корпоративных клиентов": "не ИТ",
  "Руководитель направления. Мастер системный аналитик.": "Разработка",
  "Руководитель направления -": "не ИТ",
  "Старший специалист по работе с мошенничеством": "не ИТ",
  "Территориальный менеджер по работе с партнерами": "не ИТ",
  "Senior продуктовый дизайнер": "Дизайн",
  "Руководитель отдела логистики": "не ИТ",
  "Fullstack-разработчик": "Разработка",
  "Support менеджер": "Администрирование",
  "Ведущий системный аналитик-архитектор": "Разработка",
  "Начальник отдела обслуживания вызовов": "не ИТ",
  "MLOps-инженер": "Аналитика данных/ Data Science",
  "Senior C++ разработчик": "Разработка",
  "CPO. Дивизион Лояльность": "не ИТ",
  "Главный специалист по работе с ключевыми клиентами, замещение руководителя": "не ИТ",
  "Руководитель группы офисов": "не ИТ",
  "Руководитель центра персонального обслуживания Сбербанк Премьер": "не ИТ",
  "Ведущий инженер Центра Робототехники": "Разработка",
  "Senior Java разработчик": "Разработка",
  "Специалист архива": "не ИТ",
  "Кредитный инспектор управления мониторинга кредитных операций и залогового обеспечения": "не ИТ",
  "Главный клиентский менеджер отраслевого направления \"Строительство, недвижимость\"": "не ИТ",
  "Главный специалист финансового отдела": "не ИТ",
  "Начальник сектора обслуживания корпоративных клиентов": "не ИТ",
  "Ведущий специалист по обслуживанию корпоративных клиентов": "не ИТ",
  "Помощник/ассистент руководителя ДИТ (2000 человек)": "не ИТ",
  "Менеджер по продажам зарплатных проектов": "не ИТ",
  "Заведующая складом": "не ИТ",
  "Старший менеджер обслуживания клиентов": "не ИТ",
  "Специалист по прямым продажам (DSA)": "не ИТ",
  "Менеджер по обучению и развитию": "не ИТ",
  "Главный экономист": "не ИТ",
  "КМ региональный государственный сектор СЗБ": "не ИТ",
  "Продакт-менеджер (техническое направление)": "не ИТ",
  "Старший контролёр": "не ИТ",
  "Главный инженер / SRE": "Администрирование",
  "Менеджер (Модель работы кампейна)": "не ИТ",
  "Стажер Бизнес-аналитик": "не ИТ",
  "Ведущий аудитор инженер по данным": "Аналитика данных/ Data Science",
  "Системный аналитик/Релиз менеджер": "Разработка",
  "Релиз-менеджер": "Администрирование",
  "Менеджер Управления маркетинга и коммуникаций": "не ИТ",
  "Менеджер по продажам,": "не ИТ",
  "Ведущий инженер по сопровождению": "Администрирование",
  "Старший Менеждер по обслуживанию физ лиц": "не ИТ",
  "Уборщик": "не ИТ",
  "Старший андеррайтер": "не ИТ",
  "Ведущий специалист (Оператор контактного центра)": "не ИТ",
  "Менеджер по сервисной поддержке": "не ИТ",
  "Инженер технических средств охраны": "не ИТ",
  "Software Quality Assurance Tester": "Тестирование",
  "Администратор проектов": "не ИТ",
  "Middle Разработчик Java (Ведущий инженер по разработке бизнес-процессов)": "Разработка",
  "Отраслевой координатор по фармацевтике и здравоохранению": "не ИТ",
  "Главный контролер": "не ИТ",
  "Специалист по сервисной поддержке": "не ИТ",
  "Клиентский менеджер-стажер": "не ИТ",
  "Специалист активных продаж": "не ИТ",
  "Старший специалист поддержки внутренних клиентов": "не ИТ",
  "Главный специалист отдела стоимостной экспертизы проектов строительства": "не ИТ",
  "Главный инженер по разработке, MLOps": "Аналитика данных/ Data Science",
  "Специалиаст": "не ИТ",
  "Персональный банкир": "не ИТ",
  "Специалист по обслуживанию физических лиц": "не ИТ",
  "Инженер сопровождения ПО": "Администрирование",
  "Старший инженер по разработке (тестирование ПО)": "Тестирование",
  "Старший водитель автомобиля предназначенного для Инкассирования ( перевозки) ценностей и корреспонденции. Отдел инкассации и перевозки ценностей": "не ИТ",
  "Campaign manager": "не ИТ",
  "Программист-разработчик (Kotlin/Java)": "Разработка",
  "Главный инженер сопровождения": "Администрирование",
  "Эксперт в развитии малого бизнеса": "не ИТ",
  "Руководитель направления (разработка)": "Разработка",
  "Ведущий специалист отдела машинного обучения": "Аналитика данных/ Data Science",
  "Старший специалист по работе с клиентами (чат поддержка)": "не ИТ",
  "Практикант менеджер": "не ИТ",
  "Менеджер по работе с ключевыми клиентами малого бизнеса": "не ИТ",
  "Ведущий специалист по исследованию данных": "Аналитика данных/ Data Science",
  "Заместитель управляющего отделением": "не ИТ",
  "Старший специалист отдела сбора просроченной задолженности": "не ИТ",
  "Специалист отдела банковских рассылок Сервисного центра \"Документооборот\"": "не ИТ",
  "Заместитель начальника кассовых операций и операционной работы": "не ИТ",
  "Руководитель группы поддержки клиентов": "не ИТ",
  "Специалист развития качества": "не ИТ",
  "Sales Network": "не ИТ",
  "Стажер (бизнес-анализ)": "не ИТ",
  "Старший специалист по разметке данных": "Аналитика данных/ Data Science",
  "Менеджер по сервисной поддержке продуктов эквайринга ключевых партнеров и E-com": "не ИТ",
  "CJE": "не ИТ",
  "Старший менеджер по оформлению кредитов": "не ИТ",
  "Главный специалист операционного отдела": "не ИТ",
  "Руководитель по отраслевому развитию недропользователей": "не ИТ",
  "Менеджер по работе с ключевыми клиентами малого и среднего бизнеса": "не ИТ",
  "Ведущий hr специалист": "не ИТ",
  "Оператор мониторинга": "не ИТ",
  "Старший специалист управления по работе с обращениями клиентов": "не ИТ",
  "ML-инженер": "Аналитика данных/ Data Science",
  "Главный ИТ инженер": "Администрирование",
  "Инкассатор-водитель": "не ИТ",
  "Руководитель направления в команде разработки HR продукта": "Разработка",
  "Менеджер по работе с клиентами ММБ": "не ИТ",
  "Главный юрист по работе с проблемными активами": "не ИТ",
  "Сотрудник ОПП": "не ИТ",
  "Клиентский менеджер сектора обслуживания корпоративных клиентов": "не ИТ",
  "Медиатор": "не ИТ",
  "Старший кредитный аналитик подразделения кредитования малого бизнеса": "не ИТ",
  "Секретарь руководителя": "не ИТ",
  "Главный инспектор отдела безопасности": "не ИТ",
  "DevOps": "Администрирование",
  "Старший государственный налоговый инспектор": "не ИТ",
  "Старший аудитор": "не ИТ",
  "Менеджер пр работе с корпоративными клиентами": "не ИТ",
  "Контролер-кассир, старший специалист отдела, менеджер ипотечного страхования": "не ИТ",
  "Менеджер проектов (Sberseasons)": "не ИТ",
  "Сотрудник колл центра": "не ИТ",
  "Страший менеджер по обслуживанию": "не ИТ",
  "Директор Дивизиона (Директор Управления)": "не ИТ",
  "Клиентский менаджери": "не ИТ",
  "Специалист группы сопровождения фидов": "не ИТ",
  "Корпоративный тренер,": "не ИТ",
  "Ст. Специалист отдела розничных продаж": "не ИТ",
  "Менеджер по сопровождению ипотечных кредитов": "не ИТ",
  "Водитель -инкасаьор": "не ИТ",
  "Сотрудник Дирекции Сервисных Центров": "не ИТ",
  "Менеджер (бизнес-аналитик,CJE)": "не ИТ",
  "Бухгалтер-экономист": "не ИТ",
  "Главный специалист по экономической безопасности": "не ИТ",
  "Менеджер по продажам B2C": "не ИТ",
  "Инженер строительного контроля": "не ИТ",
  "QA Manual/Auto": "Тестирование",
  "Водитель механик": "не ИТ",
  "Финансовый контролер, Финансового департамент": "не ИТ",
  "Руководитель направления CJE AI": "не ИТ",
  "Старший клиентский менджер": "не ИТ",
  "Руководитель направления аналитики": "Аналитика данных/ Data Science",
  "СОЧЛ": "не ИТ",
  "Junior data scientist": "Аналитика данных/ Data Science",
  "Начальник отдела экспертизы кредитных рисков корпоративных клиентов": "не ИТ",
  "Старший клиентский менеджер, старший менеджер по обслуживанию физических и юридических лиц": "не ИТ",
  "Управляющий директор - директор центра": "не ИТ",
  "Начальник малого бизнеса": "не ИТ",
  "Стажер Data Scientist": "Аналитика данных/ Data Science",
  "Ведущий специалист отдела  судебного и исполнительного производства": "не ИТ",
  "Менеджер по работе с VIP клиентами": "не ИТ",
  "Аналитик, python dev., virtual-assistant developer": "Разработка",
  "Менеджер Сбербанк Премьер": "не ИТ",
  "Старший специалист ОТМ": "не ИТ",
  "Старший менеджер операционист": "не ИТ",
  "Ит-инженер": "Администрирование",
  "Директор центра развития и управления эффективностью": "не ИТ",
  "Старший специалист мониторинга залогового имущества": "не ИТ",
  "Руководитель направления (благотворительная деятельность)": "не ИТ",
  "Старший водитель инкосатор": "не ИТ",
  "Клиентский менеджер (Отдел мультиканальных продаж)": "не ИТ",
  "Старший специалист ЕРКЦ": "не ИТ",
  "Старший кассир, зав.кассой, ст.ревизор": "не ИТ",
  "Senior Android-разработчик": "Разработка",
  "Менеджер по сопровождению ипотеки": "не ИТ",
  "Ведущий специалист выездного взыскания": "не ИТ",
  "Сервис менеджер": "не ИТ",
  "Product Owner Голосовая биометрия в аутентификации": "не ИТ",
  "Начальник службы безопасности": "не ИТ",
  "Начальник управления прямых продаж": "не ИТ",
  "Старший инспектор отдела учёта и отчётности": "не ИТ",
  "Главный специалист по работе с мошенничеством": "не ИТ",
  "Руководитель группы планирования": "не ИТ",
  "Персональный Менеджер Сбербанк Премьер": "не ИТ",
  "Старший инженер по сопровождению ЕФС .Переводы. Сбербанк онлайн": "Администрирование",
  "Менеджер управления клиентского опыта В2С": "не ИТ",
  "Менеджер по привлечению зарплатных проектов": "не ИТ",
  "Junior DevOps-инженер": "Администрирование",
  "Начальник сектора контроля качества": "не ИТ",
  "Менеджер по продаже зарплатных проектов": "не ИТ",
  "Менеджер по продажам финансовых продуктов": "не ИТ",
  "Электрик отдела эксплуатации инженерных систем": "не ИТ",
  "ВМО": "не ИТ",
  "Robotic Software Engineer": "Разработка",
  "Начальник отдела обучения": "не ИТ",
  "Ведущий ИТ-инженер": "Администрирование",
  "Специалист отдела развития качества": "не ИТ",
  "Менеджер удаленных продаж": "не ИТ",
  "Старший контроллер Управления мониторинга подозрительных операций ПЦП Центра Комплаенс": "не ИТ",
  "Middle-frontend": "Разработка",
  "Заместитель руководителя офиса.": "не ИТ",
  "Старший специалист экономической безопасности": "не ИТ",
  "Главный специалист Службы финансового директора": "не ИТ",
  "Ведущий эксперт по исследованию данных": "Аналитика данных/ Data Science",
  "Ведущий специалист дирекции по автокредитованию": "не ИТ",
  "Специалист дистанционного взыскания задолженности": "не ИТ",
  "Сборщик": "не ИТ",
  "Менеджер по маркетингу и PR": "не ИТ",
  "Менеджер по лизингу по работе с банком": "не ИТ",
  "Зрвсп": "не ИТ",
  "Начальник смены отдела охраны": "не ИТ",
  "Руководитель центра персонального обслжуивания": "не ИТ",
  "Ведущий специалист отдела безопасности": "не ИТ",
  "Клиенский менеджер СБЕРБАНК ПРЕМЬЕР": "не ИТ",
  "Старший инженер отдела транспортного обеспечения": "не ИТ",
  "Руководитель направления внешней дирстрибуции": "не ИТ",
  "Менеджер управления развития бренда Департамента маркетинга и коммуникаций": "не ИТ",
  "Мобильный менеджер по продажам": "не ИТ",
  "Старший менеджер по обработке кредитных документов": "не ИТ",
  "Специалист call центра": "не ИТ",
  "Ипотечный менеджер": "не ИТ",
  "Специалист по работе с письменными обращениями клиентов": "не ИТ",
  "Главный специалист управления кассовой работы": "не ИТ",
  "Старший технический специалист": "Администрирование",
  "Начальник центра ипотечного кредитования": "не ИТ",
  "Клиентский менеджер СберПремьер": "не ИТ",
  "Ведущий специалист разметки данных": "Аналитика данных/ Data Science",
  "Главный специалист по пожарной безопасности отдела охраны труда": "не ИТ",
  "Заведующий": "не ИТ",
  "Механик": "не ИТ",
  "Начальника административного отдела": "не ИТ",
  "Главный специалист Центра подготовки данных машинного обучения": "Аналитика данных/ Data Science",
  "Контролёр кассир": "не ИТ",
  "Специалист по исполнительному производству": "не ИТ",
  "Старший специалист центра заботы о клиентах": "не ИТ",
  "Специалист по залогам": "не ИТ",
  "Менеджер аналитик по работе с юридическими лицами": "не ИТ",
  "Региональный менеджер по маркетингу": "не ИТ",
  "Руководитель отделения банка": "не ИТ",
  "Инженер по сопровождению": "Администрирование",
  "Ведущий специалист поддержки внутренних клиентов": "не ИТ",
  "Начальник отдела розничных продаж и клиентского обслуживания": "не ИТ",
  "Начальник сектора операционной поддержки": "не ИТ",
  "Инспектор по кредитованию физических лиц": "не ИТ",
  "Специалист по сбору просроченной задолженности": "не ИТ",
  "Ведущий экономист": "не ИТ",
  "Клиентский менеджер по работе с ключевыми клиентами": "не ИТ",
  "Старший менеджер по сервису": "не ИТ",
  "Старший контролер-кассир": "не ИТ",
  "Менеджер партнёрского канала": "не ИТ",
  "Личный помощник руководителя - Вице-Президента": "не ИТ",
  "Ведущий специалист сектора обслуживания корпоративных клиентов": "не ИТ",
  "Менеджер по поддержке продаж продуктов благосостояния": "не ИТ",
  "Начальник учебного центра": "не ИТ",
  "Бухгалтер по кредитованию физ лиц": "не ИТ",
  "Старший специалист банковское сопровождение ( ПБСК )": "не ИТ",
  "Менеджер по продажам продуктов эквайринга": "не ИТ",
  "Руководитель проектов Отдела банковских продуктов и процессов": "не ИТ",
  "Операционный менеджер": "не ИТ",
  "Специалист по сопровождению зарплатных проектов": "не ИТ",
  "Начальник отдела - региональный руководитель Внешней дистрибуции": "не ИТ",
  "Старший менеджер отдела клиентской службы": "не ИТ",
  "Ведущий менеджер по исследованиям": "не ИТ",
  "Старший сервис-менеджер": "не ИТ",
  "Кассир-операционист,": "не ИТ",
  "Менеджер выездного сервиса ,прямых продаж банковских продуктов": "не ИТ",
  "Начальник отдела подбора, оценки, адаптации": "не ИТ",
  "Специалист сервисного обслуживания": "не ИТ",
  "Специалист HR": "не ИТ",
  "3D дизайнер": "Дизайн",
  "Оператор, оператор чата, наставник": "не ИТ",
  "Менеджер (Middle data analyst/data engineer)": "Аналитика данных/ Data Science",
  "Логист": "не ИТ",
  "Старший специалист технической поддержки ДБО": "Администрирование",
  "Менеджер, кластер Методология и GR": "не ИТ",
  "Финансовый эксперт": "не ИТ",
  "Директор Управления по работе с партнерами": "не ИТ",
  "Python developer": "Разработка",
  "Ведущий специалист отдела досудебного производства взыскания задолженности": "не ИТ",
  "Кассир": "не ИТ",
  "Ведущий инженер по сопровождению ПО": "Администрирование",
  "Ведущий программист-аналитик, специалист в области проектного управления, Аналитик ПО": "Разработка",
  "Операционист": "не ИТ",
  "Менеджер по продажам , кредитный специалист": "не ИТ",
  "Руководитель направления «Розничное взыскание и урегулирование» (Куратор филиальной сети)": "не ИТ",
  "Менеджер выездных продаж": "не ИТ",
  "Операционный менеджер по работе с ВИП-клиентами": "не ИТ",
  "Старший специалист разметки данных": "Аналитика данных/ Data Science",
  "Руководитель универсального офиса.": "не ИТ",
  "Технический руководитель": "Разработка",
  "Руководитель группы поддержки сервисов Application Security": "Администрирование",
  "Старший инженер отдела кибербезопасности": "Администрирование",
  "Руководитель направления развития бизнеса": "не ИТ",
  "Ведущий инспектор управления внутрибанковской безопасности": "не ИТ",
  "Начальник отдела сервисной поддержки": "не ИТ",
  "Руководитель группы тестирования": "Тестирование",
  "Директор по маркетингу": "не ИТ",
  "Специалист управления по работе с клиентами": "не ИТ",
  "Региональный руководитель группы ВСП по управлению продаж и обслуживания в сети": "не ИТ",
  "Фрод-мониторинг": "не ИТ",
  "Руководитель офисов": "не ИТ",
  "Операционный менеджер ВИП офиса": "не ИТ",
  "Младший iOS-разработчик": "Разработка",
  "Frontend lead": "Разработка",
  "Старший оператор колл-центра": "не ИТ",
  "Системный аналитик (стажер)": "Разработка",
  "Ведущий специалист управления судебного взыскания": "не ИТ",
  "Начальник планово-экономического отдела": "не ИТ",
  "Ведущий специалист по залогам": "не ИТ",
  "Ведущий специалист досудебного взыскания": "не ИТ",
  "Старший специалист отдела": "не ИТ",
  "Специалист сектора комплектования документов": "не ИТ",
  "Секретарь/помощник старшего вице-президента": "не ИТ",
  "Директор управления по развитию талантов в Центрально-Черноземном банке": "не ИТ",
  "Специалист по ипотеке": "не ИТ",
  "Специалист  по обслуживаю вызовов": "не ИТ",
  "Менеджер малого и микро бизнеса": "не ИТ",
  "Дежурный электрик": "не ИТ",
  "Персональный Менеджер Инвестиционных продуктов": "не ИТ",
  "Помощник по офисной поддержке": "не ИТ",
  "Старший бухгалтер отдела бухгалтерского учета и отчетности": "не ИТ",
  "Управление инвестиционного кредитования и проектного финансирования": "не ИТ",
  "Начальник центра": "не ИТ",
  "Специалист сектора по работе с проблемными активами ММБ": "не ИТ",
  "Старший кассир кассового-инкассаторского центра": "не ИТ",
  "Старший инженер по информационной безопасности": "Администрирование",
  "Старший специалист архива": "не ИТ",
  "Старший специалист контактного центра": "не ИТ",
  "Региональный менеджер по работе с ключевыми клиентами": "не ИТ",
  "Клиентский менеджер малого бизнеса": "не ИТ",
  "Менеджер по работе с задолженностью": "не ИТ",
  "Ведущий специалист отдела корпоративных клиентов": "не ИТ",
  "Главный специалист Управления формирования отчетности": "не ИТ",
  "Главный инженер техподдержки": "Администрирование",
  "Инженер-разработчик": "Разработка",
  "Специалист по аудиту": "не ИТ",
  "Ведущий разработчик React": "Разработка",
  "Специалист ООВ": "не ИТ",
  "Инженер сопровождения": "Администрирование",
  "Ведущий эксперт строительной экспертизы": "не ИТ",
  "Администратор зала": "не ИТ",
  "Заместитель начальника управления": "не ИТ",
  "Инженер по надзору за строительством": "не ИТ",
  "Менеджер отдела продаж корпоративным клиентам": "не ИТ",
  "Руководитель офиса премиального обслуживания «СберПервый»  для клиентов Top Premium город Новый Уренгой": "не ИТ",
  "Менеджер по работе с VIP клиентами.": "не ИТ",
  "Главный руководитель отдела клиентского обслуживания": "не ИТ",
  "Дежурный инкассатор": "не ИТ",
  "Бизнес партнер по управлению персоналом": "не ИТ",
  "Заместитель руководителя по общим вопросам": "не ИТ",
  "Старший ИТ-инженер": "Администрирование",
  "Контент-маркетолог": "не ИТ",
  "Менеджер отдела банкротных проектов": "не ИТ",
  "Упаковщик": "не ИТ",
  "Инкоссатор водитель": "не ИТ",
  "Ассистент руководителя": "не ИТ",
  "Старший специалист по сопровождению процедуры списания проблемных задолженностей": "не ИТ",
  "Кассир КИЦ": "не ИТ",
  "Кредитный инспектор КСБ": "не ИТ",
  "Java/Kotlin разработчик": "Разработка",
  "Менеджер внутренних коммуникаций": "не ИТ",
  "Ведущий специалист сметчик": "не ИТ",
  "Кредитный инспектор Управления по работе с предприятиями сферы недвижимости": "не ИТ",
  "Ведущий инженер-технолог": "не ИТ",
  "Региональный руководитель прямых продаж": "не ИТ",
  "Главный инспектор сектора кредитования физических лиц": "не ИТ",
  "Старший инженер по строительству и эксплуатации": "не ИТ",
  "Инкассатор АТМ": "не ИТ",
  "Ведущий специалист по цифровым технологиям аудита": "не ИТ",
  "Специалист отдела сопровождения оформления документов Управления кадрового администрирования Центра HR администрирования": "не ИТ",
  "Ведущий специалист отдела обслуживания": "не ИТ",
  "Руководитель HR-tech проектов": "не ИТ",
  "Старший менеджер по работе с ключевыми клиентами малого бизнеса": "не ИТ",
  "Аналитик сектора обучения": "не ИТ",
  "Главный специалист приоритетной поддержки vip клиентов": "не ИТ",
  "Менеджер по ключевым партнерам платежных сервисов. Направление \"Безналичные решения\" Юго- Западного банка": "не ИТ",
  "Старший специалист Архивного центра": "не ИТ",
  "Старший опереционно-кассовый работник": "не ИТ",
  "Операционно-кассовый специалист, заместитель руководителя офиса": "не ИТ",
  "Главный специалист по взысканию": "не ИТ",
  "Корпоративный бизнес-тренер": "не ИТ",
  "Специалист по делопроизводству": "не ИТ",
  "Senior Frontend React developer": "Разработка",
  "Менеджер по сопровождению": "не ИТ",
  "Старший специалист по обслуживанию": "не ИТ",
  "Младший backend-разработчик": "Разработка",
  "Помощник управляющего": "не ИТ",
  "Middle системный аналитик": "Разработка",
  "Руководитель группы офисов Управления продаж и обслуживания в сети ВСП": "не ИТ",
  "Главный инженер по тестированию": "Тестирование",
  "Старший операционный менеджер": "не ИТ",
  "Главный специалист отдела взыскания долгов": "не ИТ",
  "Инженер отдела автоматизации": "Разработка",
  "Инспектор экономической беезопасности": "не ИТ",
  "Ведущий юрисконсульт сектора правового обеспечения микро и малого бизнеса": "не ИТ",
  "Руководитель направления клиентского сервиса": "не ИТ",
  "Руководитель группы всп": "не ИТ",
  "Менеджер группы поддержки клиентов": "не ИТ",
  "Ведущий специалист по обработке претензий": "не ИТ",
  "Главный специалист Управления регуляторного контроля": "не ИТ",
  "Старший специалист по взысканию": "не ИТ",
  "Ведущий эксперт центра компетенций по анализу киберугроз": "Администрирование",
  "Старший клиентский менеджер по продажам": "не ИТ",
  "Водитель инкосатор": "не ИТ",
  "Старший инкассатор": "не ИТ",
  "Менеджер по продажам корпоративным клиентам": "не ИТ",
  "Андеррайтер клиентов малого и микробизнеса": "не ИТ",
  "Менеджер по обслуживанию физических лиц": "не ИТ",
  "Старший менджер по обслуживанию физических лиц": "не ИТ",
  "Главный специалист отдела по работе с проблемной задолженностью": "не ИТ",
  "Менеджер про продаже зарплатных проектов": "не ИТ",
  "Клиентский менеджер СберПервый": "не ИТ",
  "Кредитный инспектор (": "не ИТ",
  "Старший специалист Управления поддержки внутренних клиентов": "не ИТ",
  "Менеджер по банковским продуктам": "не ИТ",
  "Ведуший специалист одвз": "не ИТ",
  "Начальник управления": "не ИТ",
  "Эксперт Департамента безопасности": "не ИТ",
  "Разработчик SQL": "Разработка",
  "Ведущий эксперт по технологии": "не ИТ",
  "Data инженер": "Аналитика данных/ Data Science",
  "Главный специалист по работе с обращениями клиентов": "не ИТ",
  "Аналитик отдела анализа рисков розничного и микробизнеса": "Аналитика данных/ Data Science",
  "Старший специалист отдела противодействия мошенничеству": "не ИТ",
  "Ведущий менеджер операционист": "не ИТ",
  "Ведущий специалист операционной поддержки корпоративных клиентов": "не ИТ",
  "Главный клиентский менеджер по работе с ключевыми клиентами": "не ИТ",
  "Начальник отдела подбора персонала": "не ИТ",
  "Старший менеджер по обучению": "не ИТ",
  "Директор дополнительного офиса": "не ИТ",
  "Начальник отдела/руководитель направления судебного и исполнительного производства": "не ИТ",
  "Главный специалист Дивизиона Риски розничного бизнеса": "не ИТ",
  "Ведущий специалист регионального представительства по автокредитованию": "не ИТ",
  "Менеджер по ипотеке": "не ИТ",
  "Специалист сопровождения юридических лиц": "не ИТ",
  "Менеджер по операционному сопровождению транзакционного бизнеса": "не ИТ",
  "Строительный эксперт": "не ИТ",
  "Руководитель отделаов по обслуживанию физических и юридических лиц": "не ИТ",
  "Руководитель офисом": "не ИТ",
  "Старший менеджер по обслуживанию.": "не ИТ",
  "Инкассатор водитель": "не ИТ",
  "Мобильный менеджер": "не ИТ",
  "Дистанционный клиентский менеджер премиальный формат": "не ИТ",
  "Ведущий специалист Управления безопасности": "не ИТ",
  "Владелец продукта / Руководитель направления / Product Owner": "не ИТ",
  "Старший экономист": "не ИТ",
  "Исполнительный директор - начальник Центра управления сетью": "не ИТ",
  "Старший менеджер отдела продаж": "не ИТ",
  "Старший финансовый советник": "не ИТ",
  "Java Developer": "Разработка",
  "Разработчик алгоритма": "Разработка",
  "Менеджер по работе с недвижимостью": "не ИТ",
  "Исследователь": "не ИТ",
  "Старший менеджер обслуживания": "не ИТ",
  "Ведущий специалист ОЭБ управления безопасности": "не ИТ",
  "Ведущий руководитель IT-проектов": "Администрирование",
  "Руководитель группы сопровождения АС": "Администрирование",
  "Товаровед": "не ИТ",
  "Менеджер по продажам продуктов канала": "не ИТ",
  "Ведущий frontend-разработчик": "Разработка",
  "Старший специалист технической поддержки": "Администрирование",
  "Главный специалист судебного и исполнительного производства": "не ИТ",
  "Старший специалист отдела обслуживания клиентов": "не ИТ",
  "Руководитель направления разработки": "Разработка",
  "Ведущий специалист по обработке документов": "не ИТ",
  "Ведущий инженер отдела информационных технологий": "Администрирование",
  "Ведущий специалист по обслуживанию": "не ИТ",
  "Ведущий специалист отдела обучения": "не ИТ",
  "Senior Product Owner": "не ИТ",
  "Старший оператор call-центра": "не ИТ",
  "Консультант зала": "не ИТ",
  "Менеджер по продажам продуктов эквайринга (безналичные решения)": "не ИТ",
  "Ведущий специалист управления прямых продаж": "не ИТ",
  "Персональный клиентский менеджер премьер": "не ИТ",
  "Главный специалист отдела технических средств охраны": "не ИТ",
  "Ведущий специалист отдела по работе с клиентами": "не ИТ",
  "Специалист кассир": "не ИТ",
  "Заместитель начальника отдела внутреннего аудита": "не ИТ",
  "Персональный менеджер премьер офиса": "не ИТ",
  "Менеджер по продажам эквайринга": "не ИТ",
  "Ресерчер": "не ИТ",
  "старший специалист документооборота": "не ИТ",
  "Начальник отдела валютных операций": "не ИТ",
  "Специалист отделения банка": "не ИТ",
  "Руководитель направления онлайн-маркетинга": "не ИТ",
  "Senior инженер по сопровождению": "Администрирование",
  "Старший бухгалтер по расчетам": "не ИТ",
  "Руководитель направления (senior CJE/BA)": "не ИТ",
  "Старший специалист по обслуживанию физических лиц": "не ИТ",
  "Delivery Lead": "Администрирование",
  "Главный специалист технической поддержки": "Администрирование",
  "Директор": "не ИТ",
  "Ведущий специалист по работе с обращениями": "не ИТ",
  "Ведущий специалист сектора развития бизнеса": "не ИТ",
  "Старший специалист контакт-центра": "не ИТ",
  "Главный инженер (Middle Java Developer)": "Разработка",
  "Создатель контента (internal comms)": "не ИТ",
  "Специалист центра управления работоспособностью устройств самообслуживания": "не ИТ",
  "Старший эксперт по кибербезопасности": "Администрирование",
  "Слесарь КИПиА": "не ИТ",
  "Руководитель подразделения": "не ИТ",
  "Ведущий ИТ-аналитик": "Аналитика данных/ Data Science",
  "Клиентский менеджер по работе с ключевыми клиентами малого и микро бизнеса": "не ИТ",
  "Главный специалист контакт-центра": "не ИТ",
  "Старший офис менеджер": "не ИТ",
  "Руководитель кассово-инкассаторского центра": "не ИТ",
  "Заместитель руководителя офиса банка": "не ИТ",
  "Аналитик антифрода": "Аналитика данных/ Data Science",
  "Релиз Менеджер": "Администрирование",
  "Java middle developer": "Разработка",
  "Клиентский менеджер Управление прямых продаж Курганского отделения №8599 Уральского банка": "не ИТ",
  "Ведущий эксперт отдела по обработке кредитных заявок физических лиц": "не ИТ",
  "Ведущий инженер по сметной работе": "не ИТ",
  "Старший специалист технической поддержки корпоративных клиентов": "Администрирование",
  "Старший специалист планирования": "не ИТ",
  "Заместитель руководителя бизнес офиса": "не ИТ",
  "Персональный менеджер премиального уровня": "не ИТ",
  "Исследователь данных": "Аналитика данных/ Data Science",
  "Менеджер в малом бизнесе": "не ИТ",
  "Старший специалист по страхованию": "не ИТ",
  "Старший специалист отдела обеспечения": "не ИТ",
  "Старший специалист по сделкам с недвижимостью.": "не ИТ",
  "Начальник сектора инкассации": "не ИТ",
  "Senior System Analyst": "Разработка",
  "Ведущий тестировщик ПО (Penetration test)": "Тестирование",
  "Менеджер по работе с клиентами, менеджер по ипотечному кредитованию, заместитель руководителя отдела ипотечного кредитования": "не ИТ",
  "Начальник транспортного отдела": "не ИТ",
  "Менеджер по продажам и сопровождению продуктов платежных сервисов": "не ИТ",
  "Начальник отдела корпоративных продаж": "не ИТ",
  "Senior Frontend-разработчик": "Разработка",
  "Ведущий специалист отложенных операций": "не ИТ",
  "Старший специалист отдела по работе с обращениями": "не ИТ",
  "Специалист отдела по работе с проблемными активами юридических лиц": "не ИТ",
  "Выездной менеджер по продажам": "не ИТ",
  "Мерчендайзер": "не ИТ",
  "Руководитель направления (Product Owner)": "не ИТ",
  "Старший менеджер продукта": "не ИТ",
  "Водитель автомобиля предназначенного для перевозки ценостей": "не ИТ",
  "Менеджер по внутренним коммуникациям Единой службы заботы о клиентах": "не ИТ",
  "Клиентский менеджер крупного и среднего бизнеса": "не ИТ",
  "Старший специалист отдела развития инфраструктуры": "не ИТ",
  "Специалист отдела поддержки продаж": "не ИТ",
  "Заведующая филиалом": "не ИТ",
  "Управляющий директор кластера Кампании продаж КИБа": "не ИТ",
  "Ведущий специалист службы Финансового директора": "не ИТ",
  "Юрисконсульт отдела правового обеспечения розничного бизнеса": "не ИТ",
  "Старший офис-менеджер": "не ИТ",
  "Специалист нагрузочного тестирования": "Тестирование",
  "Выездной инженер": "Администрирование",
  "Руководитель Дополнительного офиса Кассово инкассаторского центра": "не ИТ",
  "Специалист сервисной поддержки продуктов эквайринга": "не ИТ",
  "Специалист сектора Экономической безопасности": "не ИТ",
  "Разметчик текстовых данных": "Аналитика данных/ Data Science",
  "Руководитель направления QA": "Тестирование",
  "Водитель автомобиля, предназначенного для инкассирования": "не ИТ",
  "Технический эксперт": "Администрирование",
  "Специалист-эксперт отдела организационно-кадровой экспертизы": "не ИТ",
  "Frontend developer": "Разработка",
  "Специалист по логистике": "не ИТ",
  "Специалист центра оперативной поддержки продаж": "не ИТ",
  "Инженер по сопровождению АС": "Администрирование",
  "Менеджер по обучению и развитию персонала": "не ИТ",
  "Старший специалист по работе с обращениями клиентов": "не ИТ",
  "Главный специалист-полиграфолог": "не ИТ",
  "Старший менеджер по нефинансовым сервисам": "не ИТ",
  "Старший инспектор ИБ": "Администрирование",
  "Менеджер службы клиентского сервиса": "не ИТ",
  "Специалист по работе с проблемной задолженностью": "не ИТ",
  "Middle Data Engineer": "Аналитика данных/ Data Science",
  "Руководитель команды разработки": "Разработка",
  "Руководитель офисов ипотечного кредитования": "не ИТ",
  "Старший меннджер по работе с клиентами": "не ИТ",
  "Кредитный инспектор Управления финансирования строительных проектов, ЦА": "не ИТ",
  "Ведущий специалист сектора подготовки кредитной документации": "не ИТ",
  "Старший финансовый контролер": "не ИТ",
  "Эксперт карьерного развития": "не ИТ",
  "Менеджер управления планирования и отчетности": "не ИТ",
  "Старший клиентский менеджер внешней дистрибуции": "не ИТ",
  "Старший инженер по разработке (SRE)": "Администрирование",
  "Ведущий qA-engineer": "Тестирование",
  "Главный специалист выездного взыскания": "не ИТ",
  "HR-бизнес-партнер": "не ИТ",
  "Старший юрисконсульт": "не ИТ",
  "Старший специалист отдела обслуживания": "не ИТ",
  "Кладовщик группы отгрузки": "не ИТ",
  "Старший специалист инженерно технического отдела": "не ИТ",
  "Старший менеджер по сделкам с недвижимостью, старший менеджер по ипотечному кредитованию": "не ИТ",
  "Стажер департамента финансов": "не ИТ",
  "Ведущий специалист управления списания задолженности": "не ИТ",
  "Руководитель группы ВСП (внутренние структурные подразделения)Временно, на время декретного отпуска.": "не ИТ",
  "Оператор постановщик / Режиссер монтажа": "не ИТ",
  "Старший специалист машинного обучения": "Аналитика данных/ Data Science",
  "Ведущий специалист по маркетингу в канале on-trade": "не ИТ",
  "UX/UI дизайнер": "Дизайн",
  "Администратор зала ВИП ВСП": "не ИТ",
  "Специалист сектора технического мониторинга": "не ИТ",
  "Старший специалист Группы недвижимости и развития инфраструктуры ЦКП РСЦ": "не ИТ",
  "Разработчик баз данных": "Разработка",
  "Оператор-кассир": "не ИТ",
  "Заместитель руководителя отделения банка": "не ИТ",
  "Руководитель направления по развитию бизнеса": "не ИТ",
  "HR Бизнес-партнёр": "не ИТ",
  "Региональный руководитель группы офисов розничное направление бизнеса": "не ИТ",
  "Data-аналитик": "Аналитика данных/ Data Science",
  "Начальник отдела ипотечного кредитования": "не ИТ",
  "Юрист (LegalTech)": "не ИТ",
  "Старший специалист подразделения приоритетной поддержки": "не ИТ",
  "Младший эксперт по исследованию данных": "Аналитика данных/ Data Science",
  "UX-исследователь": "Дизайн",
  "Старший специалист по работе с недвижимостью": "не ИТ",
  "Начальник сектора прямых продаж": "не ИТ",
  "Менеджер по продажам; Старший менеджер по сделкам с недвижимостью": "не ИТ",
  "Специалист по сопровождению ипотеки": "не ИТ",
  "Менеджер по обслуживанию частных лиц": "не ИТ",
  "Агент по недвижимости": "не ИТ",
  "Менеджер по развитию каналов действующего бизнеса": "не ИТ",
  "Директор операционного офиса": "не ИТ",
  "Линейный руководитель": "не ИТ",
  "Старший специалист по ипотечному кредитованию": "не ИТ",
  "Менеджер по продажам продуктов ВЭД": "не ИТ",
  "Product owner SberCRM": "не ИТ",
  "Главный специалист сервисного отдела": "не ИТ",
  "Руководитель по внутренним и внешним коммуникациям": "не ИТ",
  "Старший менеджер по обслуживания": "не ИТ",
  "Backend c# developer": "Разработка",
  "Разработчик python": "Разработка",
  "Ведущий специалист по разметке данных": "Аналитика данных/ Data Science",
  "Руководитель отдела доставки и сортировки": "не ИТ",
  "Стажер-инженер QA auto Java": "Тестирование",
  "Ведущий юрист": "не ИТ",
  "Уборщик офисных помещений, служебных помещений, уборка квартир": "не ИТ",
  "UX/UI Designer": "Дизайн",
  "Аналитик по расходам": "не ИТ",
  "Начальник отдела фрод мониторинга": "не ИТ",
  "Главный инженер по защите информации": "Администрирование",
  "Начальник сектора проверочно-розыскных мероприятий отдела экономической безопасности управления безопасности": "не ИТ",
  "Контролёр-кассир": "не ИТ",
  "Старший менеджер по работе с партнерами": "не ИТ",
  "Персональный тренер": "не ИТ",
  "Старший клиентский менеджер Домклик": "не ИТ",
  "Руководитель группы специалистов по прямым продажам": "не ИТ",
  "Главный клиентский менеджер корпоративного блока Средний бизнес": "не ИТ",
  "Главный специалист отдела досудебного взыскания задолженностей": "не ИТ",
  "Разработчик фронтэнда": "Разработка",
  "Главный специалист, руководитель направления": "не ИТ",
  "Старший инкассатор-инженер": "не ИТ",
  "Старший инспектор управления экономической безопасности": "не ИТ",
  "Клиентский менеджер по работе с клиентами крупного и среднего бизнеса": "не ИТ",
  "Старший специалист отдела развития": "не ИТ",
  "Менеджер по коммуникациям и событийным проектам": "не ИТ",
  "Ведущий методист": "не ИТ",
  "Зам руководителя": "не ИТ",
  "Координатор": "не ИТ",
  "Ведущий менеджер по работе с партнерами Экосистемы": "не ИТ",
  "Инженер КИБ": "Администрирование",
  "Ведущий специалист. Юрист.": "не ИТ",
  "Специалист по обучению": "не ИТ",
  "Менеджер по внедрению": "не ИТ",
  "Руководитель сектора клиентского обслуживания": "не ИТ",
  "Старший специалист колл-центре": "не ИТ",
  "Старшей менеджер по обслуживанию": "не ИТ",
  "Докуменооборот, специалист": "не ИТ",
  "Менеджер по работе с клиентами Премьер": "не ИТ",
  "Клиентский менеджер малого и микро бизнеса": "не ИТ",
  "Старший инженер сопровождения продукта": "Администрирование",
  "Ведущий специалист по работе с проблемными активами": "не ИТ",
  "Ипотечный специалист": "не ИТ",
  "Региональный менеджер центра продаж корпоративным клиентам": "не ИТ",
  "Главный Бизнес-аналитик": "не ИТ",
  "Руководитель IT направления": "Администрирование",
  "Старший менеджер по ипотечному кредитованию": "не ИТ",
  "Старший специалист по рассматриванию обращений физических лиц": "не ИТ",
  "Специалист управления сопровождения исполнительного производства": "не ИТ",
  "Middle Data Analyst": "Аналитика данных/ Data Science",
  "Руководитель партнерского канала СЗФО": "не ИТ",
  "Начальник управления продаж": "не ИТ",
  "Главный инженер по эксплуатации": "не ИТ",
  "Главный инженер по разработке (QA)": "Тестирование",
  "Категорийный менеджер": "не ИТ",
  "Менеджер по сопровождению ипотечного кредитования": "не ИТ",
  "Ведущий специалист отдела видеоаналитики и инновационных исследований.": "Аналитика данных/ Data Science",
  "Начальник участка": "не ИТ",
  "Старший клиентский менеджер по обслуживанию юридических лиц": "не ИТ",
  "Старший специалист по работе с задолженностью": "не ИТ",
  "Студент-стажер": "не ИТ",
  "Старший менеджер по обслуживанию Специализированный по обслуживанию физических лиц дополнительный офис № 7003/0886 Уральского банка": "не ИТ",
  "Клиентский менеджер внешней дистрибуции": "не ИТ",
  "SRE , руководитель направления наблюдаемости": "Администрирование",
  "Руководитель офиса/отдела": "не ИТ",
  "Старший менеджер по обслуживанию физических лиц, до этого работала консультантом": "не ИТ",
  "Начальник отдела - региональный руководитель": "не ИТ",
  "Старший менеджер по недвижимости": "не ИТ",
  "Главный специалист сектора по сопровождению процедур банкротства": "не ИТ",
  "Ведущий инспектор административного отдела": "не ИТ",
  "Главный IT-инженер/ Team Lead": "Администрирование",
  "Head of the Data Analytics Group": "Аналитика данных/ Data Science",
  "Главный специалист отдела финансового контроля": "не ИТ",
  "Бизнес-ассистент": "не ИТ",
  "Key Account Manager": "не ИТ",
  "UI/UX designer": "Дизайн",
  "Контроллер-кассир": "не ИТ",
  "Специалист отдела обработки документов банка": "не ИТ",
  "Контролер кассы пересчета": "не ИТ",
  "Старший менеджер по обслуживанию частных лиц": "не ИТ",
  "Руководитель направления Управление планирования производства и эффективности": "не ИТ",
  "Руководитель проектов / Департамент маркетинга и коммуникаций": "не ИТ",
  "Бизнес-аналитик/CJE Expert": "не ИТ",
  "Старший менеджер по жилищному кредитованию": "не ИТ",
  "Старший специалист направления развития": "не ИТ",
  "Старший клиентский менеджер. Дополнительный офис, специализированный по обслуживанию физических лиц № 8615/0386": "не ИТ",
  "Главный инженер-программист": "Разработка",
  "Управляющий филиалом": "не ИТ",
  "Руководитель сектора кадрового администрирования": "не ИТ",
  "Начальник отдела организации кредитования Управления продаж малому бизнесу": "не ИТ",
  "Старший специалист кассового центра": "не ИТ",
  "Администратор офиса": "не ИТ",
  "Специалист технической поддержки юридических лиц": "Администрирование",
  "Руководитель отдела по сделкам с недвижимостью": "не ИТ",
  "Сборщик заказов": "не ИТ",
  "Начальник отдела телемаркетинга": "не ИТ",
  "Инженер по банковским картам": "не ИТ",
  "Ведущий специалист отдела договоров и расчетов": "не ИТ",
  "Ведущий специалист по вопросам комплаенс": "не ИТ",
  "Ведущий специалист отдела учета ТМЦ": "не ИТ",
  "Начальник сектора планирования регионального контактного центра": "не ИТ",
  "Старший менеджер-администратор": "Администрирование",
  "Менеджер сервисного центра": "не ИТ",
  "Начальник сектора инкассации и безопасности": "не ИТ",
  "Агент": "не ИТ",
  "Начальник управления продаж малому бизнесу": "не ИТ",
  "Клиентский менеджер зоны Сбербанк премьер": "не ИТ",
  "ИТ-архитектор": "Разработка",
  "Ит специалист": "Администрирование",
  "Старший специалист по сопровождению внутрибанковской АС": "Администрирование",
  "Инженер Удаленной поддержки АРМ.": "Администрирование",
  "Территориальный менеджер направления по работе с состоятельными клиентами Сбер1": "не ИТ",
  "Ведущий инженер по тестировнию": "Тестирование",
  "Менеджер по сделкам с недвижимостью,ипотека": "не ИТ",
  "Заместитель руководителя ВИП ВСП": "не ИТ",
  "Специалист ОТК": "не ИТ",
  "Инвестиционный тренер": "не ИТ",
  "Ведущий бухгалтер, зам. начальника сектора баланса и учета операций": "не ИТ",
  "Промоконсультант": "не ИТ",
  "Клиентский менеджер Отдела внешней дистрибуции": "не ИТ",
  "Руководитель кластера": "не ИТ",
  "Бухгалтер по договорам аренды": "не ИТ",
  "Кредитный инспектор (юридические лица)": "не ИТ",
  "Лидер АС, тимлид команды из 11 человек": "Разработка",
  "Ведущий специалист отдела экономической безопасности": "не ИТ",
  "Региональный менеджер по развитию": "не ИТ",
  "Стажер СберПервый": "не ИТ",
  "Руководитель направления продаж, ПО": "не ИТ",
  "ИТ-аналитик": "Аналитика данных/ Data Science",
  "Старший Менеджер по работе с физическими лицами": "не ИТ",
  "Аналитик отдела сопровождения бизнес-процессов": "не ИТ",
  "Главный специалист Управления операционных рисков": "не ИТ",
  "Специалист поддержки внутренних клиентов": "не ИТ",
  "Старший специалист по экономической безопасности": "не ИТ",
  "Администратор переговорных комнат": "не ИТ",
  "Кассир в кассу пересчета": "не ИТ",
  "Старший инженер отдела мониторинга СКЗИ": "Администрирование",
  "Региональный директор департамента инвестиционной деятельности": "не ИТ",
  "Старший специалист HR": "не ИТ",
  "Руководитель по развитию бизнеса": "не ИТ",
  "Руководитель направления исследования данных": "Аналитика данных/ Data Science",
  "Региональный менеджер по взысканию": "не ИТ",
  "Менеджер по продаже продуктов эквайринга": "не ИТ",
  "Клиентский менеджер Сбербанк": "не ИТ",
  "Middle Backend Developer": "Разработка",
  "Инженер по разработке (QA Automation)": "Тестирование",
  "Старший клиентский менеджер РГС": "не ИТ",
  "Корпоративный тренер": "не ИТ",
  "Начальник отдела технических средств охраны": "не ИТ",
  "Кредитный инспектор отдела администрирования кредитов и оформления КОД в УК Кемеровского ГОСБ Сибирского банка СБ РФ. г.Кемерово": "не ИТ",
  "Инженер-тестировщик": "Тестирование",
  "Кредитный инспектор сектора администрирования кредитов": "не ИТ",
  "Ассистент клиентского менеджера по работе с юридическими лицами": "не ИТ",
  "JavaScript разработчик": "Разработка",
  "Специалист регионального контактного центра": "не ИТ",
  "Специалист судебного и исполнительного производства": "не ИТ",
  "Главный клиентский менеджер по работе с корпоративными клиентами": "не ИТ",
  "Менеджер по работе с юридическими клиентами": "не ИТ",
  "Начальник административного управления": "не ИТ",
  "Менеджер по обслуживанию специализированный по обслуживанию физ лиц": "не ИТ",
  "Директор центра развития талантов": "не ИТ",
  "Юрисконсульт в сектор защиты интересов Банка в сфере розничного бизнеса": "не ИТ",
  "Инженер POS-терминалов": "Администрирование",
  "PHP-программист": "Разработка",
  "Начальник управления продаж продуктов Благосостояния": "не ИТ",
  "И.О. начальника отдела / аналитик в сегменте госсектор": "не ИТ",
  "Кассир кассы пересчета": "не ИТ",
  "Руководитель офиса ипотечного кредитования": "не ИТ",
  "СМО": "не ИТ",
  "Риск-менеджер": "не ИТ",
  "Директор управления маркетинга и коммуникаций": "не ИТ",
  "Начальник склада": "не ИТ",
  "Старший Менеджер по работе с корпоративными клиентами розничной торговли": "не ИТ",
  "Менеджер по недвижимости": "не ИТ",
  "Digital маркетолог": "не ИТ",
  "Специалист в секторе развития инфраструктуры": "не ИТ",
  "Региональный менеджер по внешним коммуникациям": "не ИТ",
  "Главный специалист отдела технической поддержки систем ДБО": "Администрирование",
  "Кредитный инспектор ОКЧК УКЧК": "не ИТ",
  "Ведущий бухгалтер (ведущий специалист)": "не ИТ",
  "Руководитель офиса банка": "не ИТ",
  "Специалист отдела противодействия мошенничеству": "не ИТ",
  "Директор управления рисков": "не ИТ",
  "Devops инжененр": "Администрирование",
  "Ведущий специалист по страхованию": "не ИТ",
  "Ведущий инспектор контактного центра": "не ИТ",
  "Product Owner / Product Manager": "не ИТ",
  "Руководитель прямых продаж": "не ИТ",
  "Руководитель направления - Product Owner": "не ИТ",
  "Главный кредитный аналитик малого и среднего бизнеса": "не ИТ",
  "Заместитель управляющего отделением (г. Тверь)": "не ИТ",
  "Промоутер-консультант": "не ИТ",
  "Андеррайтер, контролер качества": "не ИТ",
  "Старший специалист по работе с ключевыми клиентами  малого бизнеса": "не ИТ",
  "Agile coach": "не ИТ",
  "SMM Lead, Influencer marketing manager": "не ИТ",
  "Начальник отдела безопасности по ЯНАО": "не ИТ",
  "Старший специалист фрод мониторинга": "не ИТ",
  "Инженер программист, начальник административного отдела, менеджер по качеству": "Администрирование",
  "Главный строительный эксперт": "не ИТ",
  "Руководитель направления по исследованию данных": "Аналитика данных/ Data Science",
  "Старший инкассатор-мастер": "не ИТ",
  "DWH BI Разработчик": "Аналитика данных/ Data Science",
  "Ведущий инженер СХД": "Администрирование",
  "Product owner / Project Manager": "не ИТ",
  "Руководитель направления (DevOps)": "Администрирование",
  "Региональный руководитель филиальной сети": "не ИТ",
  "Специалист по обслуживанию вызовов": "не ИТ",
  "Комплектовщик": "не ИТ",
  "Начальник сектора обслуживания физических лиц": "не ИТ",
  "Data-engineer": "Аналитика данных/ Data Science",
  "Начальник отдела подготовки внутренней отчетности": "не ИТ",
  "Ведущий UX-писатель": "Дизайн",
  "Специалист отдела андеррайтинга": "не ИТ",
  "Специалист Направления хозяйственной эксплуатации": "не ИТ",
  "Кредитный инспектор среднего и крупного бизнеса": "не ИТ",
  "Кадровый специалист": "не ИТ",
  "Главный юрисконсульт направления правового обеспечения микро и малого бизнеса": "не ИТ",
  "Руководитель направления внешней дистрибуции": "не ИТ",
  "Менеджер по организации и развитию новых каналов продаж по сложным продуктам": "не ИТ",
  "IT PL / Chief Product Owner": "не ИТ",
  "Старший клиентский менеджер Премьер": "не ИТ",
  "Руководитель направления киберкультуры": "не ИТ",
  "Ведущий специалист отдела технических средств охраны": "не ИТ",
  "Ведущий исследователь данных": "Аналитика данных/ Data Science",
  "Продуктовый дизайнер/Дизайнер UI/UX": "Дизайн",
  "Региональный руководитель розничной сети": "не ИТ",
  "Старший инженер сопровождения": "Администрирование",
  "Junior Java Developer": "Разработка",
  "Начальник подразделения": "не ИТ",
  "Директор исторического офиса Арбат": "не ИТ",
  "Оператор ЕРКЦ": "не ИТ",
  "Руководитель подразделения кассовой работы": "не ИТ",
  "Клиентский менеджер \"Микро и Малого бизнеса\"": "не ИТ",
  "Премиальный дистанционный клиентский менеджер": "не ИТ",
  "Главный эксперт отдела строительного контроля": "не ИТ",
  "Старший инкассатор мастер": "не ИТ",
  "Вед. инженер по сопровождению АС": "Администрирование",
  "Специалист по продажам онлайн услуг (промоутер)": "не ИТ",
  "Специалист отдела связи и мониторинга Управления Инкассации": "не ИТ",
  "Ассистент менеджера по работе с клиентами": "не ИТ",
  "Менеджер по продажам эквайринг": "не ИТ",
  "Руководитель направления в управлении развития технологий Рисков": "не ИТ",
  "Специалист по сделкам с недвижимостью": "не ИТ",
  "Инженер DevOps": "Администрирование",
  "Руководитель  офиса": "не ИТ",
  "Корпоративный архитектор": "Разработка",
  "DevOps ведущий инженер": "Администрирование",
  "Ведущий специалист по охране труда и пожарной безопасности": "не ИТ",
  "Специалист административно-хозяйственного обеспечения": "не ИТ",
  "Главный специалист отдела кадровой экспертизы управления кадрового администрирования ПУП Центр поддержки «Люди и культура»": "не ИТ",
  "Практикант в банк": "не ИТ",
  "Менеджер направления организации продаж зарплатных проектов": "не ИТ",
  "Разметчик данных для нейросети": "Аналитика данных/ Data Science",
  "Специалист отдела обслуживания вызовов": "не ИТ",
  "Главный специалист контактного центра": "не ИТ",
  "Аналитик по управленческой отчетности": "не ИТ",
  "Инженер ИТ": "Администрирование",
  "Ведущий экономист кредитования юридических лиц": "не ИТ",
  "Старший менеджер по работе с клиентами и юридическими лицами": "не ИТ",
  "Специалист по исследованию данных": "Аналитика данных/ Data Science",
  "Старший специалист группы обслуживания вызовов": "не ИТ",
  "Персональный менеджер премиальных клиентов": "не ИТ",
  "Кассир Кассово-инкассаторского центра": "не ИТ",
  "Руководитель направления в Центре корпоративных риск-решений": "не ИТ",
  "Старший специалист отдела развития и сопровождения инфраструктуры": "не ИТ",
  "Специалист по технической поддержке продуктов эквайринга": "Администрирование",
  "Ведущий специалист управления транзакций и формирования отчетности": "не ИТ",
  "Клиентский менеджер, АКМ": "не ИТ",
  "Региональный менеджер внешней дистрибуции Отдела внешней дистрибуции  Управления прямых продаж": "не ИТ",
  "Python Backend developer": "Разработка",
  "Старший менеджер по продажам и развитию партнёрских программ": "не ИТ",
  "Специалист отдела call  центра": "не ИТ",
  "Специалист управления сопровождения процедуры банкротства": "не ИТ",
  "Старший инспектор, заместитель начальника отдела сопровождения": "не ИТ",
  "Клиентский менеджер малого и среднего бизнеса": "не ИТ",
  "Специалист по разметке данных для нейросетей": "Аналитика данных/ Data Science",
  "Ведущий специалист по пожарной безопасности": "не ИТ",
  "Дата-инженер": "Аналитика данных/ Data Science",
  "Специалист обслуживания частных лиц": "не ИТ",
  "Senior frontend developer": "Разработка",
  "Ведущий специалист отдела кредитного администрирования": "не ИТ",
  "Специалист подразделения сервиса для экосистемы юл": "не ИТ",
  "Специалист регионального центра внутрибанковского учета и отчетности г. Самара подразделения центрального подчинения \"Служба финансового менеджмента\" ПАО Сбербанк": "не ИТ",
  "Оценщик": "не ИТ",
  "Менеджер по работе с ключевыми клиентами малого и микро бизнеса": "не ИТ",
  "Старший специалист претензионной  службы": "не ИТ",
  "Продуктовые исследования, R&D, НИОКР": "не ИТ",
  "Менеджер по рекламе и работе со СМИ": "не ИТ",
  "Ведущий специалист расчетного центра": "не ИТ",
  "Эксперт по графическому дизайну": "Дизайн",
  "Design Team Lead": "Дизайн",
  "Старший инженер отдела удаленной ИТ поддержки АРМ": "Администрирование",
  "IT-специалист": "Администрирование",
  "Ведущий специалист сектора по противодействию мошенничества": "не ИТ",
  "Руководитель персонального центра обслуживания": "не ИТ",
  "Руководитель по цифровому развитию": "не ИТ",
  "Главный андеррайтер Департамента рисков г.Москва": "не ИТ",
  "Главный специалист/Начальник сектора мониторинга залогового обеспечения (УРМ г. Тверь)": "не ИТ",
  "Руководитель направления (customer journey expert)": "не ИТ",
  "Специалист ЦДВЗ(Центр дистанционного взыскания задолжности)": "не ИТ",
  "Начальник отдела сопровождения клиентских операций": "не ИТ",
  "Менеджер по продажам.": "не ИТ",
  "Продуктовый менеджер": "не ИТ",
  "Инженер/SRE/DevOps": "Администрирование",
  "Middle backend разработчик": "Разработка",
  "Менеджер по поддержке продаж сложных продуктов благосостояния": "не ИТ",
  "Инженер ЕДС ДОПС": "не ИТ",
  "Специалист отдела вкладов и расчетов населения": "не ИТ",
  "Инспектор сектора информационной безопасности": "Администрирование",
  "Эксперт по качеству клиентского обслуживания": "не ИТ",
  "Клиентский менеджер по работе с VIP клиентами": "не ИТ",
  "Менеджер направления аудита информационных технологий": "Администрирование",
  "Заместитель руководителя отделения": "не ИТ",
  "Главный клиентский менеджер. Управление продаж клиентам малого бизнеса": "не ИТ",
  "Персональный менеджер Домклик": "не ИТ",
  "Ведущий менеджер по обслуживанию физических лиц": "не ИТ",
  "Кассир банка": "не ИТ",
  "Портфельный риск-аналитик": "Аналитика данных/ Data Science",
  "Менеджер по внутренним и внешним коммуникациям": "не ИТ",
  "Старший специалист отдела сопровождения транзакций": "не ИТ",
  "Руководитель операционного сопровождения": "не ИТ",
  "Ведущий специалист отдела по работе с мошенничеством": "не ИТ",
  "Экономист, менеджер по обслуживанию юр. лиц": "не ИТ",
  "Специалист отдела снабжения": "не ИТ",
  "Главный инженер программист": "Разработка",
  "Эксперт по мониторингу обьектов недвижимости": "не ИТ",
  "Эксперт по сопровождению аккредитации": "не ИТ",
  "Начальник отдела продаж корпоративным клиентам": "не ИТ",
  "Специалист по хозяйственной эксплуатации": "не ИТ",
  "Главный инженер отдела мониторинга информационной инфраструктуры": "Администрирование",
  "Главный ИТ-инженер": "Администрирование",
  "Стажер-программист": "Разработка",
  "Специалист отдела кредитного мониторинга": "не ИТ",
  "Главный инженер по разработке, senior UX/UI дизайнер": "Дизайн",
  "Руководитель направления тестирования": "Тестирование",
  "Ведущий инженер проекта": "не ИТ",
  "Экономист, начальник сектора обслуживания юридических лиц": "не ИТ",
  "Руководитель офиса СберПервый": "не ИТ",
  "Руководитель направления прямых продаж": "не ИТ",
  "Тренер-администратор спортивного комплекса": "не ИТ",
  "Главный специалист отдела контроля операций на финансовых рынках, управление Комплаенс": "не ИТ",
  "Клиентский менеджер Сбербанк Прьемьер": "не ИТ",
  "Менеджер поддержки продаж": "не ИТ",
  "Старший клиентский  менеджер": "не ИТ",
  "Ассистент клиентского менеджера сбербанк премьер": "не ИТ",
  "Ведущий специалист отдела недвижимости и развития инфраструктуры": "не ИТ",
  "Python разработчик": "Разработка",
  "Начальник отдела охраны": "не ИТ",
  "Владелец группы продуктов - исполнительный директор": "не ИТ",
  "Руководитель группы мониторинга": "не ИТ",
  "Водитель мобильного офиса": "не ИТ",
  "Старший специалист по открытию счетов": "не ИТ",
  "Специалист по сопровождению зарплатых проектов": "не ИТ",
  "Ведущий специалист сектора использования документов ОСЦ Архив": "не ИТ",
  "Руководитель группы активных продаж по телефону": "не ИТ",
  "Фронтенд разработчик (Старший)": "Разработка",
  "Старший инженер, Ведущий инженер, Главный инженер": "Разработка",
  "Начальник отдела кредитования": "не ИТ",
  "Менеджер по продуктам глобальных рынков для корпоративных клиентов": "не ИТ",
  "Клиентский менеджер/ заместитель руководителя/ бэк-офис отдел тхв": "не ИТ",
  "Эксперт SAP УВХД": "не ИТ",
  "Product owner Релизной команды": "не ИТ",
  "Старший инспектор отдела инкассациии": "не ИТ",
  "Менеджер по продажам и сопровождению платежных сервисов": "не ИТ",
  "Старший специалист по обслуживанию частных лиц": "не ИТ",
  "Руководитель проектов (государственные меры поддержки на единой карте МИР)": "не ИТ",
  "Golang developer": "Разработка",
  "Ведущий инженер-разработчик java": "Разработка",
  "Веб-разработчик": "Разработка",
  "Лидер стрима, Владелец продукта": "не ИТ",
  "Специалист по цифровым технологиям аудита": "не ИТ",
  "консультант юридических лиц": "не ИТ",
  "Охраник": "не ИТ",
  "Ведущий инженер сопровождения АС": "Администрирование",
  "Главный специалист отдела сопровождения банковских договоров": "не ИТ",
  "Developer Relations": "Разработка",
  "Ассистент менеджера по работе с корпоративными клиентами": "не ИТ",
  "Старший специалист отдела маркетинга": "не ИТ",
  "Стажер Data Engineer": "Аналитика данных/ Data Science",
  "Директор Управления продаж и обслуживания в сети ВСП": "не ИТ",
  "Руководитель направления внутренних коммуникаций": "не ИТ",
  "Исполнительный директор, старший бизнес-партнер": "не ИТ",
  "Начальник сектора кредитования юридидеских лиц": "не ИТ",
  "Middle Python Developer": "Разработка",
  "Аналитик, координатор кластера, личный помощник руководителя": "не ИТ",
  "Head of the Data Analysis Department": "Аналитика данных/ Data Science",
  "Ведущий менеджер по работе с клиентами в сегменте Премьер": "не ИТ",
  "Руководитель направления Центр Медицинские Продукты и Сервисы": "не ИТ",
  "Начальник отдела клиентской работы": "не ИТ",
  "Консультант ПФР": "не ИТ",
  "Аудитор-ревизор": "не ИТ",
  "Начальник смены\nУправление андеррайтинга физических лиц и микробизнеса г.Екатеринбург, Подразделение центрального подчинения Межрегион": "не ИТ",
  "Консультант по сервисам": "не ИТ",
  "Intern Data Scientist": "Аналитика данных/ Data Science",
  "Старший клиенский менеджер": "не ИТ",
  "Старший менеджер ключевых клиентов": "не ИТ",
  "Начальник отдела/Зам. начальника отдела корпоративного андеррайтинга": "не ИТ",
  "Территориальный менеджер, руководитель группы ВСП, коуч по продажам страховых продуктов": "не ИТ",
  "Директор центра персонального обслуживания": "не ИТ",
  "Секретарь коллегиальных органов": "не ИТ",
  "Backend lead developer": "Разработка",
  "Ведущий инженер по разработке (системный/бизнес аналитик)": "Разработка",
  "Специалист по сопровождению ключевых клиентов Управления прямых продаж с функцией аналитика": "не ИТ",
  "Ведущий инженер по сопровождению ITSM": "Администрирование",
  "Старший менеджер по обслуживанию физических лиц (подменный фонд)": "не ИТ",
  "Старший специалист по работе с обращениями": "не ИТ",
  "Менеджер по ключевым клиентам": "не ИТ",
  "Специалист Управления Делами": "не ИТ",
  "Старший инспектор по ценным бумагам": "не ИТ",
  "Консультант по работе с ПФР Управление по работе с партнёрами Московского банка ПАО Сбербанк": "не ИТ",
  "DevOps/SRE engineer": "Администрирование",
  "Финансовый эксперт СберПервый": "не ИТ",
  "Менеджер по продажам продуктов канала эквайринга": "не ИТ",
  "Специалист активных продаж по телефону (телемаркетинг)": "не ИТ",
  "Специалист центра заботы о клиентах": "не ИТ",
  "Старший менеджер по работе с клиентами малого бизнеса": "не ИТ",
  "Мобильный клиентский менеджер": "не ИТ",
  "Главный специалист кадрового администрирования HR": "не ИТ",
  "Специалист по урегулированию проблемной задолженности": "не ИТ",
  "Главный специалист отдела досудебного взыскания задолженности": "не ИТ",
  "Руководитель направления рыночных рисков, риски благосостояния": "не ИТ",
  "Старший специалист бэкофиса": "не ИТ",
  "Заместитель руководителя, старший менеджер по обслуживанию": "не ИТ",
  "Старший специалист отдела судебного и исполнительного производства": "не ИТ",
  "менеджер по выдаче и обслуживанию кредитов физических лиц": "не ИТ",
  "Старший специалист сектора по работе с мошенничеством": "не ИТ",
  "Инвестиционный менеджер СБ1": "не ИТ",
  "Старший инженер отдела информационных технологий": "Администрирование",
  "Ведущий инженер DevOps": "Администрирование",
  "специалист по информационным технологиям аудита": "Администрирование",
  "Старший юрисконсульт юридического отдела": "не ИТ",
  "Водитель-охранник": "не ИТ",
  "Стажёр-дизайнер": "Дизайн",
  "Заместитель начальника юридического отдела": "не ИТ",
  "Стажер \n": "не ИТ",
  "Менеджер  управления по работе с проблемной задолженностью": "не ИТ",
  "Middle Data Scientist": "Аналитика данных/ Data Science",
  "Специалист по аккредитации (группа сопровождения фидов)": "не ИТ",
  "Контроллер кассир": "не ИТ",
  "Менеджер отдела продаж по работе с ключевыми клиентами": "не ИТ",
  "Заместитель руководителя ДО": "не ИТ",
  "Руководитель дополнительного офиса банка": "не ИТ",
  "экономист отдела сопровождения и оформления банковских операций": "не ИТ",
  "Ведущий специалист судебного исполнительного производства": "не ИТ",
  "Agile-коуч": "не ИТ",
  "Начальник управления продаж и обслуживания в сети офисов": "не ИТ",
  "специалист Отдела поддержки прямых продаж Управления  прямых продаж": "не ИТ",
  "Начальник отдела кредитования юридических лиц": "не ИТ",
  "Специалист по обслуживанию корпоративных клиентов": "не ИТ",
  "Ведущий специалист ОКС": "не ИТ",
  "Менеджер продуктов эквайринга": "не ИТ",
  "Ux аналитик": "Дизайн",
  "Менеджер коммуникаций": "не ИТ",
  "Главный инспектор сектора обслуживания юридических/физических лиц": "не ИТ",
  "Руководителя офиса": "не ИТ",
  "Специалист отдела клиентских сервисов": "не ИТ",
  "Начальник управления учета и отчетности операций на глобальном рынке": "не ИТ",
  "Специалист по оформлению кредитных заявок": "не ИТ",
  "Управляющий дополнительным офисом": "не ИТ",
  "Менеджер по сервису": "не ИТ",
  "Эксперт внутренних коммуникаций": "не ИТ",
  "Руководитель направления системной аналитики": "Разработка",
  "Руководитель претензионной службы": "не ИТ",
  "Директор проектов": "не ИТ",
  "Оператор отдела по работе с корпоративными клиентами": "не ИТ",
  "Главный специалист Управления кредитных рисков малого и микро бизнеса (Центр,аппарат)": "не ИТ",
  "Руководитель VIP-офиса СберПервый": "не ИТ",
  "Главный специалист группы отложенных операций": "не ИТ",
  "Руководитель направления, начальник отдела": "не ИТ",
  "OPS engineer": "Администрирование",
  "Младший специалист HR": "не ИТ",
  "Инспектор, аналитик": "не ИТ",
  "Клиентский менеджер Сбер Первый": "не ИТ",
  "Старший инженер удалённой ИТ-поддержки": "Администрирование",
  "Ведущий специалист Управления по работе с проблемными активами юридических лиц Головного отделения по Нижегородской области Волго-Вятск": "не ИТ",
  "Старший инженер по разработке (Middle SRE engineer)": "Администрирование",
  "Главный редактор HR-платформы «Пульс»": "не ИТ",
  "Менеджер по продажам, Старший менеджер по обслуживанию физических лиц": "не ИТ",
  "Ассистент клиентского менеджера Регионального государственного сектора": "не ИТ",
  "Руководитель центра ипотечного кредитования": "не ИТ",
  "Начальник отдела сопровождения кредитных операций клиентам малого бизнеса": "не ИТ",
  "Руководитель направления по аналитике брокерского бизнеса": "Аналитика данных/ Data Science",
  "Директор Центра комплексной поддержки": "не ИТ",
  "Старший менеджер по обслуживанию физических лиц и юридических": "не ИТ",
  "Директор специализированного офиса": "не ИТ",
  "Ассистент HR-менеджера": "не ИТ",
  "Клиентский менеджер в Сбербанк Премьер": "не ИТ",
  "Ведущий эксперт по технологиям": "не ИТ",
  "Специалист ОППП (отдел поддержки прямых продаж )": "не ИТ",
  "Релизный Менеджер": "Администрирование",
  "Начальник отдела торгового эквайринга": "не ИТ",
  "Главный специалист отдела индивидуального сопровождения клиентов": "не ИТ",
  "Инженер по работе с данными": "Аналитика данных/ Data Science",
  "Клиентский менеджер отдела поддержки клиентов ипотечного кредитования": "не ИТ",
  "Эксперт по залогам": "не ИТ",
  "Ведущий инженер Go-разработчик": "Разработка",
  "Главный инженер по разработке приложений на Android": "Разработка",
  "Главный андеррайтер": "не ИТ",
  "Практикант отдела экономической безопасности Управления экономической безопасности Департамента безопасности": "не ИТ",
  "DevOps (Developer)": "Администрирование",
  "Старший специалист отдела сопровождения проблемных кредитов": "не ИТ",
  "Ведущий аналитик бизнес-моделирования": "не ИТ",
  "Таргетолог": "не ИТ",
  "Старший менеджер по обслуживанию физических лиц / администратор": "не ИТ",
  "Ведущий менеджер по обслуживанию частных лиц": "не ИТ",
  "Менеджер по продажи ВЭД": "не ИТ",
  "Менеджер зарплатного направления": "не ИТ",
  "Специалист (Отдел сопровождения операций по движению имущества)": "не ИТ",
  "Руководитель направления по исследованию данных/Senior ML Engineer": "Аналитика данных/ Data Science",
  "Специалист третьей линии поддержки": "Администрирование",
  "Региональный операционный директор": "не ИТ",
  "Эксперт по рискам": "не ИТ",
  "Старший менеджер в отделе клиентской службы": "не ИТ",
  "Ведущий специалист отдела управления клиентов и счетов": "не ИТ",
  "Ведущий специалист группы управления зарплатными проектами и эквайринговыми сетями": "не ИТ",
  "Руководитель направления корпоративного волонтерства": "не ИТ",
  "Руководитель по внедрению проектов": "не ИТ",
  "Практикант в банке": "не ИТ",
  "Ассистент клиентского менеджера ВИП офиса": "не ИТ",
  "Начальник отдела организации продаж клиентам малого бизнеса": "не ИТ",
  "Старший менеджер по обслуживанию физических лиц.": "не ИТ",
  "Ведущий менеджер по работе с клиентами крупного бизнеса АПК": "не ИТ",
  "Руководитель группы перформанс-маркетинга": "не ИТ",
  "Менеджер по обслуживанию постоянных клиентов": "не ИТ",
  "Руководитель проекта(PO)": "не ИТ",
  "Менеджер по развитию направления": "не ИТ",
  "Главный айти-инженер": "Администрирование",
  "Старший специалист центра снабжения": "не ИТ",
  "Инженер нагрузочного тестирования": "Тестирование",
  "Менеджер по продаже продуктов канала": "не ИТ",
  "Главный менеджер по работе с ключевыми клиентами малого бизнеса": "не ИТ",
  "Специалист отдела документооборота Центр делового администрирования": "не ИТ",
  "Главный разработчик контактного центра.": "Разработка",
  "Кассир-контролер": "не ИТ",
  "Клиентский менеджер по работе с физическими лицами": "не ИТ",
  "Team Lead Службы координации": "не ИТ",
  "Начальник сектора клиентских менеджеров по обслуживанию юридических лиц": "не ИТ",
  "Главный специалист отдела методологии налогообложения": "не ИТ",
  "Начальник отдела по сопровождению ипотеки": "не ИТ",
  "Старший специалист\nОтдел связи и мониторинга управления инкассации Центра управления наличным денежным обращением Регионального Сервис": "не ИТ",
  "Кладовщик-комплектовщик": "не ИТ",
  "backend Python разработчик": "Разработка",
  "Ведущий инженер по разработке (Тестирование ПО)": "Тестирование",
  "Senior data engineer": "Аналитика данных/ Data Science",
  "ИТ Лидер кластера": "Администрирование",
  "Ведущий специалист службы безопасности": "не ИТ",
  "Руководитель направления (Владелец продукта)": "не ИТ",
  "Консультант, сервисный консультант, мобильный менеджер по продажам, клиентский менеджер": "не ИТ",
  "Руководитель группы отдела кассовых операций и операционной работы": "не ИТ",
  "Руководитель отдела Премьер": "не ИТ",
  "Руководитель направления Отдела сопровождения депозитарных операций": "не ИТ",
  "Старший инженер по разработке (QA)": "Тестирование",
  "Graphic designer": "Дизайн",
  "Ведущий специалист, руководитель подразделения": "не ИТ",
  "Контролёр Кп": "не ИТ",
  "Ведущий специалист отдела отложенных операций и сохранения клиентов": "не ИТ",
  "Охранник отдела службы безопасности": "не ИТ",
  "Оператор видео наблюдения": "не ИТ",
  "Главный специалист по внутреннему PR": "не ИТ",
  "Ведущий специалист отдела урегулирования проблемной задолженности": "не ИТ",
  "Ведущего специалиста по обслуживанию частных лиц": "не ИТ",
  "водитель-инкассатор (мастер)": "не ИТ",
  "Помощник повора": "не ИТ",
  "Эксперт по информационным технологиям аудита": "Администрирование",
  "Руководитель группы отдела продаж": "не ИТ",
  "Программист-консультант 1С": "Разработка",
  "Директор управления глобальных рынков Северо-Западный банк ПАО Сбербанк, Санкт-Петербург": "не ИТ",
  "Специалист отдела претензий": "не ИТ",
  "Backend python-разработчик": "Разработка",
  "Менеджер по сделкаам с недвижимостью": "не ИТ",
  "Специалист комплектования документов": "не ИТ",
  "Стажер-инженер": "Разработка",
  "Старший клиентский менеджер/ ведущий специалист по обслуживанию корпоративных клиентов": "не ИТ",
  "Ведущий специалист по работе с проблемной задолженностью": "не ИТ",
  "Старший клиентский менежер": "не ИТ",
  "Менеджер по работе с недвижимостью и  кредиванию": "не ИТ",
  "Менеджер по продажам, менеджер по обслуживанию": "не ИТ",
  "Главный специалист по безопасности": "не ИТ",
  "Главный специалист отдела судебного и исполнительно производства": "не ИТ",
  "Начальник отдела досудебного погашения задолженности": "не ИТ",
  "Главный специалист по работе с исполнительными документами по юридическим лицам": "не ИТ",
  "Заместитель руководителя ДО, начальник сектора по работе с корпоративными клиентами": "не ИТ",
  "Старший клиентский менеджер (Финансирование недвижимости)": "не ИТ",
  "Руководитель направления (блок технологии)": "Администрирование",
  "Кредитный инспектор сектор администрирования кредитов и оформления КОД ПАО Сбербанк": "не ИТ",
  "Аналитик управления организационно-кадровой экспертизы и вознаграждения": "не ИТ",
  "Старший специалист центра оперативной поддержки продаж и ипотечного кредитования": "не ИТ",
  "Руководитель направления Сбербанк Премьер": "не ИТ",
  "Руководитель группы специалистов по прямым продажам ( DSA)": "не ИТ",
  "Специалист дистанционного взыскания": "не ИТ",
  "Ведущий инженер систем ТВК": "Администрирование",
  "Ведущий специалист отдела банковского сопровождения долевого строительства": "не ИТ",
  "Data scientist / ML engineer": "Аналитика данных/ Data Science",
  "Ведущий менеджер по обслуживанию ФЛ и ЮЛ": "не ИТ",
  "Стажер — организатор мероприятий": "не ИТ",
  "AppSec инженер": "Администрирование",
  "Клиентский Менеджер \"Сбербанк Первый\"": "не ИТ",
  "Водитель персональный": "не ИТ",
  "Специалист сектора сопровождения судебного взыскания": "не ИТ",
  "Заведующий универсальным дополнительным офисом": "не ИТ",
  "Старший инспектор сектора кадрового администрирования": "не ИТ",
  "Ведущий инженер по разработке (тестирование)": "Тестирование",
  "Главный специалист по обучению": "не ИТ",
  "Менеджер по работе с ключевыми клиентами ММБ (KAM)": "не ИТ",
  "Старший специалист по взысканию задолженности": "не ИТ",
  "Ведущиий специалист по пожарной безопасности": "не ИТ",
  "Ведущий Java-разработчик": "Разработка",
  "Ассистент Заместителя Председателя Правления ПАО Сбербанк": "не ИТ",
  "Начальник сектора по торговому эквайрингу": "не ИТ",
  "Начальник сектора Управления по работе с обращениями клиентов": "не ИТ",
  "Бизнес аналитик": "не ИТ",
  "Главный специалист по сбору задолженности": "не ИТ",
  "Инкосатор": "не ИТ",
  "Ведущий специалист по исполнительному производству": "не ИТ",
  "Специалист отдела по вопросам комплаенс": "не ИТ",
  "DevOps Team Lead": "Администрирование",
  "Повар горячего цеха": "не ИТ",
  "Руководитель направления (Java)": "Разработка",
  "Менеждер развития отношений с партнерами": "не ИТ",
  "Старший инженер отдела по противодействию кибермошенничеству": "Администрирование",
  "Ведущий финансовый аналитик": "не ИТ",
  "Специалист отдела информационной безопасности": "Администрирование",
  "Стажёр Data Science": "Аналитика данных/ Data Science",
  "Руководитель IT-департамента системного администрирования": "Администрирование",
  "Главный клиентский менеджер по работе с ключевыми клиентами малого бизнеса": "не ИТ",
  "Стажёр (КБП)": "не ИТ",
  "Специалист по зарплатным проектаи": "не ИТ",
  "Junior Golang разработчик": "Разработка",
  "Нагрузочный тестировщик": "Тестирование",
  "Руководитель группы обслуживания вызовов": "не ИТ",
  "Помощник юрисконсульта": "не ИТ",
  "Руководитель по кредитованию": "не ИТ",
  "Дата аналитик": "Аналитика данных/ Data Science",
  "Начальник отдела управления кассовой ликвидностью": "не ИТ",
  "Инженер по тестированию (QA engineer)": "Тестирование",
  "Клиентский менеджер по обслуживанию корпоративных клиентов (малый бизнес)": "не ИТ",
  "Руководитель проекта/начальник отдела": "не ИТ",
  "Специалист отдела внутреннего контроля": "не ИТ",
  "Кассир операционной кассы": "не ИТ",
  "Главный специалист отдела контроля режима санкций": "не ИТ",
  "Специалист по работе с клиентами, специалист прямых продаж": "не ИТ",
  "Разработчик PL+ АБС ЦФТ (PL/SQL)": "Разработка",
  "Старший менеджер по обслуживанию физических лиц и юр.лиц": "не ИТ",
  "Специалист отдела инкассации": "не ИТ",
  "Бизнес-партнер по управлению персоналом": "не ИТ",
  "Управляющий директор, Директор Центра продаж корпоративным клиентам": "не ИТ",
  "Ведущий специалист/Инженер по кибербезопасности": "Администрирование",
  "DSA Специалист прямых продаж/Клиентский менеджер": "не ИТ",
  "Начальник управления безопасности": "не ИТ",
  "Начальник центра продаж клиентам малого бизнеса": "не ИТ",
  "Инженер-программист группы сопровождения": "Разработка",
  "Backend Software Engineer": "Разработка",
  "Начальник управления по работе с партнерами": "не ИТ",
  "Старший специалист сопровождения брокерских операций на финансовых рынках": "не ИТ",
  "Заместитель управляющего отделением Сбербанка России": "не ИТ",
  "Начальник сектора поддержки внутренних клиентов": "не ИТ",
  "Специалист центра мониторинга": "не ИТ",
  "Старший специалист сопровождения счетов": "не ИТ",
  "Водитель-инкассатор(мастер)": "не ИТ",
  "Начальник сектора клиентских менеджеров": "не ИТ",
  "Специалист сектора разметки данных": "Аналитика данных/ Data Science",
  "консульант по банковским продуктам": "не ИТ",
  "Стажер по направлению \"Право\"": "не ИТ",
  "Служба экономической безопасности": "не ИТ",
  "Старший специалист сектора разметки и аналитики данных": "Аналитика данных/ Data Science",
  "Разработчик электронного обучения": "Разработка",
  "HR BP": "не ИТ",
  "Администратор информационной безопасности": "Администрирование",
  "Руководитель кассово - инкассаторского центра": "не ИТ",
  "Старший специалист по охране труда": "не ИТ",
  "Менеджер по продадам": "не ИТ",
  "Старший специалист отдела по работе с ключевыми клиентами": "не ИТ",
  "Инкассатор мастер": "не ИТ",
  "Начальник отдела кассовых операций кассово-инкассаторского центра": "не ИТ",
  "ABAP-разработчик": "Разработка",
  "Ведущий инженер-исследователь": "Разработка",
  "Региональный руководитель сети премиального обслуживания": "не ИТ",
  "Руководитель направления по развитию офисов нового формата": "не ИТ",
  "Главный инженер разработчик": "Разработка",
  "Старший менеджер по обслуживанию юридических лиц": "не ИТ",
  "Технический Project Manager": "Администрирование",
  "Kotlin-разработчик": "Разработка",
  "Аналитик Отдела организации сервисов": "не ИТ",
  "Управляющий директор, Казначейство": "не ИТ",
  "Менеджер зала": "не ИТ",
  "Руководитель операционного офиса": "не ИТ",
  "Руководитель сети премьер офисов": "не ИТ",
  "Старший операционно-кассовый сотрудник": "не ИТ",
  "Специалист по обучению и развитию персонала": "не ИТ",
  "Специалист по валютному контролю": "не ИТ",
  "Engineer QA": "Тестирование",
  "Ведущий специалист отдела по работе с проблемной задолженностью": "не ИТ",
  "Менеджер по работе с ключевыми клиентами (малый бизнес)": "не ИТ",
  "Ассистент клиентского менеджера Премьер": "не ИТ",
  "Менеджер (аналитик данных)": "Аналитика данных/ Data Science",
  "Middle Golang Разработчик": "Разработка",
  "Старший сетевой инженер": "Администрирование",
  "Старший эксперт по разработке": "Разработка",
  "Кредитный инспектор малого бизнеса": "не ИТ",
  "Территориальный менеджер направления прямых продаж": "не ИТ",
  "Специалист по охране труда и охране окружающей среды": "не ИТ",
  "Ст.менеджер по обслуживанию фл": "не ИТ",
  "Клиентский менеджер по ипотечному кредитованию": "не ИТ",
  "Заместитель главного бухгалтера": "не ИТ",
  "Менеджер по маркетингу и связям с общественностью": "не ИТ",
  "Старший аналитик-аудитор IT": "Администрирование",
  "Финансовый консультант, менеджер по продажам": "не ИТ",
  "Клиентский менеджер выездного сервиса": "не ИТ",
  "Специалист сектора недвижимости": "не ИТ",
  "Ведущий инженер по разработке ПО": "Разработка",
  "главный специалист залоговой службы": "не ИТ",
  "Аналитик отдела маркетинга": "не ИТ",
  "Заместитель руководителя внутреннего структурного подразделения": "не ИТ",
  "Руководитель направления по развитию IT систем": "Администрирование",
  "Ведущий специалист Направление по взысканию проблемной задолженности ФЛ и КМС": "не ИТ",
  "Ревизор контрольно-ревизионного отдела": "не ИТ",
  "Главный инженер по разработке и тестированию": "Тестирование",
  "Senior Data Analyst": "Аналитика данных/ Data Science",
  "менеджер по нефинансовым сервисам": "не ИТ",
  "Начальник отдела сопровождения кредитных операций": "не ИТ",
  "Главный специалист отдела контроля качества": "не ИТ",
  "Ведущий менеджер по обслуживанию клиентов": "не ИТ",
  "Специалист обслуживания вызовов": "не ИТ",
  "Клиентский менеджер по продажам": "не ИТ",
  "Главный юрисконсульт отдела правового сопровождения финансирования недвижимости и инфраструктуры проектов": "не ИТ",
  "Менеждер": "не ИТ",
  "Клиентский менеджер Домклик": "не ИТ",
  "Специалист  обслуживания отдела вызовов": "не ИТ",
  "Ведущий специалист группы по работе с проблемными активами": "не ИТ",
  "Специалист по Финансовым рынкам": "не ИТ",
  "Консультант по работе с ПФР": "не ИТ",
  "Главный java-разработчик": "Разработка",
  "Junior DevOps инженер": "Администрирование",
  "Старший менеджер по обслуживанию физических лиц, менеджер по продажам": "не ИТ",
  "Ведущий специалист отдела валютного контроля": "не ИТ",
  "Специалист по работе с частными лицами; менеджер по продажам (физ, юр лица).": "не ИТ",
  "Менеджер по мониторингу и совершенствованию процессов": "не ИТ",
  "Старший клиентский менеджер отдела по работе с корпоративными клиентами по финансированию недвижимости": "не ИТ",
  "Главный специалист по работе с залогами": "не ИТ",
  "Middle frontend": "Разработка",
  "Руководитель ВИП ВСП, СберПервый": "не ИТ",
  "Директор по сопровождению (Лидер OPS) дивизиона Транзакционный Бизнес": "не ИТ",
  "Консультант по сервису": "не ИТ",
  "Главный специалист в отделе по работе с просроченной задолженностью": "не ИТ",
  "Руководитель проектов ИТ": "Администрирование",
  "администратор АС": "Администрирование",
  "Full-stack developer": "Разработка",
  "Главный специалист по технической поддержке": "Администрирование",
  "Технический руководитель проектов": "Администрирование",
  "Директор Управления по работе с Проблемными Активами юр. лиц. Северо-западного Сбербанка.": "не ИТ",
  "Редактор-переводчик": "не ИТ",
  "Клиентский менеджер. Универсальный дополнительный офис №8615/0171 Сибирского банка.": "не ИТ",
  "Менеджер по инновациям и краудсорсингу": "не ИТ",
  "Старший специалист отдела безопасности": "не ИТ",
  "Старший Клиентский менеджер Сбербанк Премьер": "не ИТ",
  "Кассир- операционист": "не ИТ",
  "Главный Специалист по работе с клиентами в банк": "не ИТ",
  "Старший операционист": "не ИТ",
  "Дежурный инженер": "Администрирование",
  "Quality Engineer": "Тестирование",
  "Старший специалист группы  продаж": "не ИТ",
  "Compiter vision engineer": "Разработка",
  "Менеджер по продажам\n": "не ИТ",
  "Эксперт по исследованию данных": "Аналитика данных/ Data Science",
  "Старший менеджер по продаже жилищных кредитов": "не ИТ",
  "Руководитель по сопровождению ипотеки": "не ИТ",
  "Начальник кассового центра": "не ИТ",
  "Ведущий юрисконсульт по работе с юридическими лицами": "не ИТ",
  "Специалист отдела обучения сотрудников ЕРКЦ": "не ИТ",
  "Специалист сектора сопровождения процедур банкротств": "не ИТ",
  "Бизнес ИТ аналитик": "Аналитика данных/ Data Science",
  "Начальник отдела кредитования физических и юридических лиц": "не ИТ",
  "Менеджер по работе с ключевым клиентами малого бизнеса": "не ИТ",
  "Специалист по обслуживания частных лиц": "не ИТ",
  "Главный специалист СЦ Архив": "не ИТ",
  "Специалист по обработке архива": "не ИТ",
  "Стажер-data-scientist": "Аналитика данных/ Data Science",
  "Инженер сопровождения и внедрения банковских продуктов": "Администрирование",
  "Региональный менеджер ипотечного кредитования": "не ИТ",
  "Старший инспектор по работе с проблемной задолженностью": "не ИТ",
  "Младший менеджер обслуживания": "не ИТ",
  "Специалист по работе с предприятиями": "не ИТ",
  "Инженер автоматизированного тестирования": "Тестирование",
  "Старший инженер QA": "Тестирование",
  "Начальник отдела кредитования корпоративных клиентов": "не ИТ",
  "Начальник отдела по работе с предприятиями": "не ИТ",
  "Специалист сопровождения договоров и расчетов": "не ИТ",
  "Старший специалист поддержки корпоративных клиентов": "не ИТ",
  "Главный специалист отдела кредитования": "не ИТ",
  "Региональный менеджер отдела по работе с партнерами": "не ИТ",
  "Андеррайтер сегмента малого бизнеса": "не ИТ",
  "Middle Frontend-разработчик": "Разработка",
  "Специалист по коммерческой недвижимости": "не ИТ",
  "Специалист отдела безопасности": "не ИТ",
  "Менеджер сектора формирования отчетности": "не ИТ",
  "Ведущий специалист секретариата": "не ИТ",
  "Ведущий специалист отдела андеррайтинга": "не ИТ",
  "Кредитный администратор": "не ИТ",
  "Старший кассир кассы пересчета": "не ИТ",
  "Личный водитель": "не ИТ",
  "Менеджер управления операционной поддержки крупнейших клиентов": "не ИТ",
  "Начальник отдела оформления и сопровождения банковских операций": "не ИТ",
  "Старший специалист прямых продаж": "не ИТ",
  "Директор управления по финансовой грамотности и поддержке продаж продуктов благосостояния": "не ИТ",
  "Начальник транспортного обеспечения": "не ИТ",
  "Ведущий специалист управления по работе с проблемной задолженностью": "не ИТ",
  "Старший экономист управления продаж розничных продуктов": "не ИТ",
  "Инспектор по кассовой работе": "не ИТ",
  "Специалист отдела сопровождения договоров и расчетов": "не ИТ",
  "Руководитель направления развития дебетовых карт": "не ИТ",
  "Менеджер первой линии": "не ИТ",
  "Ведущий специалист отдела инкассации": "не ИТ",
  "Руководитель направления отдела прямых продаж": "не ИТ",
  "Старший кредитный инспектор отдела кредитования юридических лиц": "не ИТ",
  "Главный специалист по прогнозированию": "не ИТ",
  "Руководитель направления в ЦА": "не ИТ",
  "Старший менеджер по развитию отношений с партнерами": "не ИТ",
  "Менеджер по обслуживанию ипотечного кредитования": "не ИТ",
  "Ведущий специалист по работе с обращениями клиентов": "не ИТ",
  "Главный системный аналитик": "Разработка",
  "Главный кредитный аналитик малого бизнеса": "не ИТ",
  "Senior data scientist": "Аналитика данных/ Data Science",
  "Старший инженер-разработчик": "Разработка",
  "Customer sales Manager": "не ИТ",
  "Руководитель проектов Департамента стратегии и развития": "не ИТ",
  "Специалист по снабжению": "не ИТ",
  "Руководитель направления по работе с ключевыми клиентами": "не ИТ",
  "Консультант по продажам банковски услуг": "не ИТ",
  "Операционист-кассир": "не ИТ",
  "Специалист по работе с персоналом": "не ИТ",
  "Старший менеджер подразделения по работе с предприятиями оптовой торговли": "не ИТ",
  "Менеджер внешней дистрибуции": "не ИТ",
  "Начальник отдела по работе с корпоративными клиентами недвижимости": "не ИТ",
  "Эксперт центра операционной поддержки устройств самообслуживания": "не ИТ",
  "Фрод-аналитик": "Аналитика данных/ Data Science",
  "Главный специалист управления по работе с проблемными активами юридических лиц": "не ИТ",
  "Менеджер по событийным мероприятиям и коммуникациям": "не ИТ",
  "Менеджер по новым сервисам Сбера": "не ИТ",
  "Ведущий инженер-тестировщик": "Тестирование",
  "Начальник отдела контроля качества": "не ИТ",
  "Руководитель КПК в сегменте крупный и средний бизнес": "не ИТ",
  "Начальник отдела эксплуатации": "не ИТ",
  "Старший страховой брокер": "не ИТ",
  "Руководитель направления оценки рисков цифрового бизнеса": "не ИТ",
  "Специалист управления регионального контактного центра": "не ИТ",
  "Руководитель направления Премьер": "не ИТ",
  "Начальник производства": "не ИТ",
  "Психолог-консультант": "не ИТ",
  "Аналитик рисков": "Аналитика данных/ Data Science",
  "Пенсионный консультант": "не ИТ",
  "Старший специалист сектора недвижимости и развития инфраструктуры": "не ИТ",
  "Ведущий специалист операционной поддержки юридических лиц": "не ИТ",
  "Инженер по сопровождению DevOps инструментов": "Администрирование",
  "Процессный аналитик": "не ИТ",
  "Главный специалист отдела сопровождения кредитных операций": "не ИТ",
  "Менеджер по ключевым клиентам микро и малого бизнеса": "не ИТ",
  "Администратор SAP BASIS": "Администрирование",
  "Инженер 5 разряда": "не ИТ",
  "Cyber Security Chief Engineer": "Администрирование",
  "Senior Frontend": "Разработка",
  "Старший специалист подразделения операционной поддержки": "не ИТ",
  "Менеджер по работе с крупными клиентами": "не ИТ",
  "Старший специалист регионального контактного центра": "не ИТ",
  "Руководитель направления по развитию ИТ-систем": "Администрирование",
  "Мобильный менеджер продаж": "не ИТ",
  "Инструктор по йоге": "не ИТ",
  "Старший специалист сектора судебного и исполнительного производства": "не ИТ",
  "Эксперт управления маркетинга и коммуникаций": "не ИТ",
  "Главный специалист по работе с просроченной задолженностью": "не ИТ",
  "Программист Go": "Разработка",
  "Старший водитель": "не ИТ",
  "Специалист сектора обслуживания вызовов управления регионального контактного центра": "не ИТ",
  "Эксперт учебного центра": "не ИТ",
  "Менеджер по продуктам глобальных рынков": "не ИТ",
  "Эксперт по охране труда": "не ИТ",
  "Операционно-кассовый работник": "не ИТ",
  "Начальник отдела взыскания": "не ИТ",
  "Начальник отдела урегулирования проблемной задолженности": "не ИТ",
  "Старший клиентский менеджер по работе с клиентами среднего и крупного бизнеса": "не ИТ",
  "Специалист отдела дистанционного взыскания": "не ИТ",
  "Ручной тестировщик": "Тестирование",
  "Главный специалист отдела вкладов и специальных программ": "не ИТ",
  "Старший менеджер по обслуживанию ВСП": "не ИТ",
  "Старший инженер сервисного центра": "Администрирование",
  "Специалист по административно-хозяйственной работе": "не ИТ",
  "Старший специалист отдела по работе с юридическими лицами": "не ИТ",
  "Менеджер по оформлению кредитов": "не ИТ",
  "Старший специалист-аналитик": "Аналитика данных/ Data Science",
  "IT-Recruiter": "не ИТ",
  "Руководитель проектов CJE": "не ИТ",
  "Старший клиентский менеджер физических лиц": "не ИТ",
  "Методолог производственного процесса": "не ИТ",
  "Старший дизайнер продукта": "Дизайн",
  "Старший инспектор отдела кадров": "не ИТ",
  "Инженер отдела мониторинга и территориального развития": "не ИТ",
  "Руководитель дополнительного специализированного офиса": "не ИТ",
  "Группа организации и развития продаж по телефону": "не ИТ",
  "Эксперт подбора персонала": "не ИТ",
  "Менеджер направления маркетинга и коммуникаций": "не ИТ",
  "Специалист отдела судебного и исполнительного производства": "не ИТ",
  "Программист ABAP": "Разработка",
  "Старший специалист управления по работе с просроченной задолженностью": "не ИТ",
  "Ведущий специалист группы обслуживания вызовов": "не ИТ",
  "Инспектор отдела корпоративных клиентов": "не ИТ",
  "Начальник сектора инкассации и перевозки ценностей": "не ИТ",
  "Менеджер по развитию продаж и обучению": "не ИТ",
  "Middle DevOps Engineer": "Администрирование",
  "Заместитель начальника отдела проблемных активов юридических лиц": "не ИТ",
  "Ведущий разработчик Java": "Разработка",
  "Начальник отдела безопасности": "не ИТ",
  "Старший инспектор сектора валютных и неторговых операций": "не ИТ",
  "Главный специалист сектора недвижимости": "не ИТ",
  "Менеджер по сопровождению зарплатных проектов": "не ИТ",
  "Менеджер по ВЭД": "не ИТ",
  "Старший специалист центра урегулирования задолженности": "не ИТ",
  "Корпоративный андеррайтер": "не ИТ",
  "Ведущий бухгалтер отдела контроля за вкладными операциями": "не ИТ",
  "Главный аудитор-аналитик данных": "Аналитика данных/ Data Science",
  "Старший IT-аудитор": "Администрирование",
  "Старший эксперт отдела привлечения": "не ИТ",
  "Менеджер по работе с зарплатными клиентами": "не ИТ",
  "Специалист по прямым продажам отдела по работе с предприятиями": "не ИТ",
  "Специалист отдела мониторинга залогового обеспечения": "не ИТ",
  "Старший инспектор по работе с проблемными активами юридических лиц": "не ИТ",
  "Водитель-механик": "не ИТ",
  "Начальник отдела подбора и адаптации персонала": "не ИТ",
  "Специалист отдела взыскания и просроченной задолженности": "не ИТ",
  "Менеджер по координации деятельности": "не ИТ",
  "Специалист по обработке письменных обращений": "не ИТ",
  "Старший специалист отдела качества сервисов": "не ИТ",
  "Заместитель управляющего отделением банка": "не ИТ",
  "Главный клиентский менеджер по привлечению ключевых клиентов малого бизнеса": "не ИТ",
  "Старший инженер отдела администрирования средств защиты": "Администрирование",
  "Главный специалист отдела досудебного взыскания": "не ИТ",
  "Специалист по работе с обращениями сотрудников": "не ИТ",
  "Руководитель офиса по работе с премиальными клиентами": "не ИТ",
  "Территориальный менеджер по технической поддержке продуктов эквайринга": "Администрирование",
  "Исполнительный  директор по hr стратегии и управлению опытом сотрудников": "не ИТ",
  "Руководитель направления IOS разработки": "Разработка",
  "Ведущий специалист отдела недвижимости": "не ИТ",
  "Менеджер по сопровождению эквайринга": "не ИТ",
  "Менеджер направления (HR-менеджер подразделения)": "не ИТ",
  "Специалист офиса продаж": "не ИТ",
  "Кассир сектора пересчета денежной наличности и ценностей": "не ИТ",
  "Региональный руководитель офисов": "не ИТ",
  "Специалист управления по работе с персоналом": "не ИТ",
  "Руководитель центра персонального обслуживания Премьер": "не ИТ",
  "Руководитель направления автоматизации": "не ИТ",
  "Главный аналитик отдела кредитования": "не ИТ",
  "Специалист по обслуживанию ключевых клиентов": "не ИТ",
  "Менеджер высокодоходного сектора": "не ИТ",
  "Территориальный директор": "не ИТ",
  "Специалист единого распределительного контактного центра": "не ИТ",
  "Специалист операционной поддержки": "не ИТ",
  "Начальник отдела недвижимости и развития инфраструктуры": "не ИТ",
  "Ведущий инспектор отдела обработки операций по банковским картам": "не ИТ",
  "Руководитель отдела внешних и внутренних коммуникаций": "не ИТ",
  "Инспектор службы безопасности": "не ИТ",
  "Старший специалист по обслуживанию клиентов": "не ИТ",
  "Региональный менеджер отдела продаж малого бизнеса": "не ИТ",
  "Ассистент по работе с ВИП-клиентами": "не ИТ",
  "Главный инженер отдела информатики": "Администрирование",
  "Ведущий специалист сектора проведения исследований и опросов": "не ИТ",
  "Начальник отдела по работе с клиентами": "не ИТ",
  "Ведущий специалист отдела сопровождения": "не ИТ",
  "Управляющий филиалом банка": "не ИТ",
  "Старший инженер отдела разработки": "Разработка",
  "Начальник сектора по работе с ключевыми клиентами ММБ": "не ИТ",
  "Заместитель начальника отдела обслуживания вызовов": "не ИТ",
  "Ведущий специалист группы противодействия мошенничеству": "не ИТ",
  "Руководитель направления веб-разработки": "Разработка",
  "Руководитель универсального дополнительного офиса": "не ИТ",
  "Инженер группы сопровождения платформы": "Администрирование",
  "Главный специалист отдела информационных технологий": "Администрирование",
  "Менеджер по организации и поддержке продаж сложных продуктов / тренер": "не ИТ",
  "SAM аналитик": "Аналитика данных/ Data Science",
  "Менеджер по работе с ключевыми клиентами ММБ": "не ИТ",
  "Старший специалист по сопровождению и контролю банковских операций": "не ИТ",
  "Старший контролер Комплаенс": "не ИТ",
  "Специалист центра поддержки партнеров и ипотечного кредитования": "не ИТ",
  "Старший сервис-менеджер по обслуживанию зарплатных проектов": "не ИТ",
  "Senior expert": "не ИТ",
  "Менеджер по работе с физическими лицами": "не ИТ",
  "Руководитель универсального офиса": "не ИТ",
  "Специалист технической поддержки торгового эквайринга": "Администрирование",
  "Ведущий бухгалтер отдела бухгалтерского учета и отчетности": "не ИТ",
  "Уборщица": "не ИТ",
  "Менеджер по дистанционному обучению": "не ИТ",
  "Заведующий кассой дополнительного офиса": "не ИТ",
  "Дизайнер презентаций": "Дизайн",
  "Специалист отдела автоматизации": "Разработка",
  "Старший аналитик отдела анализа розничного и микробизнеса": "Аналитика данных/ Data Science",
  "Руководитель групп тестирования": "Тестирование",
  "Клиентский менеджер по работе с премиальными клиентами": "не ИТ",
  "Эксперт отдела маркетинга и коммуникаций": "не ИТ",
  "Дизайнер продукта": "Дизайн",
  "Механик по выпуску спецавтомобилей": "не ИТ",
  "Начальник юридического отдела": "не ИТ",
  "Инженер группы сопровождения": "Администрирование",
  "Инспектор управления формирования сводной отчетности": "не ИТ",
  "Менеджер по логистике": "не ИТ",
  "Менеджер по развитию прямых продаж и работе с ключевыми клиентами": "не ИТ",
  "Менеджер агентского канала": "не ИТ",
  "Кассир ОКО": "не ИТ",
  "Главный специалист по работе с проблемной задолженностью": "не ИТ",
  "Старший клиентский менеджер корпоративного блока": "не ИТ",
  "Ведущий инженер по сопровождению и администрированию сложных интегрированных ИТ-систем": "Администрирование",
  "Разработчик Java": "Разработка",
  "Начальник сектора продаж по работе с ключевыми клиентами": "не ИТ",
  "Менеджер по продажам торгового эквайринга": "не ИТ",
  "Главный инспектор управления безопасности": "не ИТ",
  "Сварщик": "не ИТ",
  "Руководитель направления по инженерно-техническому сопровождению": "не ИТ",
  "Главный специалист ЦЗК": "не ИТ",
  "Куратор проекта": "не ИТ",
  "Стажер экономического отдела": "не ИТ",
  "Управляющий отделением": "не ИТ",
  "Инженер-стажер": "Разработка",
  "Специалист отдела расчетов по оплате труда": "не ИТ",
  "Менеджер по ипотечным сделкам": "не ИТ",
  "Старший специалист по прямым продажам": "не ИТ",
  "Водитель-инкассатор ПТС": "не ИТ",
  "Старший специалист отдела по безопасности": "не ИТ",
  "Менеджер по управлению рисками": "не ИТ",
  "Ведущий специалист по работе с обращениями клиентов юридических лиц": "не ИТ",
  "QA Automation Java": "Тестирование",
  "Аналитик учебного центра": "не ИТ",
  "Тренинг-менеджер": "не ИТ",
  "Сервисный инженер": "Администрирование",
  "Ведущий специалист отдела обработки документов исполнительного производства": "не ИТ",
  "Специалист по работе с базами данных": "Аналитика данных/ Data Science",
  "Руководитель по кредитованию управления кредитования": "не ИТ",
  "Старший специалист отдела мониторинга залогов": "не ИТ",
  "Начальник управления по работе с проблемными активами юридических лиц": "не ИТ",
  "Ведущий специалист отдела мониторига залогов": "не ИТ",
  "Главный инженер отдела экспертизы": "не ИТ",
  "Специалист по работе с первичной документацией": "не ИТ",
  "Менеджер по поддержке страховых продуктов": "не ИТ",
  "Менеджер по сопровождению ключевых клиентов": "не ИТ",
  "Руководитель ипотечного центра": "не ИТ",
  "Начальник финансового отдела": "не ИТ",
  "Менеджер центра управления проектами и процессами": "не ИТ",
  "PHP Developer": "Разработка",
  "Back-end Java Junior": "Разработка",
  "Инженер мобильной разработки": "Разработка",
  "Фронтенд-разработчик": "Разработка",
  "Клиентский менеджер крупного бизнеса": "не ИТ",
  "Главный специалист отдела мониторинга залога": "не ИТ",
  "Директор проектного офиса": "не ИТ",
  "Technical Project Manager": "Администрирование",
  "Руководитель направления ИТ": "Администрирование",
  "Инспектор по работе с персоналом": "не ИТ",
  "Главный инженер сектора поддержки ИТ-процессов": "Администрирование",
  "Ведущий инженер по сопровождению АС": "Администрирование",
  "Ведущий инспектор управления инкассации": "не ИТ",
  "Начальник службы экономической безопасности": "не ИТ",
  "Начальник юридического сектора": "не ИТ",
  "Начальник управления кредитования": "не ИТ",
  "Специалист административного отдела": "не ИТ",
  "Специалист поддержки продаж": "не ИТ",
  "Трекер продуктового офиса": "не ИТ",
  "Операционист сектора юридических лиц": "не ИТ",
  "Руководитель направления обучения и развития": "не ИТ",
  "Руководитель по работе с предприятиями и партнерами": "не ИТ",
  "Эксперт по закупкам": "не ИТ",
  "Руководитель направления управления по работе с проблемными активами": "не ИТ",
  "Старший специалист залоговой экспертизы": "не ИТ",
  "Ведущий инженер по автоматизации": "Разработка",
  "Региональный менеджер по торговому маркетингу": "не ИТ",
  "Старший специалист развития инфраструктуры": "не ИТ",
  "Специалист отдела ипотечного кредитования": "не ИТ",
  "Специалист отдела залоговой экспертизы": "не ИТ",
  "Старший специалист отдела взыскания физических лиц": "не ИТ",
  "Ведущий специалист администрирования кредитов крупного и среднего бизнеса": "не ИТ",
  "Руководитель отдела внедрения проектов": "не ИТ",
  "Юрисконсульт дополнительного офиса": "не ИТ",
  "Старший специалист единого распределенного контактного центра": "не ИТ",
  "UX Lead": "Дизайн",
  "Ведущий специалист отдела сопровождения судебного и исполнительного производства": "не ИТ",
  "Инженер дежурной смены DevOps": "Администрирование",
  "Ведущий специалист управления качества и эффективности": "не ИТ",
  "Аналитик-эксперт": "Аналитика данных/ Data Science",
  "BI-разработчик": "Аналитика данных/ Data Science",
  "Специалист по планово-аналитической работе и обеспечению": "не ИТ",
  "Специалист по обработке обращений клиентов": "не ИТ",
  "Менеджер пункта выдачи": "не ИТ",
  "Координатор тестовых стендов": "Тестирование",
  "Тренер тренажерного зала": "не ИТ",
  "IT валидатор": "Администрирование",
  "Ведущий бухгалтер по валютным операциям": "не ИТ",
  "Заместитель руководителя офиса по работе с физическими лицами": "не ИТ",
  "Менеджер по сервисной поддержке эквайринга": "не ИТ",
  "UX-Writer": "Дизайн",
  "Главный инженер по нагрузочному тестированию": "Тестирование",
  "Специалист отдела управления делами": "не ИТ",
  "UX UI Designer": "Дизайн",
  "Руководитель предприятия": "не ИТ",
  "Старший инспектор сектора кредитования физических лиц": "не ИТ",
  "Руководитель направления поддержки Экосистемы Сбербанка": "не ИТ",
  "Начальник отдела претензионной службы банка": "не ИТ",
  "Бухгалтер сектора сводной отчетности": "не ИТ",
  "Администратор электронного документооборота": "Администрирование",
  "Главный IT-инженер": "Администрирование",
  "Инспектор сектора сопровождения": "не ИТ",
  "Аналитик финансово-экономического управления": "не ИТ",
  "Врач-терапевт": "не ИТ",
  "Кредитный аналитик крупного и среднего бизнеса": "не ИТ",
  "Старший специалист дирекции сервисных центров": "не ИТ",
  "Инженер-DevOps": "Администрирование",
  "Frontеnd Team Lead": "Разработка",
  "Клиентский менеджер по работе с юридическими лицами": "не ИТ",
  "Тестировщик программного обеспечения": "Тестирование",
  "Менеджер по работе с крупным и средним бизнесом": "не ИТ",
  "Программист Java": "Разработка",
  "Инспектор отдела досудебного погашения задолженности": "не ИТ",
  "Менеджер по подбору цифровых талантов": "не ИТ",
  "Заместитель руководителя офиса по обслуживанию значимых клиентов": "не ИТ",
  "Старший специалист отдела сопровождения документарных и конверсионных операций": "не ИТ",
  "Руководитель группы отдела телемаркетинга": "не ИТ",
  "Оператор сектора делопроизводства": "не ИТ",
  "Операционный менеджер по работе с состоятельными клиентами": "не ИТ",
  "Старший специалист АХО": "не ИТ",
  "Региональный менеджер по ВЭД": "не ИТ",
  "Старший менеджер по продажам жилищных кредитов": "не ИТ",
  "Специалист по сопровождению операций": "не ИТ",
  "Начальник управления дистанционного взыскания": "не ИТ",
  "Специалист отдела проверки документации": "не ИТ",
  "Аналитик рисков в кредитовании": "Аналитика данных/ Data Science",
  "Начальник сектора кредитования": "не ИТ",
  "Специалист по выдаче зарплатных карт": "не ИТ",
  "Администратор Linux": "Администрирование",
  "Младший тестировщик": "Тестирование",
  "Заместитель руководителя центра ипотечного кредитования": "не ИТ",
  "Middle Frontend Developer": "Разработка",
  "Senior QA Automation Engineer": "Тестирование",
  "Executive Director": "не ИТ",
  "Главный специалист отдела досудебного погашения задолженности": "не ИТ",
  "Ведущий инженер мониторинга IT-инфраструктуры": "Администрирование",
  "Специалист отдела расчетов": "не ИТ",
  "Ведущий ревизор": "не ИТ",
  "Head of Projects": "не ИТ",
  "Руководитель отдела по работе с корпоративными клиентами": "не ИТ",
  "Старший специалист по банковским картам": "не ИТ",
  "Ведущий специалист отдела залоговой экспертизы": "не ИТ",
  "Заместитель руководителя операционного офиса": "не ИТ",
  "Инженер пользовательских операционных систем": "Администрирование",
  "Инспектор отдела кредитной работы": "не ИТ",
  "Заместитель руководителя филиала по работе с физическими лицами": "не ИТ",
  "Специалист отдела экономической безопасности": "не ИТ",
  "Аналитик-дизайнер": "Аналитика данных/ Data Science",
  "Начальник сектора технической поддержки": "Администрирование",
  "Ведущий it-инженер": "Администрирование",
  "Операционный руководитель": "не ИТ",
  "Руководитель группы специалистов прямых продаж": "не ИТ",
  "Менеджер по внедрению стандартных операционных процедур": "не ИТ",
  "Ведущий кассир-операционист": "не ИТ",
  "Старший специалист управления кассовой работы": "не ИТ",
  "Специалист рекрутингового центра": "не ИТ",
  "Инженер ПТО": "не ИТ",
  "Руководитель направления ИТ-архитектуры": "Разработка",
  "Scrum Master": "не ИТ",
  "Главный руководитель направления развития информационных систем": "Администрирование",
  "Ведущий инженер разработки": "Разработка",
  "Инспектор сектора обслуживания юридических лиц": "не ИТ",
  "Старший специалист по работе с обращениями юридических лиц": "не ИТ",
  "Ведущий специалист по работе с просроченной задолженностью": "не ИТ",
  "Клиентский менеджер управления продаж малого бизнеса": "не ИТ",
  "Intern": "не ИТ",
  "Менеджер по работе с зарплатными проектами": "не ИТ",
  "Главный специалист банковского сопровождения контрактов": "не ИТ",
  "Заместитель руководителя обособленного подразделения": "не ИТ",
  "Junior Backend-разработчик": "Разработка",
  "Старший специалист отдела прямых продаж": "не ИТ",
  "Менеджер сектора обучения": "не ИТ",
  "Специалист сектора отложенных операций": "не ИТ",
  "Начальник РКО": "не ИТ",
  "Ресечер": "не ИТ",
  "Специалист отдела урегулирования проблемной задолженности": "не ИТ",
  "Главный клиентский менеджер малого бизнеса": "не ИТ",
  "Сотрудник охраны": "не ИТ",
  "Старший специалист по взысканию отдела досудебного взыскания задолженности": "не ИТ",
  "Менеджер управления продаж малому бизнесу": "не ИТ",
  "Главный специалист центра снабжения": "не ИТ",
  "Руководитель направления по системной аналитике": "Разработка",
  "Ведущий специалист отдела поддержки": "не ИТ",
  "Ведущий специалист по закупкам": "не ИТ",
  "Заместитель руководителя по продажам": "не ИТ",
  "Старший инженер межрегионального центра технической поддержки": "Администрирование",
  "Менеджер отдела технической поддержки": "Администрирование",
  "Начальник сектора по обслуживанию юридических лиц": "не ИТ",
  "Ведущий инженер-экономист": "не ИТ",
  "Специалист отдела выездных проверок": "не ИТ",
  "Главный контент-менеджер": "не ИТ",
  "Главный клиентский менеджер по привлечению корпоративных клиентов": "не ИТ",
  "Региональный менеджер по внутренним коммуникациям": "не ИТ",
  "Аналитик отдела подбора и адаптации персонала": "не ИТ",
  "Начальник сектора кредитования клиентов малого бизнеса": "не ИТ",
  "Дизайнер-верстальщик": "Дизайн",
  "Руководитель центра премиального обслуживания": "не ИТ",
  "Территориальный менеджер по работе с партнерами малого бизнеса": "не ИТ",
  "Начальник сектора комплектования": "не ИТ",
  "Специалист отдела зарплатных проектов": "не ИТ",
  "Менеджер по внутренним коммуникациям и корпоративной культуре": "не ИТ",
  "Middle UX/UI Дизайнер": "Дизайн",
  "Экономист-аналитик": "не ИТ",
  "Ведущий специалист отдела верификации подозрительных транзакций": "не ИТ",
  "Эксперт по подбору и оценке персонала": "не ИТ",
  "Заведующий хозяйством": "не ИТ",
  "Инцидент-менеджер": "Администрирование",
  "Руководитель КИЦ": "не ИТ",
  "Главный специалист отдела обработки запросов": "не ИТ",
  "Эксперт отдела качества клиентского обслуживания": "не ИТ",
  "Ассистент отдела по работе с ключевыми клиентами": "не ИТ",
  "Специалист по технической поддержке эквайринга": "Администрирование",
  "Начальник отдела прямых продаж": "не ИТ",
  "Главный специалист по международной отчетности": "не ИТ",
  "Корпоративный менеджер крупного и среднего бизнеса": "не ИТ",
  "Инженер-энергетик": "не ИТ",
  "HR Бизнес-партнер": "не ИТ",
  "Senior Engineer": "Разработка",
  "Начальник сектора кассовых операций": "не ИТ",
  "Специалист управления поддержки внутренних клиентов": "не ИТ",
  "Эксперт по информационной безопасности": "Администрирование",
  "Начальник отдела по работе с корпоративными клиентами": "не ИТ",
  "Стажер инженер-разработчик": "Разработка",
  "Операционный эксперт": "не ИТ",
  "Старший инспектор сектора защиты информационных технологий": "Администрирование",
  "Эксперт отдела подбора персонала": "не ИТ",
  "Главный специалист отдела по работе с отложенными обращениями": "не ИТ",
  "Территориальный менеджер по ипотечному кредитованию": "не ИТ",
  "Эксперт управления по работе с персоналом": "не ИТ",
  "Руководитель финансового подразделения": "не ИТ",
  "Ведущий специалист экономического отдела": "не ИТ",
  "Главный экономист по кредитованию юридических лиц": "не ИТ",
  "Middle backend-разработчик": "Разработка",
  "Кредитный аналитик малого бизнеса": "не ИТ",
  "Руководитель направления корпоративного бизнеса": "не ИТ",
  "Старший специалист по развитию инфраструктуры": "не ИТ",
  "Начальник центра обучения и развития": "не ИТ",
  "Менеджер по товару": "не ИТ",
  "Специалист подразделения маркетинга и коммуникаций": "не ИТ",
  "Ведущий специалист по экономической безопасности": "не ИТ",
  "Старший специалист отдела обработки документов": "не ИТ",
  "Старший специалист по работе с обращениями физических лиц": "не ИТ",
  "Руководитель разработки": "Разработка",
  "Старший инженер ЦОД": "Администрирование",
  "Старший специалист по работе с залогами": "не ИТ",
  "Сервис-консультант": "не ИТ",
  "Специалист внутренней поддержки клиентов": "не ИТ",
  "Старший инженер-разработчик Java": "Разработка",
  "Главный специалист ипотечного кредитования": "не ИТ",
  "Senior Golang Developer": "Разработка",
  "Старший редактор-переводчик": "не ИТ",
  "Эксперт центра реализации HR-стратегии функциональных блоков": "не ИТ",
  "Руководитель направления развития IT-систем": "Администрирование",
  "Кредитный инспектор отдела мониторинга корпоративных клиентов": "не ИТ",
  "Специалист отдела недвижимости": "не ИТ",
  "Старший специалист по сопровождению клиентов": "не ИТ",
  "Ведущий специалист отдела сопровождения бизнеса": "не ИТ",
  "Старший ипотечный менеджер": "не ИТ",
  "Специалист сопровождения зарплатных проектов": "не ИТ",
  "Эксперт по безопасности": "не ИТ",
  "Ведущий специалист по учету и отчетности": "не ИТ",
  "Java Software Developer": "Разработка",
  "Начальник отдела технической экспертизы межрегионального центра технической поддержки": "Администрирование",
  "Специалист по работе с партнёрами": "не ИТ",
  "Старший специалист по работе с персоналом": "не ИТ",
  "Специалист по операционно-кассовой работе": "не ИТ",
  "Главный инженер DevOps": "Администрирование",
  "Инженер по сопровождению АРМ": "Администрирование",
  "Сервисный менеджер": "не ИТ",
  "Кредитный инспектор отдела кредитования юридических лиц": "не ИТ",
  "Ведущий менеджер по персоналу": "не ИТ",
  "Специалист сервисной поддержки": "не ИТ",
  "Аналитик отдела недвижимости": "не ИТ",
  "Старший инженер отдела банковских карт": "не ИТ",
  "Старший клиентский менеджер по работе с корпоративными клиентами": "не ИТ",
  "Ведущий эксперт по информационной безопасности": "Администрирование",
  "Главный специалист претензионной службы": "не ИТ",
  "IOS Developer": "Разработка",
  "Специалист по обучению персонала": "не ИТ",
  "Инженер отдела технической поддержки": "Администрирование",
  "Специалист отдела технической поддержки": "Администрирование",
  "Руководитель направления контроля качества": "не ИТ",
  "Инженер-механик": "не ИТ",
  "Помощник HR-директора": "не ИТ",
  "Старший специалист по работе с юридическими лицами": "не ИТ",
  "Главный специалист отдела контроля и отчетности": "не ИТ",
  "Community Manager": "не ИТ",
  "Руководитель отдела маркетинга и коммуникаций": "не ИТ",
  "Инженер контроля качества": "Тестирование",
  "Начальник отдела сделок с недвижимостью": "не ИТ",
  "Старший специалист управления внутрибанковской безопасности": "не ИТ",
  "Менеджер по развитию валютного контроля и ВЭД": "не ИТ",
  "Юрисконсульт-стажер": "не ИТ",
  "Старший специалист отдела мониторинга": "не ИТ",
  "Аналитик учебно-методических программ": "не ИТ",
  "Консьерж": "не ИТ",
  "Ассистент маркетолога": "не ИТ",
  "Старший инженер отдела организации сервисной поддержки": "Администрирование",
  "Старший специалист по сопровождению кредитных операций": "не ИТ",
  "Менеджер по сопровождению ипотечных продуктов": "не ИТ",
  "Chief Product Owner": "не ИТ",
  "Эксперт разработки": "Разработка",
  "Старший специалист по операционному обслуживанию": "не ИТ",
  "Клиентский менеджер компаний горно-металлургического сектора": "не ИТ",
  "Ведущий специалист отдела прямых и выездных продаж": "не ИТ",
  "Директор развития бизнеса": "не ИТ",
  "QA по нагрузочному тестированию": "Тестирование",
  "Директор премиального обслуживания": "не ИТ",
  "Заместитель начальник отдела": "не ИТ",
  "Начальник дополнительного офиса": "не ИТ",
  "Старший архивариус": "не ИТ",
  "Ведущий специалист по работе с юридическими лицами": "не ИТ",
  "Территориальный руководитель по розничному бизнесу": "не ИТ",
  "Специалист отдела по обработке не голосовых обращений": "не ИТ",
  "Ведущий специалист отдела финансового мониторинга": "не ИТ",
  "Андеррайтер корпоративных клиентов": "не ИТ",
  "Руководитель экспертизы по сопровождению": "не ИТ",
  "Менеджер по сервисному обслуживанию": "не ИТ",
  "Директор по развитию Средний бизнес": "не ИТ",
  "Администратор ВДЛ": "Администрирование",
  "Начальник операционного офиса": "не ИТ",
  "Начальник ДО": "не ИТ",
  "Главный премиальный менеджер": "не ИТ",
  "Резервный секретарь": "не ИТ",
  "ДРКЦ Ведущий Эксперт отдела Мониторинга, планирования и отчетности": "не ИТ",
  "Вице-президент начальник департамента транзакционного контроля": "не ИТ",
  "Валютный контролер": "не ИТ",
  "Главный инженер по системному анализу Центра экспертизы системного анализа": "Разработка",
  "Заведующий кассы": "не ИТ",
  "Начальник отдела снабжения": "не ИТ",
  "Руководитель направления СБП для бизнеса": "не ИТ",
  "Специалист группы обслуживания": "не ИТ",
  "Ведущий специалист отдела прямых продаж": "не ИТ",
  "Управляющий директор по развитию эквайринга": "не ИТ",
  "Руководитель группы в интернет канале": "не ИТ",
  "Руководитель портфеля проектов": "не ИТ",
  "Начальник отдела развития технологий управленческой отчетности": "не ИТ",
  "Ведущий экономист МСБ": "не ИТ",
  "Специалист по автоматизации тестирования": "Тестирование",
  "Специалист по работе с учебными заведениями": "не ИТ",
  "Управляющий директор развития региональных продаж": "не ИТ",
  "Главный экономист Управления сопровождения сделок отраслевого финансирования": "не ИТ",
  "Начальник управления развития самоинкассации": "не ИТ",
  "Руководитель направления мониторинга залога отдела корпоративной защиты Управления экономической безопасности Департамента безопасности": "не ИТ",
  "Эксперт группы оптимизации бизнес процессов": "не ИТ",
  "Директор Департамента": "не ИТ",
  "Ведущий специалист группы сопровождения": "не ИТ",
  "Менеджер отдела администрирования": "не ИТ",
  "Специалист  УЭБ": "не ИТ",
  "Менеджер по малому и среднему бизнесу": "не ИТ",
  "Руководитель по развитию партнёрской сети": "не ИТ",
  "Ведущий экономист отдела сопровождения банковских кредитных продуктов": "не ИТ",
  "Главный специалист по обслуживанию клиентов": "не ИТ",
  "Финансовый менеджер товарного кредитования": "не ИТ",
  "Руководитель по работе с ключевыми клиентами": "не ИТ",
  "Менеджер по продажам МСБ": "не ИТ",
  "Старший ведущий персональный менеджер": "не ИТ",
  "Главный финансовый консультант": "не ИТ",
  "Старший финансовый эксперт": "не ИТ",
  "Заместитель регионального директора (Уральская дирекция)": "не ИТ",
  "Финансовый консультант торговых сетей": "не ИТ",
  "Оператор медицинского пульта": "не ИТ",
  "Специалист управления экономической безопасности": "не ИТ",
  "Главный специалист по подбору персонала (удаленно)": "не ИТ",
  "Ведущий финансовый консультант": "не ИТ",
  "Ведущий специалист чата": "не ИТ",
  "Главный эксперт по ипотечному кредитованию": "не ИТ",
  "Руководитель отдела прямых продаж": "не ИТ",
  "IT-менеджер": "Администрирование",
  "Кредитный Эксперт 3 Категории": "не ИТ",
  "Старший финансовый консультант": "не ИТ",
  "Ведущий специалист 4 категории": "не ИТ",
  "Специалист службы экономической безопасности": "не ИТ",
  "Руководитель группы контактного центра": "не ИТ",
  "Руководитель группы телемаркетинга": "не ИТ",
  "Ведущий оператор чат-поддержки": "не ИТ",
  "Сотрудник чат-поддержки": "не ИТ",
  "Руководитель розничной сети": "не ИТ",
  "Специалист отдела обслуживания и продаж": "не ИТ",
  "Директор КЦ": "не ИТ",
  "Управляющий": "не ИТ",
  "Работник склада": "не ИТ",
  "Лидер контроля качества ПО": "Тестирование",
  "Менеджер по активным продажам": "не ИТ",
  "Руководитель клиентского центра": "не ИТ",
  "Руководитель направления ДБО": "не ИТ",
  "Менеджер по работе с партнерской сетью": "не ИТ",
  "Территориальный руководитель развития продаж": "не ИТ",
  "Исполнительный директор макрорегиона «Поволжье» (группа VTB)": "не ИТ",
  "Главный специалист по работе с клиентами на поздней просрочки": "не ИТ",
  "Руководитель направления судебной и претензионной работы": "не ИТ",
  "Кредитный консультантультант": "не ИТ",
  "Руководитель по работе с ключевыми партнерами": "не ИТ",
  "Главный разработчик отдела разработки мобильных приложений": "Разработка",
  "Главный эксперт руководитель группы": "не ИТ",
  "Территориальный менеджер розничной сети": "не ИТ",
  "Специалист Направления контроля и качества": "не ИТ",
  "Специалист отдела продаж и клиентского обслуживания": "не ИТ",
  "Эксперт клиентского центра": "не ИТ",
  "Руководитель направления, технолог бизнес-процессов": "не ИТ",
  "Специалист поддержки чат/звонок": "не ИТ",
  "Специалист Клиентской службы г. Оренбург (контакт-центр)": "не ИТ",
  "Управляющий Республиканским центром": "не ИТ",
  "Специалист по работе в Чате": "не ИТ",
  "Главный специалист Отдела стандартов и методологии операционной деятельности Службы операционной поддержки бизнеса": "не ИТ",
  "Специалист группы информационной поддержки клиентов": "не ИТ",
  "Специалист по активным продажам": "не ИТ",
  "Территориальный менеджер pos": "не ИТ",
  "Территориальный менеджер по развитию продаж": "не ИТ",
  "Руководитель по активным продажам": "не ИТ",
  "Главный специалист выездного сбора": "не ИТ",
  "Ведущий специалист андеррайтинга": "не ИТ",
  "Начальник службы поддержки электронного бизнеса (ДБО)": "Администрирование",
  "Методолог продаж Дирекции коммерческого развития": "не ИТ",
  "Начальник отдела скоринга, риски": "не ИТ",
  "Менеджер активных продаж,": "не ИТ",
  "Главный системный  аналитик": "Разработка",
  "Администратор тестовых сред": "Администрирование",
  "Главный специалист группы андеррайтинга": "не ИТ",
  "Старший руководитель проектов": "не ИТ",
  "Специалист отдела обслуживания и продаж Клиентской службы": "не ИТ",
  "Руководитель корпоративного бизнеса": "не ИТ",
  "Специалист по оценке и сопровождению залогового имущества": "не ИТ",
  "Специалист группы оперативного мониторинга": "не ИТ",
  "Специалист по услугами и продажам": "не ИТ",
  "Ведущий специалист по связям с общественностью": "не ИТ",
  "Ведущий специалист, группа по работе с клиентской документацией": "не ИТ",
  "Старший специалист отдела по обслуживанию физических лиц": "не ИТ",
  "Директор Клиентского Центра": "не ИТ"
}

In [ ]:
sdf = pd.read_csv("reviews_llm_analysis.csv")
df = pd.DataFrame.from_dict(category, orient='index').reset_index()
df.columns = ['profession', 'category']
df = df[df['category'] != "не ИТ"]
result = sdf.merge(df, left_on='position', right_on='profession', how="inner")